# Notebook 13 — Phase 2A: F4 Representation Mechanism Analysis

## Frozen purpose

Phase 1 established a **heterogeneous** F3 → F4 effect: categorical Panchang
features were helpful in some model/horizon conditions and harmful in others.

This notebook does **not** search for a new best model. It asks why the F4
representation changes predictive behaviour.

### Core question

> Does correctly aligned Panchang structure contribute information beyond the
> effect of adding a high-dimensional categorical basis?

### Frozen principles

- Preserve the Phase-1 5-minute dataset, target, lags, horizons, and chronological splits.
- Preserve the canonical F3 and F4 definitions.
- Do not retune a model separately for each Phase-2 control.
- Use one frozen F4-selected configuration per model/horizon when models are fitted.
- Treat SOLETE Phase 2 as a **post-Phase1 mechanistic follow-up**, not as a new
  untouched confirmatory test.
- External validation on PVOD is reserved for Phase 2B.


In [1]:
# ============================================================
# Phase 2A protocol freeze
# ============================================================

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

search_locations = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    path
    for path in search_locations
    if (path / "src").is_dir()
    and (path / "solete_dataset").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PHASE2_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "notebook13_phase2_f4_representation_analysis"
)
PHASE2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PHASE2_PROTOCOL = {'phase': 'Phase 2A', 'notebook': '13_phase2_f4_representation_analysis.ipynb', 'status': 'frozen_before_phase2_results', 'research_question': 'Why does categorical Panchang representation alter PV forecasting performance differently across model architectures and forecast horizons?', 'mechanisms': {'dimension_or_flexibility': 'The F3->F4 effect is largely explained by adding a 97-dimensional categorical basis rather than by correct Panchang alignment.', 'aligned_structure': 'Correct timestamp-aligned Panchang structure contributes predictive information beyond dimensionality alone.', 'coding_sensitivity': 'Part of the effect, especially for linear models, may depend on the full one-hot coding convention.'}, 'frozen_phase1': {'resolution_minutes': 5, 'forecast_horizons_minutes': [15, 30, 60], 'target': 'P_Solar[kW]', 'feature_counts': {'F0': 118, 'F1': 133, 'F2': 138, 'F3': 147, 'F4': 244}, 'panchang_indicator_count': 97, 'test_reuse_status': 'Phase 2A is a post-Phase1 mechanistic follow-up on SOLETE. SOLETE test results are not treated as a new independent confirmatory test.'}, 'core_variants': {'F3': 'Continuous astronomical baseline; 147 features.', 'F4_real': 'F3 + correctly aligned 97-column Panchang one-hot block.', 'F4_shift_17d': 'F3 + Panchang block circularly shifted 17 days within each split.', 'F4_shift_37d': 'F3 + Panchang block circularly shifted 37 days within each split.', 'F4_shift_61d': 'F3 + Panchang block circularly shifted 61 days within each split.', 'F4_perm_1301': 'F3 + joint Panchang rows permuted within each split; seed 1301.', 'F4_perm_1302': 'F3 + joint Panchang rows permuted within each split; seed 1302.', 'F4_perm_1303': 'F3 + joint Panchang rows permuted within each split; seed 1303.', 'F4_reference': 'F3 + reference-coded Panchang block with one canonical indicator dropped from each of the five families.'}, 'component_screen': ['F3_plus_tithi', 'F3_plus_paksha', 'F3_plus_karana', 'F3_plus_nakshatra', 'F3_plus_yoga'], 'initial_model_strata': [{'model': 'Ridge', 'reason': 'linear/regularized architecture; tests sensitivity to high-dimensional categorical basis'}, {'model': 'LightGBM', 'reason': 'nonlinear boosted-tree architecture'}, {'model': 'LSTM', 'reason': 'recurrent deep-learning architecture'}, {'model': 'TCN', 'reason': 'temporal-convolution architecture'}], 'model_control_rule': 'For each model and horizon, use one fixed F4-selected architecture/hyperparameter configuration across F3 and every Phase-2 representation variant. Do not retune per variant.', 'primary_metric': 'MAE_kW', 'secondary_metrics': ['RMSE_kW', 'R2'], 'evaluation_scopes': ['all', 'daylight'], 'uncertainty': {'method': 'paired UTC calendar-day block bootstrap', 'daily_blocks': 92, 'replicates': 10000, 'seed': 42, 'interval': '95% percentile', 'model_retraining_during_bootstrap': False}, 'interpretation_guardrails': ['No causal physical claim.', 'No universal claim that F4 is superior.', 'SOLETE Phase-2 analysis is mechanistic/post-hoc relative to Phase 1.', 'External PVOD validation will be the independent generalization step.', 'Do not change variant definitions after seeing Phase-2 outcomes.']}

with open(
    PHASE2_RESULTS_DIR / "phase2_protocol.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(PHASE2_PROTOCOL, f, indent=2)

print("Project root:", PROJECT_ROOT)
print("Phase-2 results:", PHASE2_RESULTS_DIR)
print("Protocol status:", PHASE2_PROTOCOL["status"])
print("Protocol freeze: PASSED")


Project root: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project
Phase-2 results: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis
Protocol status: frozen_before_phase2_results
Protocol freeze: PASSED


## Reconstruct the frozen Phase-1 modelling data

The following cells are copied from the **data/feature construction portion**
of Notebook 09. They stop before baseline/model fitting and reproduce the same
F0–F4 matrices used in Phase 1.


In [2]:
# ============================================================
# Notebook 09 — Modelling Dataset and Experimental Protocol
# Step 1: Load the canonical 5-minute SOLETE dataset
# ============================================================

from pathlib import Path
import sys

import pandas as pd

# ------------------------------------------------------------
# 1. Locate the project root
# ------------------------------------------------------------
# This notebook is stored inside:
#
#   Project/notebooks/
#
# Depending on how VS Code launches the Jupyter kernel,
# Path.cwd() may be either the project directory itself or a
# directory somewhere inside it.
#
# Rather than assuming the current working directory, search
# upward until we find the project directory containing both:
#
#   src/
#   solete_dataset/
# ------------------------------------------------------------

search_locations = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    path
    for path in search_locations
    if (path / "src").is_dir()
    and (path / "solete_dataset").is_dir()
)

print("Project root:")
print(PROJECT_ROOT)


# ------------------------------------------------------------
# 2. Make the reusable src package importable
# ------------------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete


# ------------------------------------------------------------
# 3. Locate the 5-minute HDF5 dataset
# ------------------------------------------------------------
# We deliberately search for the file rather than hard-coding a
# filename here. This prevents the notebook from depending on an
# assumed file name while still requiring exactly one 5-minute
# HDF5 dataset to be present.
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "solete_dataset"

five_minute_files = [
    path
    for path in DATA_DIR.rglob("*.h5")
    if "5min" in path.name.lower().replace("_", "")
]

print("\n5-minute HDF5 candidates:")
for path in five_minute_files:
    print(" -", path.relative_to(PROJECT_ROOT))

assert len(five_minute_files) == 1, (
    "Expected exactly one 5-minute SOLETE .h5 file, "
    f"but found {len(five_minute_files)}."
)

DATA_PATH_5MIN = five_minute_files[0]


# ------------------------------------------------------------
# 4. Load the dataset using our reusable loader
# ------------------------------------------------------------

df_5min = load_solete(DATA_PATH_5MIN)


# ------------------------------------------------------------
# 5. Minimal loading sanity check
# ------------------------------------------------------------

print("\nLoaded dataset:")
print(f"Shape: {df_5min.shape}")
print(f"Start: {df_5min.index.min()}")
print(f"End:   {df_5min.index.max()}")

display(df_5min.head())

Project root:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project

5-minute HDF5 candidates:
 - solete_dataset/SOLETE_Pombo_5min.h5

Loaded dataset:
Shape: (131617, 11)
Start: 2018-06-01 00:00:00+00:00
End:   2019-09-01 00:00:00+00:00


,TEMPERATURE[degC],HUMIDITY[%],WIND_SPEED[m1s],WIND_DIR[deg],GHI[kW1m2],POA Irr[kW1m2],P_Gaia[kW],P_Solar[kW],Pressure[mbar],Azimuth[deg],Elevation[deg]
Timestamp,,,,,,,,,,,
2018-06-01 00:00:00+00:00,14.561204,0.7,1.293645,111.635452,0.0,0.0,0.046917,0.0,1017.559985,0.0,0.0
2018-06-01 00:05:00+00:00,14.487667,0.7,1.442333,91.236667,0.0,0.0,0.000267,0.0,1017.539990,0.0,0.0
2018-06-01 00:10:00+00:00,14.400000,0.7,1.408667,104.423333,0.0,0.0,0.000000,0.0,1017.493668,0.0,0.0
2018-06-01 00:15:00+00:00,14.240000,0.7,1.379333,103.976667,0.0,0.0,0.039050,0.0,1017.466342,0.0,0.0
2018-06-01 00:20:00+00:00,14.200000,0.7,1.453333,96.606667,0.0,0.0,0.003157,0.0,1017.533658,0.0,0.0


In [3]:
# ============================================================
# Step 2: Apply the validated observational preprocessing
# ============================================================

from src.preprocessing import build_cleaned_solete_features


# ------------------------------------------------------------
# 1. Build the cleaned observational dataset
# ------------------------------------------------------------
# This function:
#
#   - preserves all original SOLETE columns;
#   - corrects the known pressure artifact;
#   - cleans/wraps wind direction to [0, 360);
#   - creates sine/cosine wind-direction components;
#   - applies the validated causal humidity cleaning.
#
# IMPORTANT:
# We intentionally use build_cleaned_solete_features() here,
# rather than build_preprocessed_solete_features().
#
# Gregorian features will later be generated separately at the
# forecast-valid timestamp (t + h), according to the frozen
# experimental protocol.
# ------------------------------------------------------------

df_clean = build_cleaned_solete_features(df_5min)


# ------------------------------------------------------------
# 2. Verify the resulting dataset
# ------------------------------------------------------------

print("Cleaned dataset:")
print(f"Shape: {df_clean.shape}")
print(f"Start: {df_clean.index.min()}")
print(f"End:   {df_clean.index.max()}")

print("\nMissing values:")
print(df_clean.isna().sum().sum())

print("\nAdded cleaned columns:")
added_columns = [
    column
    for column in df_clean.columns
    if column not in df_5min.columns
]

for column in added_columns:
    print(" -", column)

display(df_clean.head())

Cleaned dataset:
Shape: (131617, 16)
Start: 2018-06-01 00:00:00+00:00
End:   2019-09-01 00:00:00+00:00

Missing values:
0

Added cleaned columns:
 - Pressure_clean[mbar]
 - Humidity_clean
 - Wind_dir_clean[deg]
 - wind_direction_sin
 - wind_direction_cos


,TEMPERATURE[degC],HUMIDITY[%],WIND_SPEED[m1s],WIND_DIR[deg],GHI[kW1m2],POA Irr[kW1m2],P_Gaia[kW],P_Solar[kW],Pressure[mbar],Azimuth[deg],Elevation[deg],Pressure_clean[mbar],Humidity_clean,Wind_dir_clean[deg],wind_direction_sin,wind_direction_cos
Timestamp,,,,,,,,,,,,,,,,
2018-06-01 00:00:00+00:00,14.561204,0.7,1.293645,111.635452,0.0,0.0,0.046917,0.0,1017.559985,0.0,0.0,1017.559985,0.7,111.635452,0.929549,-0.368700
2018-06-01 00:05:00+00:00,14.487667,0.7,1.442333,91.236667,0.0,0.0,0.000267,0.0,1017.539990,0.0,0.0,1017.539990,0.7,91.236667,0.999767,-0.021582
2018-06-01 00:10:00+00:00,14.400000,0.7,1.408667,104.423333,0.0,0.0,0.000000,0.0,1017.493668,0.0,0.0,1017.493668,0.7,104.423333,0.968482,-0.249084
2018-06-01 00:15:00+00:00,14.240000,0.7,1.379333,103.976667,0.0,0.0,0.039050,0.0,1017.466342,0.0,0.0,1017.466342,0.7,103.976667,0.970394,-0.241527
2018-06-01 00:20:00+00:00,14.200000,0.7,1.453333,96.606667,0.0,0.0,0.003157,0.0,1017.533658,0.0,0.0,1017.533658,0.7,96.606667,0.993359,-0.115053


In [4]:
# ============================================================
# Step 3: Construct the F0 historical / autoregressive block
# ============================================================

# ------------------------------------------------------------
# 1. Verify that row-wise shifting is safe
# ------------------------------------------------------------
# The lag construction below uses pandas .shift(), so the
# dataframe must represent a complete chronological 5-minute
# sequence.
# ------------------------------------------------------------

assert df_clean.index.is_monotonic_increasing, (
    "The 5-minute index is not chronologically ordered."
)

assert not df_clean.index.has_duplicates, (
    "Duplicate timestamps exist in the 5-minute dataset."
)

time_differences = df_clean.index.to_series().diff().dropna()

assert (time_differences == pd.Timedelta(minutes=5)).all(), (
    "The dataset is not a complete 5-minute grid."
)

print("5-minute timeline check: PASSED")


# ------------------------------------------------------------
# 2. Define the nine observed variables frozen for F0
# ------------------------------------------------------------
# Keys:
#   modelling-friendly base names
#
# Values:
#   corresponding columns in the cleaned SOLETE dataframe
# ------------------------------------------------------------

F0_HISTORY_SOURCES = {
    "P_Solar_kw": "P_Solar[kW]",
    "GHI_kw1m2": "GHI[kW1m2]",
    "POA_Irr_kw1m2": "POA Irr[kW1m2]",
    "temperature_degC": "TEMPERATURE[degC]",
    "humidity_clean": "Humidity_clean",
    "pressure_clean_mbar": "Pressure_clean[mbar]",
    "wind_speed_m1s": "WIND_SPEED[m1s]",
    "wind_direction_sin": "wind_direction_sin",
    "wind_direction_cos": "wind_direction_cos",
}


# ------------------------------------------------------------
# 3. Define the recent-history lag structure
# ------------------------------------------------------------
# At 5-minute resolution:
#
#   lag_0  = t
#   lag_1  = t - 5 min
#   ...
#   lag_12 = t - 60 min
# ------------------------------------------------------------

RECENT_LAGS = list(range(13))


# ------------------------------------------------------------
# 4. Build the recent-history feature dictionary
# ------------------------------------------------------------

f0_columns = {}

for feature_name, source_column in F0_HISTORY_SOURCES.items():
    for lag in RECENT_LAGS:
        column_name = f"{feature_name}_lag_{lag}"

        f0_columns[column_name] = df_clean[source_column].shift(lag)


# ------------------------------------------------------------
# 5. Add the frozen 24-hour seasonal PV lag
# ------------------------------------------------------------
# 24 hours × 60 minutes / 5-minute resolution = 288 steps.
# ------------------------------------------------------------

DAILY_LAG_STEPS = 288

f0_columns["P_Solar_kw_lag_288"] = (
    df_clean["P_Solar[kW]"].shift(DAILY_LAG_STEPS)
)


# ------------------------------------------------------------
# 6. Assemble the F0 historical feature dataframe
# ------------------------------------------------------------

df_f0_history = pd.DataFrame(
    f0_columns,
    index=df_clean.index,
)


# ------------------------------------------------------------
# 7. Verify the dimensionality against the frozen protocol
# ------------------------------------------------------------
# Expected:
#
#   9 variables × 13 recent lags = 117
#   + 1 daily PV lag            =   1
#                                  ---
#                                  118 features
# ------------------------------------------------------------

EXPECTED_F0_FEATURES = 118

assert df_f0_history.shape[1] == EXPECTED_F0_FEATURES, (
    f"Expected {EXPECTED_F0_FEATURES} F0 features, "
    f"but found {df_f0_history.shape[1]}."
)


# ------------------------------------------------------------
# 8. Inspect missingness caused by lag construction
# ------------------------------------------------------------
# Missing values at the beginning are expected because the
# 24-hour lag is unavailable for the first 288 observations.
#
# We do NOT fill these values artificially.
# ------------------------------------------------------------

complete_f0_rows = df_f0_history.notna().all(axis=1)

print("\nF0 historical feature block:")
print(f"Shape before dropping initial lag rows: {df_f0_history.shape}")
print(f"Number of F0 features: {df_f0_history.shape[1]}")
print(f"Complete rows: {complete_f0_rows.sum()}")
print(f"Rows requiring initial-history removal: {(~complete_f0_rows).sum()}")

print("\nFirst complete F0 timestamp:")
print(df_f0_history.index[complete_f0_rows][0])

print("\nLast F0 timestamp:")
print(df_f0_history.index[-1])

display(df_f0_history.loc[complete_f0_rows].head())

5-minute timeline check: PASSED

F0 historical feature block:
Shape before dropping initial lag rows: (131617, 118)
Number of F0 features: 118
Complete rows: 131329
Rows requiring initial-history removal: 288

First complete F0 timestamp:
2018-06-02 00:00:00+00:00

Last F0 timestamp:
2019-09-01 00:00:00+00:00


,P_Solar_kw_lag_0,P_Solar_kw_lag_1,P_Solar_kw_lag_2,P_Solar_kw_lag_3,P_Solar_kw_lag_4,P_Solar_kw_lag_5,P_Solar_kw_lag_6,P_Solar_kw_lag_7,P_Solar_kw_lag_8,P_Solar_kw_lag_9,...,wind_direction_cos_lag_4,wind_direction_cos_lag_5,wind_direction_cos_lag_6,wind_direction_cos_lag_7,wind_direction_cos_lag_8,wind_direction_cos_lag_9,wind_direction_cos_lag_10,wind_direction_cos_lag_11,wind_direction_cos_lag_12,P_Solar_kw_lag_288
Timestamp,,,,,,,,,,,,,,,,,,,,,
2018-06-02 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.300595,0.327768,0.428935,0.346335,0.996195,0.993082,0.922021,0.681998,0.681998,0.0
2018-06-02 00:05:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.292372,0.300595,0.327768,0.428935,0.346335,0.996195,0.993082,0.922021,0.681998,0.0
2018-06-02 00:10:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.292372,0.292372,0.300595,0.327768,0.428935,0.346335,0.996195,0.993082,0.922021,0.0
2018-06-02 00:15:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.292372,0.292372,0.292372,0.300595,0.327768,0.428935,0.346335,0.996195,0.993082,0.0
2018-06-02 00:20:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.963263,0.292372,0.292372,0.292372,0.300595,0.327768,0.428935,0.346335,0.996195,0.0


In [5]:
# ============================================================
# Step 4: Construct horizon-specific forecast targets
# ============================================================

# ------------------------------------------------------------
# 1. Define the frozen forecast horizons
# ------------------------------------------------------------
# At 5-minute resolution:
#
#   15 minutes =  3 steps
#   30 minutes =  6 steps
#   60 minutes = 12 steps
# ------------------------------------------------------------

FORECAST_HORIZONS = {
    15: 3,
    30: 6,
    60: 12,
}


# ------------------------------------------------------------
# 2. Build one target table for each forecast horizon
# ------------------------------------------------------------
# Each row is indexed by forecast issue time t.
#
# For horizon h:
#
#   issue_time = t
#   valid_time = t + h
#   target     = P_Solar(t + h)
#
# The target is constructed with a negative shift because we
# want a future value to appear on the current issue-time row.
# ------------------------------------------------------------

target_tables = {}

for horizon_minutes, horizon_steps in FORECAST_HORIZONS.items():

    df_target = pd.DataFrame(
        index=df_clean.index
    )

    # Keep issue time explicitly for later auditing.
    df_target["issue_time"] = df_target.index

    # Deterministic features will later be evaluated at this
    # forecast-valid timestamp.
    df_target["valid_time"] = (
        df_target["issue_time"]
        + pd.Timedelta(minutes=horizon_minutes)
    )

    # Future PV target:
    #
    # row at t receives P_Solar from t + h.
    df_target["target_P_Solar_kW"] = (
        df_clean["P_Solar[kW]"].shift(-horizon_steps)
    )

    # Store horizon metadata explicitly.
    df_target["horizon_minutes"] = horizon_minutes

    target_tables[horizon_minutes] = df_target


# ------------------------------------------------------------
# 3. Verify each target alignment
# ------------------------------------------------------------

for horizon_minutes, horizon_steps in FORECAST_HORIZONS.items():

    df_target = target_tables[horizon_minutes]

    target_available = df_target["target_P_Solar_kW"].notna()

    expected_available_rows = len(df_clean) - horizon_steps

    assert target_available.sum() == expected_available_rows, (
        f"{horizon_minutes}-minute target row count mismatch."
    )

    # The final available target should point to the final
    # timestamp in the original SOLETE dataset.
    last_issue_time = df_target.index[target_available][-1]
    last_valid_time = df_target.loc[last_issue_time, "valid_time"]

    assert last_valid_time == df_clean.index[-1], (
        f"{horizon_minutes}-minute final target alignment failed."
    )

    # Direct value-level alignment check.
    expected_target = df_clean.loc[
        last_valid_time,
        "P_Solar[kW]"
    ]

    actual_target = df_target.loc[
        last_issue_time,
        "target_P_Solar_kW"
    ]

    assert actual_target == expected_target, (
        f"{horizon_minutes}-minute target value alignment failed."
    )

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Steps ahead:              {horizon_steps}")
    print(f"  Available target rows:    {target_available.sum()}")
    print(f"  First issue time:         {df_target.index[0]}")
    print(f"  First valid time:         {df_target['valid_time'].iloc[0]}")
    print(f"  Last valid issue time:    {last_issue_time}")
    print(f"  Last valid target time:   {last_valid_time}")


15-minute horizon:
  Steps ahead:              3
  Available target rows:    131614
  First issue time:         2018-06-01 00:00:00+00:00
  First valid time:         2018-06-01 00:15:00+00:00
  Last valid issue time:    2019-08-31 23:45:00+00:00
  Last valid target time:   2019-09-01 00:00:00+00:00

30-minute horizon:
  Steps ahead:              6
  Available target rows:    131611
  First issue time:         2018-06-01 00:00:00+00:00
  First valid time:         2018-06-01 00:30:00+00:00
  Last valid issue time:    2019-08-31 23:30:00+00:00
  Last valid target time:   2019-09-01 00:00:00+00:00

60-minute horizon:
  Steps ahead:              12
  Available target rows:    131605
  First issue time:         2018-06-01 00:00:00+00:00
  First valid time:         2018-06-01 01:00:00+00:00
  Last valid issue time:    2019-08-31 23:00:00+00:00
  Last valid target time:   2019-09-01 00:00:00+00:00


In [6]:
# ============================================================
# Step 5: Combine F0 history, targets, and chronological splits
# ============================================================

# ------------------------------------------------------------
# 1. Define the frozen chronological split boundaries
# ------------------------------------------------------------

TRAIN_START = pd.Timestamp("2018-06-01 00:00:00", tz="UTC")
VALID_START = pd.Timestamp("2019-04-01 00:00:00", tz="UTC")
TEST_START = pd.Timestamp("2019-06-01 00:00:00", tz="UTC")
TEST_END = pd.Timestamp("2019-09-01 00:00:00", tz="UTC")


# ------------------------------------------------------------
# 2. Build one base modelling table per forecast horizon
# ------------------------------------------------------------
# Each table contains:
#
#   - issue-time index
#   - F0 historical features
#   - issue_time metadata
#   - valid_time metadata
#   - target
#   - horizon metadata
#   - chronological split label
#
# Deterministic F1-F4 features are NOT added yet.
# ------------------------------------------------------------

base_modelling_tables = {}

for horizon_minutes in FORECAST_HORIZONS:

    # Start with F0 historical predictors.
    df_model = df_f0_history.copy()

    # Add the previously validated target metadata.
    df_model = df_model.join(
        target_tables[horizon_minutes][
            [
                "issue_time",
                "valid_time",
                "target_P_Solar_kW",
                "horizon_minutes",
            ]
        ]
    )

    # --------------------------------------------------------
    # Keep only rows for which:
    #
    #   1. all 118 F0 historical features are available;
    #   2. the future target exists;
    #   3. valid_time lies inside the frozen experiment period.
    #
    # TEST_END is exclusive, so the single dataset-boundary
    # timestamp 2019-09-01 00:00 UTC is not part of the test set.
    # --------------------------------------------------------

    usable_mask = (
        df_model[df_f0_history.columns].notna().all(axis=1)
        & df_model["target_P_Solar_kW"].notna()
        & (df_model["valid_time"] >= TRAIN_START)
        & (df_model["valid_time"] < TEST_END)
    )

    df_model = df_model.loc[usable_mask].copy()

    # --------------------------------------------------------
    # Assign split according to FORECAST-VALID time, not issue
    # time.
    # --------------------------------------------------------

    df_model["split"] = pd.NA

    df_model.loc[
        (df_model["valid_time"] >= TRAIN_START)
        & (df_model["valid_time"] < VALID_START),
        "split",
    ] = "train"

    df_model.loc[
        (df_model["valid_time"] >= VALID_START)
        & (df_model["valid_time"] < TEST_START),
        "split",
    ] = "validation"

    df_model.loc[
        (df_model["valid_time"] >= TEST_START)
        & (df_model["valid_time"] < TEST_END),
        "split",
    ] = "test"

    # Every retained row must belong to exactly one split.
    assert df_model["split"].notna().all(), (
        f"Unassigned split rows found for "
        f"{horizon_minutes}-minute horizon."
    )

    base_modelling_tables[horizon_minutes] = df_model


# ------------------------------------------------------------
# 3. Report row counts and timing for each horizon
# ------------------------------------------------------------

for horizon_minutes, df_model in base_modelling_tables.items():

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Total usable rows: {len(df_model)}")

    split_counts = (
        df_model["split"]
        .value_counts()
        .reindex(["train", "validation", "test"])
    )

    print("  Split counts:")
    print(split_counts.to_string())

    print(
        f"  First issue time: "
        f"{df_model['issue_time'].iloc[0]}"
    )

    print(
        f"  First valid time: "
        f"{df_model['valid_time'].iloc[0]}"
    )

    print(
        f"  Last issue time:  "
        f"{df_model['issue_time'].iloc[-1]}"
    )

    print(
        f"  Last valid time:  "
        f"{df_model['valid_time'].iloc[-1]}"
    )


# ------------------------------------------------------------
# 4. Sanity check the split boundary itself
# ------------------------------------------------------------

df_60 = base_modelling_tables[60]

print("\n60-minute boundary examples:")

print("\nLast training row:")
display(
    df_60.loc[df_60["split"] == "train",
              ["issue_time", "valid_time", "target_P_Solar_kW"]]
    .tail(1)
)

print("\nFirst validation row:")
display(
    df_60.loc[df_60["split"] == "validation",
              ["issue_time", "valid_time", "target_P_Solar_kW"]]
    .head(1)
)

print("\nFirst test row:")
display(
    df_60.loc[df_60["split"] == "test",
              ["issue_time", "valid_time", "target_P_Solar_kW"]]
    .head(1)
)


15-minute horizon:
  Total usable rows: 131325
  Split counts:
split
train         87261
validation    17568
test          26496
  First issue time: 2018-06-02 00:00:00+00:00
  First valid time: 2018-06-02 00:15:00+00:00
  Last issue time:  2019-08-31 23:40:00+00:00
  Last valid time:  2019-08-31 23:55:00+00:00

30-minute horizon:
  Total usable rows: 131322
  Split counts:
split
train         87258
validation    17568
test          26496
  First issue time: 2018-06-02 00:00:00+00:00
  First valid time: 2018-06-02 00:30:00+00:00
  Last issue time:  2019-08-31 23:25:00+00:00
  Last valid time:  2019-08-31 23:55:00+00:00

60-minute horizon:
  Total usable rows: 131316
  Split counts:
split
train         87252
validation    17568
test          26496
  First issue time: 2018-06-02 00:00:00+00:00
  First valid time: 2018-06-02 01:00:00+00:00
  Last issue time:  2019-08-31 22:55:00+00:00
  Last valid time:  2019-08-31 23:55:00+00:00

60-minute boundary examples:

Last training row:


,issue_time,valid_time,target_P_Solar_kW
Timestamp,,,
2019-03-31 22:55:00+00:00,2019-03-31 22:55:00+00:00,2019-03-31 23:55:00+00:00,0.0



First validation row:


,issue_time,valid_time,target_P_Solar_kW
Timestamp,,,
2019-03-31 23:00:00+00:00,2019-03-31 23:00:00+00:00,2019-04-01 00:00:00+00:00,0.0



First test row:


,issue_time,valid_time,target_P_Solar_kW
Timestamp,,,
2019-05-31 23:00:00+00:00,2019-05-31 23:00:00+00:00,2019-06-01 00:00:00+00:00,0.001667


In [7]:
# ============================================================
# Step 6: Build F1 Gregorian features at forecast-valid time
# ============================================================

from src.preprocessing import build_gregorian_features


# ------------------------------------------------------------
# 1. Build one forecast-valid Gregorian feature table for
#    each forecast horizon
# ------------------------------------------------------------
#
# IMPORTANT:
#
# The reusable Gregorian function accepts a DatetimeIndex and
# calculates the 15 validated deterministic UTC features.
#
# For forecasting, these features must describe:
#
#     valid_time = t + h
#
# rather than the issue time t.
#
# This is allowed because calendar information is deterministic
# and known in advance.
# ------------------------------------------------------------

gregorian_feature_tables = {}

for horizon_minutes, df_model in base_modelling_tables.items():

    # Extract the future forecast-valid timestamps.
    valid_times = pd.DatetimeIndex(
        df_model["valid_time"],
        name="valid_time",
    )

    # Generate the validated 15 Gregorian features at t + h.
    df_gregorian = build_gregorian_features(valid_times)

    # --------------------------------------------------------
    # Verify that the feature builder preserved the exact
    # forecast-valid timestamp sequence.
    # --------------------------------------------------------

    assert df_gregorian.index.equals(valid_times), (
        f"Gregorian timestamp alignment failed for "
        f"{horizon_minutes}-minute horizon."
    )

    assert df_gregorian.shape[1] == 15, (
        f"Expected 15 Gregorian features, but found "
        f"{df_gregorian.shape[1]}."
    )

    assert df_gregorian.isna().sum().sum() == 0, (
        f"Missing Gregorian values found for "
        f"{horizon_minutes}-minute horizon."
    )

    # --------------------------------------------------------
    # Prefix the columns with 'valid_' to make their timing
    # explicit inside the modelling dataset.
    # --------------------------------------------------------

    df_gregorian = df_gregorian.add_prefix("valid_")

    # The feature values describe valid_time, but modelling
    # rows are indexed by issue time.
    #
    # After validating the timestamp alignment above, assign
    # the issue-time index so the features can later be joined
    # safely to the corresponding modelling rows.
    df_gregorian.index = df_model.index

    gregorian_feature_tables[horizon_minutes] = df_gregorian


# ------------------------------------------------------------
# 2. Inspect all three horizons
# ------------------------------------------------------------

for horizon_minutes, df_gregorian in gregorian_feature_tables.items():

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Shape:          {df_gregorian.shape}")
    print(
        f"  Missing values: "
        f"{df_gregorian.isna().sum().sum()}"
    )

    print(
        f"  First issue:    "
        f"{df_gregorian.index[0]}"
    )

    print(
        f"  First valid:    "
        f"{base_modelling_tables[horizon_minutes]['valid_time'].iloc[0]}"
    )

    print(
        f"  Valid hour:     "
        f"{df_gregorian['valid_utc_hour'].iloc[0]}"
    )

    print(
        f"  Valid minute:   "
        f"{df_gregorian['valid_utc_minute'].iloc[0]}"
    )


# ------------------------------------------------------------
# 3. Explicit 60-minute alignment sanity check
# ------------------------------------------------------------
#
# The first usable 60-minute modelling row has:
#
#   issue_time = 2018-06-02 00:00 UTC
#   valid_time = 2018-06-02 01:00 UTC
#
# Therefore its Gregorian features must describe 01:00,
# not 00:00.
# ------------------------------------------------------------

df_greg_60 = gregorian_feature_tables[60]
df_model_60 = base_modelling_tables[60]

assert (
    df_model_60["valid_time"].iloc[0]
    == pd.Timestamp("2018-06-02 01:00:00", tz="UTC")
)

assert df_greg_60["valid_utc_hour"].iloc[0] == 1
assert df_greg_60["valid_utc_minute"].iloc[0] == 0

print("\n60-minute forecast-valid alignment check: PASSED")

display(df_greg_60.head())


15-minute horizon:
  Shape:          (131325, 15)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:15:00+00:00
  Valid hour:     0
  Valid minute:   15

30-minute horizon:
  Shape:          (131322, 15)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:30:00+00:00
  Valid hour:     0
  Valid minute:   30

60-minute horizon:
  Shape:          (131316, 15)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 01:00:00+00:00
  Valid hour:     1
  Valid minute:   0

60-minute forecast-valid alignment check: PASSED


,valid_utc_year,valid_utc_month,valid_utc_day,valid_utc_day_of_week,valid_utc_day_of_year,valid_utc_hour,valid_utc_minute,valid_utc_minute_of_day,valid_utc_is_weekend,valid_utc_time_sin,valid_utc_time_cos,valid_utc_weekday_sin,valid_utc_weekday_cos,valid_utc_year_sin,valid_utc_year_cos
Timestamp,,,,,,,,,,,,,,,
2018-06-02 00:00:00+00:00,2018,6,2,5,153,1,0,60,1,0.258819,0.965926,-0.974928,-0.222521,0.500621,-0.865667
2018-06-02 00:05:00+00:00,2018,6,2,5,153,1,5,65,1,0.279829,0.960050,-0.974928,-0.222521,0.500569,-0.865696
2018-06-02 00:10:00+00:00,2018,6,2,5,153,1,10,70,1,0.300706,0.953717,-0.974928,-0.222521,0.500518,-0.865726
2018-06-02 00:15:00+00:00,2018,6,2,5,153,1,15,75,1,0.321439,0.946930,-0.974928,-0.222521,0.500466,-0.865756
2018-06-02 00:20:00+00:00,2018,6,2,5,153,1,20,80,1,0.342020,0.939693,-0.974928,-0.222521,0.500414,-0.865786


In [8]:
# ============================================================
# Step 7: Build F2 solar astronomy at forecast-valid time
# ============================================================

from src.astronomy_features import build_solar_f2_features


# ------------------------------------------------------------
# 1. Build one solar-feature table for each forecast horizon
# ------------------------------------------------------------
#
# Important timing convention:
#
#   modelling valid time = t + h
#
# The reusable astronomy function internally evaluates each
# SOLETE timestamp at:
#
#   SOLETE timestamp + 2 minutes
#
# This represents the approximate midpoint of the underlying
# 5-minute aggregation interval established during Notebook 07.
#
# Therefore:
#
#   DO NOT manually add another +2 minutes here.
# ------------------------------------------------------------

solar_feature_tables = {}

for horizon_minutes, df_model in base_modelling_tables.items():

    # Forecast-valid SOLETE timestamps.
    valid_times = pd.DatetimeIndex(
        df_model["valid_time"],
        name="valid_time",
    )

    # Build the validated five F2 solar features.
    #
    # Internally:
    #
    #   valid_time
    #       -> valid_time + 2 minutes
    #       -> pvlib solar-position calculation
    #
    # The returned dataframe retains valid_time as its index.
    df_solar = build_solar_f2_features(valid_times)

    # --------------------------------------------------------
    # Validate the output before changing its index.
    # --------------------------------------------------------

    assert df_solar.index.equals(valid_times), (
        f"Solar timestamp alignment failed for "
        f"{horizon_minutes}-minute horizon."
    )

    assert df_solar.shape[1] == 5, (
        f"Expected 5 F2 solar features, but found "
        f"{df_solar.shape[1]}."
    )

    assert df_solar.isna().sum().sum() == 0, (
        f"Missing solar values found for "
        f"{horizon_minutes}-minute horizon."
    )

    # Expected canonical F2 feature names.
    expected_solar_columns = [
        "solar_elevation_deg",
        "solar_cos_zenith",
        "solar_azimuth_sin",
        "solar_azimuth_cos",
        "solar_daylight",
    ]

    assert list(df_solar.columns) == expected_solar_columns, (
        "Unexpected F2 solar feature columns."
    )

    # --------------------------------------------------------
    # Prefix with valid_ so the feature timing remains explicit
    # inside the modelling table.
    # --------------------------------------------------------

    df_solar = df_solar.add_prefix("valid_")

    # Values describe valid_time, but rows in the modelling
    # dataset are indexed by issue time.
    df_solar.index = df_model.index

    solar_feature_tables[horizon_minutes] = df_solar


# ------------------------------------------------------------
# 2. Report dimensionality and alignment for each horizon
# ------------------------------------------------------------

for horizon_minutes, df_solar in solar_feature_tables.items():

    df_model = base_modelling_tables[horizon_minutes]

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Shape:          {df_solar.shape}")
    print(
        f"  Missing values: "
        f"{df_solar.isna().sum().sum()}"
    )

    print(
        f"  First issue:    "
        f"{df_model['issue_time'].iloc[0]}"
    )

    print(
        f"  First valid:    "
        f"{df_model['valid_time'].iloc[0]}"
    )

    print(
        f"  Astronomy eval: "
        f"{df_model['valid_time'].iloc[0] + pd.Timedelta(minutes=2)}"
    )


# ------------------------------------------------------------
# 3. Explicit 60-minute timing check
# ------------------------------------------------------------
#
# First 60-minute modelling row:
#
#   issue_time       = 2018-06-02 00:00 UTC
#   valid_time       = 2018-06-02 01:00 UTC
#   astronomy eval   = 2018-06-02 01:02 UTC
# ------------------------------------------------------------

df_solar_60 = solar_feature_tables[60]
df_model_60 = base_modelling_tables[60]

first_valid_60 = df_model_60["valid_time"].iloc[0]
first_astro_60 = first_valid_60 + pd.Timedelta(minutes=2)

assert first_valid_60 == pd.Timestamp(
    "2018-06-02 01:00:00",
    tz="UTC",
)

assert first_astro_60 == pd.Timestamp(
    "2018-06-02 01:02:00",
    tz="UTC",
)

print("\n60-minute astronomy timing check: PASSED")

print("\nFirst 60-minute forecast solar features:")
print(f"Issue time:      {df_model_60['issue_time'].iloc[0]}")
print(f"Valid time:      {first_valid_60}")
print(f"Astronomy time:  {first_astro_60}")

display(df_solar_60.head())


15-minute horizon:
  Shape:          (131325, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:15:00+00:00
  Astronomy eval: 2018-06-02 00:17:00+00:00

30-minute horizon:
  Shape:          (131322, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:30:00+00:00
  Astronomy eval: 2018-06-02 00:32:00+00:00

60-minute horizon:
  Shape:          (131316, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 01:00:00+00:00
  Astronomy eval: 2018-06-02 01:02:00+00:00

60-minute astronomy timing check: PASSED

First 60-minute forecast solar features:
Issue time:      2018-06-02 00:00:00+00:00
Valid time:      2018-06-02 01:00:00+00:00
Astronomy time:  2018-06-02 01:02:00+00:00


,valid_solar_elevation_deg,valid_solar_cos_zenith,valid_solar_azimuth_sin,valid_solar_azimuth_cos,valid_solar_daylight
Timestamp,,,,,
2018-06-02 00:00:00+00:00,-8.571247,-0.149039,0.441480,0.897271,0
2018-06-02 00:05:00+00:00,-8.253567,-0.143554,0.459015,0.888428,0
2018-06-02 00:10:00+00:00,-7.923620,-0.137853,0.476304,0.879281,0
2018-06-02 00:15:00+00:00,-7.581581,-0.131938,0.493341,0.869836,0
2018-06-02 00:20:00+00:00,-7.227632,-0.125812,0.510120,0.860103,0


In [9]:
# ============================================================
# Step 8: Inspect the reusable F3 lunar-feature API
# ============================================================

import inspect
import src.astronomy_features as astronomy_features


# ------------------------------------------------------------
# Find functions related to the lunar / F3 feature pipeline.
#
# We inspect the module rather than guessing the exact function
# signature, particularly because the lunar calculations also
# depend on the local JPL ephemeris file.
# ------------------------------------------------------------

candidate_names = [
    name
    for name in dir(astronomy_features)
    if any(
        keyword in name.lower()
        for keyword in ["lunar", "moon", "f3", "ephemeris"]
    )
    and callable(getattr(astronomy_features, name))
]


print("Relevant astronomy functions:\n")

for name in candidate_names:
    function = getattr(astronomy_features, name)

    try:
        signature = inspect.signature(function)
    except (TypeError, ValueError):
        signature = "(signature unavailable)"

    print(f"{name}{signature}")

Relevant astronomy functions:

build_lunar_f3_features(timestamps: 'pd.DatetimeIndex', ephemeris) -> 'pd.DataFrame'
load_jpl_ephemeris(ephemeris_path: 'str | Path')
moon_ecliptic_latitude_and_distance(ephemeris, skyfield_times) -> 'tuple[np.ndarray, np.ndarray]'
moon_illumination_fraction(ephemeris, skyfield_times) -> 'np.ndarray'
moon_sun_phase_angle_deg(sun_longitude_deg: 'float | np.ndarray | pd.Series', moon_longitude_deg: 'float | np.ndarray | pd.Series') -> 'float | np.ndarray | pd.Series'


In [10]:
# ============================================================
# Step 9: Build F3 lunar / Sun-Moon astronomical features
# ============================================================

from src.astronomy_features import (
    load_jpl_ephemeris,
    build_lunar_f3_features,
)


# ------------------------------------------------------------
# 1. Locate and load the local JPL ephemeris
# ------------------------------------------------------------
# Notebook 07 established DE440s as the astronomical ephemeris
# used by the reusable lunar feature pipeline.
#
# The file is stored locally and is intentionally not tracked
# by Git because of its size.
# ------------------------------------------------------------

EPHEMERIS_PATH = PROJECT_ROOT / "ephemeris" / "de440s.bsp"

assert EPHEMERIS_PATH.is_file(), (
    f"JPL ephemeris file not found:\n{EPHEMERIS_PATH}"
)

print("JPL ephemeris:")
print(EPHEMERIS_PATH.relative_to(PROJECT_ROOT))


# Load the ephemeris once and reuse the same object.
ephemeris = load_jpl_ephemeris(EPHEMERIS_PATH)

print("\nEphemeris loaded successfully.")


# ------------------------------------------------------------
# 2. Compute F3 lunar features ONCE for the complete canonical
#    5-minute SOLETE timestamp grid
# ------------------------------------------------------------
#
# This is more efficient than recalculating the same astronomy
# separately for the 15-, 30-, and 60-minute forecast tables.
#
# IMPORTANT:
#
# build_lunar_f3_features() internally follows the validated
# SOLETE astronomy-time convention:
#
#     SOLETE timestamp + 2 minutes
#
# Therefore we pass the original canonical timestamp grid and
# DO NOT manually add another +2 minutes.
# ------------------------------------------------------------

df_lunar_full = build_lunar_f3_features(
    df_clean.index,
    ephemeris,
)


# ------------------------------------------------------------
# 3. Validate the complete F3 lunar table
# ------------------------------------------------------------

expected_lunar_columns = [
    "moon_phase_sin",
    "moon_phase_cos",
    "moon_longitude_sin",
    "moon_longitude_cos",
    "sun_ecliptic_longitude_sin",
    "sun_ecliptic_longitude_cos",
    "moon_ecliptic_latitude_deg",
    "moon_illumination_fraction",
    "earth_moon_distance_km",
]

assert df_lunar_full.index.equals(df_clean.index), (
    "Full F3 lunar timestamp alignment failed."
)

assert list(df_lunar_full.columns) == expected_lunar_columns, (
    "Unexpected F3 lunar feature columns."
)

assert df_lunar_full.shape == (len(df_clean), 9), (
    f"Unexpected F3 lunar shape: {df_lunar_full.shape}"
)

assert df_lunar_full.isna().sum().sum() == 0, (
    "Missing values found in the full F3 lunar feature table."
)

print("\nFull F3 lunar table:")
print(f"Shape:          {df_lunar_full.shape}")
print(f"Missing values: {df_lunar_full.isna().sum().sum()}")
print(f"Start:          {df_lunar_full.index.min()}")
print(f"End:            {df_lunar_full.index.max()}")


# ------------------------------------------------------------
# 4. Select F3 features at each horizon's forecast-valid times
# ------------------------------------------------------------
#
# For every modelling row:
#
#     issue time = t
#     valid time = t + h
#
# We select the already-computed astronomy corresponding to
# valid_time.
#
# The selected values therefore describe the future target
# interval while remaining deterministic and future-known.
# ------------------------------------------------------------

lunar_feature_tables = {}

for horizon_minutes, df_model in base_modelling_tables.items():

    valid_times = pd.DatetimeIndex(
        df_model["valid_time"],
        name="valid_time",
    )

    # Select exact forecast-valid rows from the canonical
    # full-timeline lunar feature table.
    df_lunar = df_lunar_full.loc[valid_times].copy()

    # Verify that selection preserved the requested order.
    assert df_lunar.index.equals(valid_times), (
        f"F3 lunar valid-time alignment failed for "
        f"{horizon_minutes}-minute horizon."
    )

    assert df_lunar.shape[1] == 9

    assert df_lunar.isna().sum().sum() == 0

    # Prefix columns so their timing is explicit.
    df_lunar = df_lunar.add_prefix("valid_")

    # Values describe valid_time, but modelling rows themselves
    # are indexed by forecast issue time.
    df_lunar.index = df_model.index

    lunar_feature_tables[horizon_minutes] = df_lunar


# ------------------------------------------------------------
# 5. Report all horizon-specific F3 tables
# ------------------------------------------------------------

for horizon_minutes, df_lunar in lunar_feature_tables.items():

    df_model = base_modelling_tables[horizon_minutes]

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Shape:          {df_lunar.shape}")
    print(
        f"  Missing values: "
        f"{df_lunar.isna().sum().sum()}"
    )

    print(
        f"  First issue:    "
        f"{df_model['issue_time'].iloc[0]}"
    )

    print(
        f"  First valid:    "
        f"{df_model['valid_time'].iloc[0]}"
    )

    print(
        f"  Astronomy eval: "
        f"{df_model['valid_time'].iloc[0] + pd.Timedelta(minutes=2)}"
    )


# ------------------------------------------------------------
# 6. Explicit 60-minute alignment check
# ------------------------------------------------------------

df_lunar_60 = lunar_feature_tables[60]
df_model_60 = base_modelling_tables[60]

first_valid_60 = df_model_60["valid_time"].iloc[0]

assert first_valid_60 == pd.Timestamp(
    "2018-06-02 01:00:00",
    tz="UTC",
)

# Confirm that the first modelling F3 row is numerically equal
# to the canonical full-timeline astronomy at that valid time.
canonical_first_60 = (
    df_lunar_full
    .loc[first_valid_60]
    .to_numpy()
)

model_first_60 = (
    df_lunar_60
    .iloc[0]
    .to_numpy()
)

assert (
    abs(canonical_first_60 - model_first_60) < 1e-12
).all(), (
    "60-minute F3 value alignment failed."
)

print("\n60-minute F3 alignment check: PASSED")

print("\nFirst 60-minute forecast lunar features:")
print(f"Issue time:      {df_model_60['issue_time'].iloc[0]}")
print(f"Valid time:      {first_valid_60}")
print(
    f"Astronomy time:  "
    f"{first_valid_60 + pd.Timedelta(minutes=2)}"
)

display(df_lunar_60.head())

JPL ephemeris:
ephemeris/de440s.bsp

Ephemeris loaded successfully.

Full F3 lunar table:
Shape:          (131617, 9)
Missing values: 0
Start:          2018-06-01 00:00:00+00:00
End:            2019-09-01 00:00:00+00:00

15-minute horizon:
  Shape:          (131325, 9)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:15:00+00:00
  Astronomy eval: 2018-06-02 00:17:00+00:00

30-minute horizon:
  Shape:          (131322, 9)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:30:00+00:00
  Astronomy eval: 2018-06-02 00:32:00+00:00

60-minute horizon:
  Shape:          (131316, 9)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 01:00:00+00:00
  Astronomy eval: 2018-06-02 01:02:00+00:00

60-minute F3 alignment check: PASSED

First 60-minute forecast lunar features:
Issue time:      2018-06-02 00:00:00+00:00
Valid time:      2018-06-02 01:00:00+00:00
Astronom

,valid_moon_phase_sin,valid_moon_phase_cos,valid_moon_longitude_sin,valid_moon_longitude_cos,valid_sun_ecliptic_longitude_sin,valid_sun_ecliptic_longitude_cos,valid_moon_ecliptic_latitude_deg,valid_moon_illumination_fraction,valid_earth_moon_distance_km
Timestamp,,,,,,,,,
2018-06-02 00:00:00+00:00,-0.617980,-0.786194,-0.941826,0.336101,0.948162,0.317789,1.544368,0.893493,405091.945014
2018-06-02 00:05:00+00:00,-0.618498,-0.785787,-0.941585,0.336776,0.948180,0.317734,1.540866,0.893291,405094.028839
2018-06-02 00:10:00+00:00,-0.619015,-0.785379,-0.941343,0.337451,0.948198,0.317679,1.537363,0.893089,405096.101941
2018-06-02 00:15:00+00:00,-0.619532,-0.784972,-0.941101,0.338125,0.948217,0.317624,1.533859,0.892887,405098.164313
2018-06-02 00:20:00+00:00,-0.620049,-0.784563,-0.940859,0.338799,0.948235,0.317568,1.530355,0.892684,405100.215948


In [11]:
# ============================================================
# Step 10: Inspect the reusable F4 Panchang construction APIs
# ============================================================

import inspect

import src.astronomy_features as astronomy_features
import src.vedic_calendar as vedic_calendar


# ------------------------------------------------------------
# 1. Astronomy helper required by F4
# ------------------------------------------------------------
# F4 needs the raw tropical ecliptic longitudes of:
#
#   - Sun
#   - Moon
#
# evaluated using the same validated astronomy convention used
# throughout Notebook 07 and Notebook 08.
# ------------------------------------------------------------

astronomy_api_names = [
    "tropical_longitudes_for_solete",
]


print("Astronomy API:\n")

for name in astronomy_api_names:
    function = getattr(astronomy_features, name)
    print(f"{name}{inspect.signature(function)}")


# ------------------------------------------------------------
# 2. Vedic-calendar helpers required by F4
# ------------------------------------------------------------
# We need:
#
#   1. canonical Lahiri ayanamsha for each UTC timestamp;
#   2. the high-level builder that converts the validated
#      Sun/Moon geometry into the five raw Panchang features.
# ------------------------------------------------------------

vedic_api_names = [
    "lahiri_ayanamsha_for_utc_times",
    "build_panchang_f4_raw",
]


print("\nVedic-calendar API:\n")

for name in vedic_api_names:
    function = getattr(vedic_calendar, name)
    print(f"{name}{inspect.signature(function)}")

Astronomy API:

tropical_longitudes_for_solete(timestamps: 'pd.DatetimeIndex', ephemeris) -> 'pd.DataFrame'

Vedic-calendar API:

lahiri_ayanamsha_for_utc_times(timestamps: 'pd.DatetimeIndex') -> 'np.ndarray'
build_panchang_f4_raw(sun_tropical_longitude_deg: 'pd.Series', moon_tropical_longitude_deg: 'pd.Series', ayanamsha_deg: 'pd.Series') -> 'pd.DataFrame'


In [12]:
# ============================================================
# Step 11: Build canonical F4 Panchang features
# ============================================================

import numpy as np

from src.astronomy_features import (
    solete_astronomy_reference_times,
    tropical_longitudes_for_solete,
)

from src.vedic_calendar import (
    lahiri_ayanamsha_for_utc_times,
    build_panchang_f4_raw,
)


# ------------------------------------------------------------
# 1. Compute raw tropical Sun/Moon longitudes for the complete
#    canonical 5-minute SOLETE timeline
# ------------------------------------------------------------
#
# tropical_longitudes_for_solete() internally applies the
# validated SOLETE astronomy convention:
#
#     SOLETE timestamp + 2 minutes
#
# Therefore the returned Sun and Moon tropical longitudes
# correspond to the approximate midpoint of each 5-minute
# aggregation interval.
# ------------------------------------------------------------

df_tropical_full = tropical_longitudes_for_solete(
    df_clean.index,
    ephemeris,
)


# ------------------------------------------------------------
# 2. Verify the tropical-longitude output
# ------------------------------------------------------------

expected_tropical_columns = [
    "sun_tropical_longitude_deg",
    "moon_tropical_longitude_deg",
]

assert df_tropical_full.index.equals(df_clean.index), (
    "Tropical-longitude timestamp alignment failed."
)

assert list(df_tropical_full.columns) == expected_tropical_columns, (
    "Unexpected tropical-longitude columns."
)

assert df_tropical_full.shape == (len(df_clean), 2), (
    f"Unexpected tropical-longitude shape: "
    f"{df_tropical_full.shape}"
)

assert df_tropical_full.isna().sum().sum() == 0, (
    "Missing values found in tropical longitudes."
)

print("Full tropical-longitude table:")
print(f"Shape:          {df_tropical_full.shape}")
print(
    f"Missing values: "
    f"{df_tropical_full.isna().sum().sum()}"
)


# ------------------------------------------------------------
# 3. Construct the SAME +2-minute astronomy reference times
#    for the Lahiri ayanamsha calculation
# ------------------------------------------------------------
#
# Unlike tropical_longitudes_for_solete(), the reusable
# lahiri_ayanamsha_for_utc_times() function is generic and
# operates on the UTC timestamps supplied to it.
#
# Therefore we explicitly generate:
#
#     SOLETE timestamp + 2 minutes
#
# so that:
#
#   Sun tropical longitude
#   Moon tropical longitude
#   Lahiri ayanamsha
#
# all refer to exactly the same astronomical instant.
# ------------------------------------------------------------

astronomy_reference_times = solete_astronomy_reference_times(
    df_clean.index
)

assert len(astronomy_reference_times) == len(df_clean)

assert astronomy_reference_times[0] == (
    df_clean.index[0] + pd.Timedelta(minutes=2)
)

print("\nFirst SOLETE timestamp:")
print(df_clean.index[0])

print("First astronomy reference time:")
print(astronomy_reference_times[0])


# ------------------------------------------------------------
# 4. Compute canonical extended Lahiri ayanamsha
# ------------------------------------------------------------
#
# Notebook 08 established the canonical convention using the
# Swiss Ephemeris extended Lahiri calculation.
#
# The returned NumPy array contains one ayanamsha value for
# every +2-minute astronomy reference timestamp.
# ------------------------------------------------------------

ayanamsha_values = lahiri_ayanamsha_for_utc_times(
    astronomy_reference_times
)

assert len(ayanamsha_values) == len(df_clean)

assert np.isfinite(ayanamsha_values).all(), (
    "Non-finite Lahiri ayanamsha values found."
)

# Store the values as a Series indexed by the corresponding
# SOLETE timestamps. The VALUES refer to timestamp + 2 min,
# while the index identifies the SOLETE aggregation interval.
lahiri_ayanamsha_full = pd.Series(
    ayanamsha_values,
    index=df_clean.index,
    name="lahiri_ayanamsha_deg",
)


# ------------------------------------------------------------
# 5. Validate the known canonical Notebook 08 reference point
# ------------------------------------------------------------
#
# For SOLETE timestamp:
#
#     2018-06-01 00:00 UTC
#
# astronomy is evaluated at:
#
#     2018-06-01 00:02 UTC
#
# The validated canonical Lahiri ayanamsha is approximately:
#
#     24.110343268621794 degrees
# ------------------------------------------------------------

KNOWN_FIRST_AYANAMSHA = 24.110343268621794

assert np.isclose(
    lahiri_ayanamsha_full.iloc[0],
    KNOWN_FIRST_AYANAMSHA,
    atol=1e-10,
    rtol=0.0,
), (
    "Canonical first-point Lahiri ayanamsha check failed."
)

print("\nLahiri ayanamsha:")
print(
    f"First value: "
    f"{lahiri_ayanamsha_full.iloc[0]}"
)
print(
    f"Minimum:     "
    f"{lahiri_ayanamsha_full.min()}"
)
print(
    f"Maximum:     "
    f"{lahiri_ayanamsha_full.max()}"
)

print("\nCanonical Lahiri reference check: PASSED")


# ------------------------------------------------------------
# 6. Build the raw five-column F4 Panchang table
# ------------------------------------------------------------
#
# Inputs:
#
#   tropical Sun longitude
#   tropical Moon longitude
#   canonical extended Lahiri ayanamsha
#
# Output:
#
#   tithi
#   paksha
#   karana
#   nakshatra
#   yoga
# ------------------------------------------------------------

df_panchang_full = build_panchang_f4_raw(
    sun_tropical_longitude_deg=(
        df_tropical_full["sun_tropical_longitude_deg"]
    ),
    moon_tropical_longitude_deg=(
        df_tropical_full["moon_tropical_longitude_deg"]
    ),
    ayanamsha_deg=lahiri_ayanamsha_full,
)


# ------------------------------------------------------------
# 7. Validate the complete raw F4 table
# ------------------------------------------------------------

expected_panchang_columns = [
    "tithi",
    "paksha",
    "karana",
    "nakshatra",
    "yoga",
]

assert df_panchang_full.index.equals(df_clean.index), (
    "F4 Panchang timestamp alignment failed."
)

assert list(df_panchang_full.columns) == expected_panchang_columns, (
    "Unexpected raw F4 Panchang columns."
)

assert df_panchang_full.shape == (len(df_clean), 5), (
    f"Unexpected raw F4 shape: "
    f"{df_panchang_full.shape}"
)

assert df_panchang_full.isna().sum().sum() == 0, (
    "Missing values found in raw F4 Panchang features."
)

print("\nFull raw F4 Panchang table:")
print(f"Shape:          {df_panchang_full.shape}")
print(
    f"Missing values: "
    f"{df_panchang_full.isna().sum().sum()}"
)


# ------------------------------------------------------------
# 8. Verify the canonical first Panchang reference case
# ------------------------------------------------------------
#
# Notebook 08 established that the first SOLETE timestamp,
# evaluated astronomically at 00:02 UTC, must produce:
#
#   Tithi      = 18
#   Paksha     = Krishna
#   Karana     = Vanija
#   Nakshatra  = Mula
#   Yoga       = Shubha
# ------------------------------------------------------------

first_panchang = df_panchang_full.iloc[0]

assert int(first_panchang["tithi"]) == 18
assert first_panchang["paksha"] == "Krishna"
assert first_panchang["karana"] == "Vanija"
assert first_panchang["nakshatra"] == "Mula"
assert first_panchang["yoga"] == "Shubha"

print("\nCanonical first Panchang reference check: PASSED")

print("\nFirst raw F4 row:")
display(df_panchang_full.head(1))


# ------------------------------------------------------------
# 9. Select F4 categories at each forecast-valid time
# ------------------------------------------------------------
#
# As with F1-F3, Panchang features describe:
#
#     valid_time = t + h
#
# rather than the forecast issue time.
# ------------------------------------------------------------

panchang_feature_tables = {}

for horizon_minutes, df_model in base_modelling_tables.items():

    valid_times = pd.DatetimeIndex(
        df_model["valid_time"],
        name="valid_time",
    )

    # Select the already-computed Panchang state corresponding
    # exactly to each future forecast-valid timestamp.
    df_panchang = df_panchang_full.loc[valid_times].copy()

    assert df_panchang.index.equals(valid_times), (
        f"F4 valid-time alignment failed for "
        f"{horizon_minutes}-minute horizon."
    )

    assert df_panchang.shape[1] == 5

    assert df_panchang.isna().sum().sum() == 0

    # Prefix raw categorical columns to make their forecast
    # timing explicit.
    df_panchang = df_panchang.add_prefix("valid_")

    # Values describe valid_time, but modelling rows themselves
    # remain indexed by issue time.
    df_panchang.index = df_model.index

    panchang_feature_tables[horizon_minutes] = df_panchang


# ------------------------------------------------------------
# 10. Report horizon-specific raw F4 tables
# ------------------------------------------------------------

for horizon_minutes, df_panchang in (
    panchang_feature_tables.items()
):

    df_model = base_modelling_tables[horizon_minutes]

    print(f"\n{horizon_minutes}-minute horizon:")
    print(f"  Shape:          {df_panchang.shape}")
    print(
        f"  Missing values: "
        f"{df_panchang.isna().sum().sum()}"
    )

    print(
        f"  First issue:    "
        f"{df_model['issue_time'].iloc[0]}"
    )

    print(
        f"  First valid:    "
        f"{df_model['valid_time'].iloc[0]}"
    )


# ------------------------------------------------------------
# 11. Explicit 60-minute F4 alignment check
# ------------------------------------------------------------

df_panchang_60 = panchang_feature_tables[60]
df_model_60 = base_modelling_tables[60]

first_valid_60 = df_model_60["valid_time"].iloc[0]


# Canonical Panchang row at the forecast-valid timestamp.
canonical_f4_first_60 = df_panchang_full.loc[first_valid_60]


# First modelling Panchang row.
#
# Its column names contain the "valid_" prefix, whereas the
# canonical full-timeline table uses the original raw names.
model_f4_first_60 = df_panchang_60.iloc[0].copy()

model_f4_first_60.index = (
    model_f4_first_60.index
    .str.removeprefix("valid_")
)


# Now both Series have identical semantic labels and values.
assert canonical_f4_first_60.equals(
    model_f4_first_60
), (
    "60-minute F4 value alignment failed."
)


print("\n60-minute F4 alignment check: PASSED")

print("\nCanonical F4 at first 60-minute valid time:")
display(canonical_f4_first_60.to_frame("canonical"))

print("\nModelling F4 at the same valid time:")
display(model_f4_first_60.to_frame("model"))

print("\nFirst five 60-minute forecast-valid Panchang rows:")
display(df_panchang_60.head())

Full tropical-longitude table:
Shape:          (131617, 2)
Missing values: 0

First SOLETE timestamp:
2018-06-01 00:00:00+00:00
First astronomy reference time:
2018-06-01 00:02:00+00:00

Lahiri ayanamsha:
First value: 24.110343268621794
Minimum:     24.110343268621794
Maximum:     24.12737225615406

Canonical Lahiri reference check: PASSED

Full raw F4 Panchang table:
Shape:          (131617, 5)
Missing values: 0

Canonical first Panchang reference check: PASSED

First raw F4 row:


,tithi,paksha,karana,nakshatra,yoga
Timestamp,,,,,
2018-06-01 00:00:00+00:00,18,Krishna,Vanija,Mula,Shubha



15-minute horizon:
  Shape:          (131325, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:15:00+00:00

30-minute horizon:
  Shape:          (131322, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 00:30:00+00:00

60-minute horizon:
  Shape:          (131316, 5)
  Missing values: 0
  First issue:    2018-06-02 00:00:00+00:00
  First valid:    2018-06-02 01:00:00+00:00

60-minute F4 alignment check: PASSED

Canonical F4 at first 60-minute valid time:


,canonical
tithi,19
paksha,Krishna
karana,Bava
nakshatra,Purva Ashadha
yoga,Shukla



Modelling F4 at the same valid time:


,model
tithi,19
paksha,Krishna
karana,Bava
nakshatra,Purva Ashadha
yoga,Shukla



First five 60-minute forecast-valid Panchang rows:


,valid_tithi,valid_paksha,valid_karana,valid_nakshatra,valid_yoga
Timestamp,,,,,
2018-06-02 00:00:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:05:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:10:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:15:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:20:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla


In [13]:
# ============================================================
# Step 12: Assemble the nested F0-F4 feature-family tables
# ============================================================

# ------------------------------------------------------------
# 1. Storage structure
# ------------------------------------------------------------
#
# feature_family_tables[horizon][family]
#
# Example:
#
#   feature_family_tables[60]["F3"]
#
# returns the complete F3 model-input dataframe for the
# 60-minute forecast horizon.
# ------------------------------------------------------------

feature_family_tables = {}


# ------------------------------------------------------------
# 2. Assemble each feature family for every forecast horizon
# ------------------------------------------------------------
#
# Frozen hierarchy:
#
#   F0 = historical / autoregressive features
#   F1 = F0 + Gregorian
#   F2 = F1 + solar astronomy
#   F3 = F2 + lunar / Sun-Moon astronomy
#   F4_raw = F3 + raw Panchang categories
#
# F4 is called F4_raw here because the five Panchang columns
# have not yet been one-hot encoded.
# ------------------------------------------------------------

for horizon_minutes, df_model in base_modelling_tables.items():

    issue_index = df_model.index

    # --------------------------------------------------------
    # F0
    # --------------------------------------------------------

    df_f0 = df_f0_history.loc[issue_index].copy()

    # --------------------------------------------------------
    # F1 = F0 + 15 Gregorian features
    # --------------------------------------------------------

    df_f1 = pd.concat(
        [
            df_f0,
            gregorian_feature_tables[horizon_minutes],
        ],
        axis=1,
    )

    # --------------------------------------------------------
    # F2 = F1 + 5 solar-astronomy features
    # --------------------------------------------------------

    df_f2 = pd.concat(
        [
            df_f1,
            solar_feature_tables[horizon_minutes],
        ],
        axis=1,
    )

    # --------------------------------------------------------
    # F3 = F2 + 9 lunar / Sun-Moon features
    # --------------------------------------------------------

    df_f3 = pd.concat(
        [
            df_f2,
            lunar_feature_tables[horizon_minutes],
        ],
        axis=1,
    )

    # --------------------------------------------------------
    # F4_raw = F3 + 5 raw categorical Panchang features
    # --------------------------------------------------------

    df_f4_raw = pd.concat(
        [
            df_f3,
            panchang_feature_tables[horizon_minutes],
        ],
        axis=1,
    )

    feature_family_tables[horizon_minutes] = {
        "F0": df_f0,
        "F1": df_f1,
        "F2": df_f2,
        "F3": df_f3,
        "F4_raw": df_f4_raw,
    }


# ------------------------------------------------------------
# 3. Expected dimensions before F4 one-hot encoding
# ------------------------------------------------------------

EXPECTED_RAW_FEATURE_COUNTS = {
    "F0": 118,
    "F1": 133,
    "F2": 138,
    "F3": 147,
    "F4_raw": 152,   # 147 + 5 raw categorical columns
}


# ------------------------------------------------------------
# 4. Validate dimensions, alignment, and missingness
# ------------------------------------------------------------

for horizon_minutes, family_tables in feature_family_tables.items():

    expected_rows = len(base_modelling_tables[horizon_minutes])

    print(f"\n{horizon_minutes}-minute horizon:")

    for family_name, df_family in family_tables.items():

        expected_features = EXPECTED_RAW_FEATURE_COUNTS[
            family_name
        ]

        assert len(df_family) == expected_rows, (
            f"{horizon_minutes}-minute {family_name}: "
            "row-count mismatch."
        )

        assert df_family.shape[1] == expected_features, (
            f"{horizon_minutes}-minute {family_name}: "
            f"expected {expected_features} features, "
            f"found {df_family.shape[1]}."
        )

        assert df_family.index.equals(
            base_modelling_tables[horizon_minutes].index
        ), (
            f"{horizon_minutes}-minute {family_name}: "
            "issue-time index mismatch."
        )

        assert df_family.isna().sum().sum() == 0, (
            f"{horizon_minutes}-minute {family_name}: "
            "missing values found."
        )

        print(
            f"  {family_name:<6} "
            f"shape = {df_family.shape}"
        )


# ------------------------------------------------------------
# 5. Verify strict nested feature hierarchy
# ------------------------------------------------------------
#
# Every earlier family's columns must appear unchanged at the
# start of the next family.
# ------------------------------------------------------------

for horizon_minutes, family_tables in feature_family_tables.items():

    assert list(
        family_tables["F1"].columns[:118]
    ) == list(
        family_tables["F0"].columns
    )

    assert list(
        family_tables["F2"].columns[:133]
    ) == list(
        family_tables["F1"].columns
    )

    assert list(
        family_tables["F3"].columns[:138]
    ) == list(
        family_tables["F2"].columns
    )

    assert list(
        family_tables["F4_raw"].columns[:147]
    ) == list(
        family_tables["F3"].columns
    )

print("\nNested F0 -> F1 -> F2 -> F3 -> F4_raw checks: PASSED")


# ------------------------------------------------------------
# 6. Inspect the five raw F4 categorical columns
# ------------------------------------------------------------

RAW_PANCHANG_COLUMNS = [
    "valid_tithi",
    "valid_paksha",
    "valid_karana",
    "valid_nakshatra",
    "valid_yoga",
]

assert (
    list(feature_family_tables[60]["F4_raw"].columns[-5:])
    == RAW_PANCHANG_COLUMNS
)

print("\nRaw F4 categorical columns:")
for column in RAW_PANCHANG_COLUMNS:
    print(" -", column)


# ------------------------------------------------------------
# 7. Display a compact 60-minute example
# ------------------------------------------------------------

print("\nFirst 60-minute F4_raw modelling row:")

display(
    feature_family_tables[60]["F4_raw"][
        RAW_PANCHANG_COLUMNS
    ].head()
)


15-minute horizon:
  F0     shape = (131325, 118)
  F1     shape = (131325, 133)
  F2     shape = (131325, 138)
  F3     shape = (131325, 147)
  F4_raw shape = (131325, 152)

30-minute horizon:
  F0     shape = (131322, 118)
  F1     shape = (131322, 133)
  F2     shape = (131322, 138)
  F3     shape = (131322, 147)
  F4_raw shape = (131322, 152)

60-minute horizon:
  F0     shape = (131316, 118)
  F1     shape = (131316, 133)
  F2     shape = (131316, 138)
  F3     shape = (131316, 147)
  F4_raw shape = (131316, 152)

Nested F0 -> F1 -> F2 -> F3 -> F4_raw checks: PASSED

Raw F4 categorical columns:
 - valid_tithi
 - valid_paksha
 - valid_karana
 - valid_nakshatra
 - valid_yoga

First 60-minute F4_raw modelling row:


,valid_tithi,valid_paksha,valid_karana,valid_nakshatra,valid_yoga
Timestamp,,,,,
2018-06-02 00:00:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:05:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:10:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:15:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla
2018-06-02 00:20:00+00:00,19,Krishna,Bava,Purva Ashadha,Shukla


In [14]:
# ============================================================
# Step 13: Inspect canonical Panchang category vocabularies
# ============================================================

import src.vedic_calendar as vedic_calendar


# ------------------------------------------------------------
# Find module-level constants related to Panchang category
# names.
#
# We want the one-hot encoder to use the same canonical
# vocabulary as the validated calendar implementation rather
# than maintaining a second independent spelling/order list
# inside this notebook.
# ------------------------------------------------------------

candidate_constants = [
    name
    for name in dir(vedic_calendar)
    if any(
        keyword in name.upper()
        for keyword in [
            "KARANA",
            "NAKSHATRA",
            "YOGA",
        ]
    )
    and name.isupper()
]


print("Candidate Panchang constants:\n")

for name in candidate_constants:
    value = getattr(vedic_calendar, name)

    if isinstance(value, (list, tuple)):
        print(f"{name}:")
        print(value)
        print()

Candidate Panchang constants:

KARANA_SEQUENCE:
('Kimstughna', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Shakuni', 'Chatushpada', 'Naga')

MOVABLE_KARANAS:
('Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti')

NAKSHATRA_NAMES:
('Ashwini', 'Bharani', 'Krittika', 'Rohini', 'Mrigashirsha', 'Ardra', 'Punarvasu', 'Pushya', 'Ashlesha', 'Magha', 'Purva Phalguni', 'Uttara Phalguni', 'Hasta', 'Chitra', 'Swati', 'Vishakha', 'Anuradha', 'Jyeshtha', 'Mula', 'Purva Ashadha', 'Uttara Ashadh

In [15]:
# ============================================================
# Step 14: One-hot encode the Panchang F4 feature block
# ============================================================

# ------------------------------------------------------------
# 1. Define the fixed canonical category vocabularies
# ------------------------------------------------------------
#
# These vocabularies are NOT learned from the modelling data.
#
# They come from the deterministic Panchang system itself.
# This guarantees that:
#
#   - train / validation / test use the same categories;
#   - all forecast horizons use the same categories;
#   - F4 dimensionality is fixed;
#   - a temporarily absent category does not change columns.
# ------------------------------------------------------------

TITHI_CATEGORIES = tuple(range(1, 31))

PAKSHA_CATEGORIES = (
    "Shukla",
    "Krishna",
)

# KARANA_SEQUENCE contains repeated movable Karanas across the
# 60 half-Tithi slots.
#
# dict.fromkeys(...) removes repetitions while preserving the
# canonical first-appearance order.
KARANA_CATEGORIES = tuple(
    dict.fromkeys(vedic_calendar.KARANA_SEQUENCE)
)

NAKSHATRA_CATEGORIES = tuple(
    vedic_calendar.NAKSHATRA_NAMES
)

YOGA_CATEGORIES = tuple(
    vedic_calendar.YOGA_NAMES
)


# ------------------------------------------------------------
# 2. Verify canonical vocabulary sizes
# ------------------------------------------------------------

assert len(TITHI_CATEGORIES) == 30
assert len(PAKSHA_CATEGORIES) == 2
assert len(KARANA_CATEGORIES) == 11
assert len(NAKSHATRA_CATEGORIES) == 27
assert len(YOGA_CATEGORIES) == 27

TOTAL_PANCHANG_INDICATORS = (
    len(TITHI_CATEGORIES)
    + len(PAKSHA_CATEGORIES)
    + len(KARANA_CATEGORIES)
    + len(NAKSHATRA_CATEGORIES)
    + len(YOGA_CATEGORIES)
)

assert TOTAL_PANCHANG_INDICATORS == 97

print("Canonical Panchang vocabulary sizes:")
print(f"  Tithi:      {len(TITHI_CATEGORIES)}")
print(f"  Paksha:     {len(PAKSHA_CATEGORIES)}")
print(f"  Karana:     {len(KARANA_CATEGORIES)}")
print(f"  Nakshatra:  {len(NAKSHATRA_CATEGORIES)}")
print(f"  Yoga:       {len(YOGA_CATEGORIES)}")
print(f"  Total:      {TOTAL_PANCHANG_INDICATORS}")


# ------------------------------------------------------------
# 3. Inspect the derived canonical Karana vocabulary
# ------------------------------------------------------------

print("\nCanonical Karana categories:")
print(KARANA_CATEGORIES)


# ------------------------------------------------------------
# 4. Helper: fixed-vocabulary Panchang one-hot encoding
# ------------------------------------------------------------

def encode_panchang_fixed_vocabulary(
    df_panchang: pd.DataFrame,
) -> pd.DataFrame:
    """
    Convert the five raw forecast-valid Panchang columns into
    exactly 97 deterministic one-hot indicator columns.

    The category vocabularies are fixed in advance and are not
    learned from the supplied dataframe.
    """

    required_columns = [
        "valid_tithi",
        "valid_paksha",
        "valid_karana",
        "valid_nakshatra",
        "valid_yoga",
    ]

    assert list(df_panchang.columns) == required_columns, (
        "Unexpected raw Panchang column structure."
    )

    # --------------------------------------------------------
    # Before converting to pandas Categoricals, explicitly
    # verify that every observed value belongs to the canonical
    # vocabulary.
    #
    # This prevents an unexpected category from silently being
    # converted to NaN.
    # --------------------------------------------------------

    vocabulary_map = {
        "valid_tithi": TITHI_CATEGORIES,
        "valid_paksha": PAKSHA_CATEGORIES,
        "valid_karana": KARANA_CATEGORIES,
        "valid_nakshatra": NAKSHATRA_CATEGORIES,
        "valid_yoga": YOGA_CATEGORIES,
    }

    for column, categories in vocabulary_map.items():

        observed = set(df_panchang[column].unique())
        allowed = set(categories)

        unexpected = observed - allowed

        assert not unexpected, (
            f"Unexpected categories in {column}: "
            f"{sorted(unexpected)}"
        )

    # --------------------------------------------------------
    # Convert each raw variable to a Categorical with an
    # explicitly fixed category set.
    # --------------------------------------------------------

    df_categorical = pd.DataFrame(
        index=df_panchang.index
    )

    for column, categories in vocabulary_map.items():

        df_categorical[column] = pd.Categorical(
            df_panchang[column],
            categories=categories,
            ordered=False,
        )

    # --------------------------------------------------------
    # Full one-hot encoding.
    #
    # Because the Categorical vocabularies were explicitly
    # supplied above, pandas creates columns for every canonical
    # state even if a category were absent from this particular
    # modelling interval.
    # --------------------------------------------------------

    df_encoded = pd.get_dummies(
        df_categorical,
        prefix={
            "valid_tithi": "valid_tithi",
            "valid_paksha": "valid_paksha",
            "valid_karana": "valid_karana",
            "valid_nakshatra": "valid_nakshatra",
            "valid_yoga": "valid_yoga",
        },
        prefix_sep="_",
        dtype="int8",
    )

    assert df_encoded.shape[1] == 97, (
        f"Expected 97 Panchang indicators, "
        f"found {df_encoded.shape[1]}."
    )

    assert df_encoded.isna().sum().sum() == 0

    return df_encoded


# ------------------------------------------------------------
# 5. Encode F4 for each forecast horizon
# ------------------------------------------------------------

panchang_onehot_tables = {}

for horizon_minutes in FORECAST_HORIZONS:

    df_raw_panchang = panchang_feature_tables[
        horizon_minutes
    ]

    df_onehot = encode_panchang_fixed_vocabulary(
        df_raw_panchang
    )

    panchang_onehot_tables[horizon_minutes] = df_onehot


# ------------------------------------------------------------
# 6. Verify each one-hot group activates exactly once per row
# ------------------------------------------------------------
#
# Every modelling row must contain:
#
#   exactly one Tithi
#   exactly one Paksha
#   exactly one Karana
#   exactly one Nakshatra
#   exactly one Yoga
#
# Therefore the 97-column Panchang block must contain exactly
# five active indicators per row.
# ------------------------------------------------------------

for horizon_minutes, df_onehot in panchang_onehot_tables.items():

    row_sums = df_onehot.sum(axis=1)

    assert (row_sums == 5).all(), (
        f"{horizon_minutes}-minute Panchang one-hot rows "
        "do not contain exactly five active indicators."
    )

    print(
        f"\n{horizon_minutes}-minute one-hot Panchang:"
    )
    print(f"  Shape: {df_onehot.shape}")
    print(
        f"  Active indicators per row: "
        f"{row_sums.min()} to {row_sums.max()}"
    )


# ------------------------------------------------------------
# 7. Construct the final F4 feature table
# ------------------------------------------------------------
#
# F4 is:
#
#     F3
#     +
#     97 one-hot Panchang indicators
#
# The five raw categorical columns are NOT included as separate
# model inputs.
# ------------------------------------------------------------

for horizon_minutes in FORECAST_HORIZONS:

    df_f3 = feature_family_tables[
        horizon_minutes
    ]["F3"]

    df_panchang_encoded = panchang_onehot_tables[
        horizon_minutes
    ]

    df_f4 = pd.concat(
        [
            df_f3,
            df_panchang_encoded,
        ],
        axis=1,
    )

    assert df_f4.shape[1] == 244, (
        f"{horizon_minutes}-minute F4 expected 244 "
        f"features, found {df_f4.shape[1]}."
    )

    assert df_f4.isna().sum().sum() == 0

    assert df_f4.index.equals(df_f3.index)

    feature_family_tables[
        horizon_minutes
    ]["F4"] = df_f4


# ------------------------------------------------------------
# 8. Final F0-F4 dimensionality check
# ------------------------------------------------------------

EXPECTED_FINAL_FEATURE_COUNTS = {
    "F0": 118,
    "F1": 133,
    "F2": 138,
    "F3": 147,
    "F4": 244,
}

print("\nFinal model-input feature dimensions:")

for horizon_minutes in FORECAST_HORIZONS:

    print(f"\n{horizon_minutes}-minute horizon:")

    for family_name, expected_count in (
        EXPECTED_FINAL_FEATURE_COUNTS.items()
    ):

        df_family = feature_family_tables[
            horizon_minutes
        ][family_name]

        assert df_family.shape[1] == expected_count

        print(
            f"  {family_name}: "
            f"{df_family.shape}"
        )


# ------------------------------------------------------------
# 9. Confirm final nested F3 -> F4 structure
# ------------------------------------------------------------

for horizon_minutes in FORECAST_HORIZONS:

    df_f3 = feature_family_tables[
        horizon_minutes
    ]["F3"]

    df_f4 = feature_family_tables[
        horizon_minutes
    ]["F4"]

    assert list(
        df_f4.columns[:147]
    ) == list(
        df_f3.columns
    )

print("\nFinal F3 -> F4 nesting check: PASSED")


# ------------------------------------------------------------
# 10. Inspect the first 60-minute Panchang encoding
# ------------------------------------------------------------

df_onehot_60 = panchang_onehot_tables[60]

first_active_columns = (
    df_onehot_60
    .iloc[0]
    .loc[lambda s: s == 1]
    .index
    .tolist()
)

print(
    "\nActive Panchang indicators for the first "
    "60-minute modelling row:"
)

for column in first_active_columns:
    print(" -", column)

Canonical Panchang vocabulary sizes:
  Tithi:      30
  Paksha:     2
  Karana:     11
  Nakshatra:  27
  Yoga:       27
  Total:      97

Canonical Karana categories:
('Kimstughna', 'Bava', 'Balava', 'Kaulava', 'Taitila', 'Gara', 'Vanija', 'Vishti', 'Shakuni', 'Chatushpada', 'Naga')

15-minute one-hot Panchang:
  Shape: (131325, 97)
  Active indicators per row: 5 to 5

30-minute one-hot Panchang:
  Shape: (131322, 97)
  Active indicators per row: 5 to 5

60-minute one-hot Panchang:
  Shape: (131316, 97)
  Active indicators per row: 5 to 5

Final model-input feature dimensions:

15-minute horizon:
  F0: (131325, 118)
  F1: (131325, 133)
  F2: (131325, 138)
  F3: (131325, 147)
  F4: (131325, 244)

30-minute horizon:
  F0: (131322, 118)
  F1: (131322, 133)
  F2: (131322, 138)
  F3: (131322, 147)
  F4: (131322, 244)

60-minute horizon:
  F0: (131316, 118)
  F1: (131316, 133)
  F2: (131316, 138)
  F3: (131316, 147)
  F4: (131316, 244)

Final F3 -> F4 nesting check: PASSED

Active Panchang 

In [16]:
# ============================================================
# Step 15: Build model-ready X / y / split bundles
# ============================================================

# ------------------------------------------------------------
# 1. Storage convention
# ------------------------------------------------------------
#
# modelling_bundles[horizon] contains:
#
#   "X"              -> dictionary of F0-F4 feature matrices
#   "y"              -> future P_Solar target
#   "metadata"       -> issue time, valid time, split, horizon
#   "train_mask"     -> chronological training rows
#   "validation_mask"-> chronological validation rows
#   "test_mask"      -> chronological test rows
#   "daylight_mask"  -> solar daylight at forecast-valid time
#
# Nothing is scaled or fitted here.
# ------------------------------------------------------------

modelling_bundles = {}


# ------------------------------------------------------------
# 2. Construct one bundle per forecast horizon
# ------------------------------------------------------------

for horizon_minutes in FORECAST_HORIZONS:

    df_base = base_modelling_tables[horizon_minutes]

    # --------------------------------------------------------
    # Target
    # --------------------------------------------------------
    #
    # Keep the target as a Series indexed by forecast issue time.
    # --------------------------------------------------------

    y = (
        df_base["target_P_Solar_kW"]
        .copy()
        .rename("target_P_Solar_kW")
    )


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    metadata = df_base[
        [
            "issue_time",
            "valid_time",
            "horizon_minutes",
            "split",
        ]
    ].copy()


    # --------------------------------------------------------
    # Chronological split masks
    # --------------------------------------------------------

    train_mask = metadata["split"].eq("train")

    validation_mask = metadata["split"].eq("validation")

    test_mask = metadata["split"].eq("test")


    # --------------------------------------------------------
    # Daylight mask
    # --------------------------------------------------------
    #
    # Daylight is defined deterministically from the F2 solar
    # geometry at forecast-valid time:
    #
    #     solar elevation > 0 degrees
    #
    # The reusable F2 pipeline already expresses this through:
    #
    #     valid_solar_daylight
    #
    # This avoids defining daylight using the observed target.
    # --------------------------------------------------------

    daylight_mask = (
        solar_feature_tables[horizon_minutes][
            "valid_solar_daylight"
        ]
        .eq(1)
    )


    # --------------------------------------------------------
    # F0-F4 model-input matrices
    # --------------------------------------------------------

    X_tables = {
        family_name: (
            feature_family_tables[horizon_minutes][family_name]
            .copy()
        )
        for family_name in ["F0", "F1", "F2", "F3", "F4"]
    }


    # --------------------------------------------------------
    # 3. Alignment checks
    # --------------------------------------------------------

    canonical_index = df_base.index

    assert y.index.equals(canonical_index)

    assert metadata.index.equals(canonical_index)

    assert daylight_mask.index.equals(canonical_index)

    for family_name, X in X_tables.items():

        assert X.index.equals(canonical_index), (
            f"{horizon_minutes}-minute {family_name}: "
            "X/y index alignment failed."
        )

        assert len(X) == len(y)

        assert X.isna().sum().sum() == 0


    # Target must also be complete.
    assert y.isna().sum() == 0


    # Every row must belong to exactly one chronological split.
    split_membership_count = (
        train_mask.astype(int)
        + validation_mask.astype(int)
        + test_mask.astype(int)
    )

    assert (split_membership_count == 1).all(), (
        f"{horizon_minutes}-minute split membership error."
    )


    # --------------------------------------------------------
    # 4. Store the complete bundle
    # --------------------------------------------------------

    modelling_bundles[horizon_minutes] = {
        "X": X_tables,
        "y": y,
        "metadata": metadata,
        "train_mask": train_mask,
        "validation_mask": validation_mask,
        "test_mask": test_mask,
        "daylight_mask": daylight_mask,
    }


# ------------------------------------------------------------
# 5. Report the finalized modelling structure
# ------------------------------------------------------------

for horizon_minutes, bundle in modelling_bundles.items():

    print(f"\n{horizon_minutes}-minute horizon:")

    print(f"  Total rows:       {len(bundle['y'])}")
    print(
        f"  Train rows:       "
        f"{bundle['train_mask'].sum()}"
    )
    print(
        f"  Validation rows:  "
        f"{bundle['validation_mask'].sum()}"
    )
    print(
        f"  Test rows:        "
        f"{bundle['test_mask'].sum()}"
    )
    print(
        f"  Daylight rows:    "
        f"{bundle['daylight_mask'].sum()}"
    )

    print("  Feature shapes:")

    for family_name in ["F0", "F1", "F2", "F3", "F4"]:
        print(
            f"    {family_name}: "
            f"{bundle['X'][family_name].shape}"
        )


# ------------------------------------------------------------
# 6. Explicit 60-minute target alignment audit
# ------------------------------------------------------------

bundle_60 = modelling_bundles[60]

first_issue_60 = bundle_60["metadata"]["issue_time"].iloc[0]
first_valid_60 = bundle_60["metadata"]["valid_time"].iloc[0]
first_target_60 = bundle_60["y"].iloc[0]

expected_target_60 = df_clean.loc[
    first_valid_60,
    "P_Solar[kW]",
]

assert first_target_60 == expected_target_60

print("\n60-minute first-row audit:")
print(f"  Issue time:       {first_issue_60}")
print(f"  Valid time:       {first_valid_60}")
print(f"  Stored target:    {first_target_60}")
print(f"  Source P_Solar:   {expected_target_60}")

print("\nModel-ready bundle construction: PASSED")


15-minute horizon:
  Total rows:       131325
  Train rows:       87261
  Validation rows:  17568
  Test rows:        26496
  Daylight rows:    70543
  Feature shapes:
    F0: (131325, 118)
    F1: (131325, 133)
    F2: (131325, 138)
    F3: (131325, 147)
    F4: (131325, 244)

30-minute horizon:
  Total rows:       131322
  Train rows:       87258
  Validation rows:  17568
  Test rows:        26496
  Daylight rows:    70543
  Feature shapes:
    F0: (131322, 118)
    F1: (131322, 133)
    F2: (131322, 138)
    F3: (131322, 147)
    F4: (131322, 244)

60-minute horizon:
  Total rows:       131316
  Train rows:       87252
  Validation rows:  17568
  Test rows:        26496
  Daylight rows:    70543
  Feature shapes:
    F0: (131316, 118)
    F1: (131316, 133)
    F2: (131316, 138)
    F3: (131316, 147)
    F4: (131316, 244)

60-minute first-row audit:
  Issue time:       2018-06-02 00:00:00+00:00
  Valid time:       2018-06-02 01:00:00+00:00
  Stored target:    0.0
  Source P_Solar:  

# Phase 2A control representations

The controls operate **only on the 97-column Panchang block** appended to F3.

Two dimension-matched null families are used:

1. **Circular-shift controls** preserve dimensionality, one-hot structure,
   category relationships within each row, and temporal persistence, while
   breaking correct calendar alignment.
2. **Joint-row permutation controls** preserve dimensionality and the empirical
   joint distribution of the five Panchang states within each chronological
   split, but destroy temporal alignment and persistence.

A reference-coded F4 variant is also constructed to check whether full
fixed-vocabulary one-hot redundancy materially affects results, especially for
linear models.

Finally, five add-one-family variants screen Tithi, Paksha, Karana, Nakshatra,
and Yoga separately on top of F3.


In [17]:

# ============================================================
# Step P2-1: Identify the five Panchang one-hot groups
# ============================================================

PANCHANG_GROUP_PREFIXES = {
    "tithi": "valid_tithi_",
    "paksha": "valid_paksha_",
    "karana": "valid_karana_",
    "nakshatra": "valid_nakshatra_",
    "yoga": "valid_yoga_",
}

EXPECTED_GROUP_SIZES = {
    "tithi": 30,
    "paksha": 2,
    "karana": 11,
    "nakshatra": 27,
    "yoga": 27,
}

PANCHANG_GROUP_COLUMNS = {}

reference_onehot = panchang_onehot_tables[60]

for family, prefix in PANCHANG_GROUP_PREFIXES.items():
    columns = [
        column
        for column in reference_onehot.columns
        if column.startswith(prefix)
    ]

    assert len(columns) == EXPECTED_GROUP_SIZES[family], (
        f"{family}: expected {EXPECTED_GROUP_SIZES[family]} columns, "
        f"found {len(columns)}."
    )

    PANCHANG_GROUP_COLUMNS[family] = columns

all_group_columns = [
    column
    for family in [
        "tithi",
        "paksha",
        "karana",
        "nakshatra",
        "yoga",
    ]
    for column in PANCHANG_GROUP_COLUMNS[family]
]

assert all_group_columns == list(reference_onehot.columns)
assert len(all_group_columns) == 97

print("Panchang group audit:")
for family, columns in PANCHANG_GROUP_COLUMNS.items():
    print(f"  {family:<10}: {len(columns):>2} indicators")
print("  total     :", len(all_group_columns))


Panchang group audit:
  tithi     : 30 indicators
  paksha    :  2 indicators
  karana    : 11 indicators
  nakshatra : 27 indicators
  yoga      : 27 indicators
  total     : 97


In [18]:
# ============================================================
# Step P2-2: Splitwise null-control builders
# ============================================================

def splitwise_joint_permutation(
    block: pd.DataFrame,
    metadata: pd.DataFrame,
    seed: int,
) -> pd.DataFrame:
    """
    Jointly permute complete 97-column Panchang rows *within*
    each chronological split.

    This preserves:
      - 97 dimensions;
      - exactly five active indicators per row;
      - empirical category frequencies within each split;
      - cross-family combinations within a permuted row.

    It destroys:
      - correct timestamp alignment;
      - temporal sequence/persistence.
    """
    out = pd.DataFrame(
        index=block.index,
        columns=block.columns,
        dtype="int8",
    )

    split_names = ["train", "validation", "test"]

    for split_offset, split_name in enumerate(split_names):
        split_mask = metadata["split"].eq(split_name)
        split_index = metadata.index[split_mask]

        values = block.loc[split_index].to_numpy(copy=True)

        rng = np.random.default_rng(seed + split_offset)
        order = rng.permutation(len(values))

        out.loc[split_index, :] = values[order]

    return out.astype("int8")


def splitwise_circular_shift(
    block: pd.DataFrame,
    metadata: pd.DataFrame,
    shift_days: int,
    resolution_minutes: int = 5,
) -> pd.DataFrame:
    """
    Circularly shift complete Panchang rows within each split.

    This preserves:
      - dimensionality;
      - one-hot row structure;
      - cross-family state combinations;
      - local temporal persistence/transitions.

    It breaks:
      - correct astronomical/calendar alignment to the target time.
    """
    steps_per_day = (24 * 60) // resolution_minutes
    shift_steps = shift_days * steps_per_day

    out = pd.DataFrame(
        index=block.index,
        columns=block.columns,
        dtype="int8",
    )

    for split_name in ["train", "validation", "test"]:
        split_mask = metadata["split"].eq(split_name)
        split_index = metadata.index[split_mask]

        values = block.loc[split_index].to_numpy(copy=True)

        effective_shift = shift_steps % len(values)

        shifted = np.roll(
            values,
            shift=effective_shift,
            axis=0,
        )

        out.loc[split_index, :] = shifted

    return out.astype("int8")


In [19]:
# ============================================================
# Step P2-3: Construct all representation variants
# ============================================================

SHIFT_DAYS = [17, 37, 61]
PERMUTATION_SEEDS = [1301, 1302, 1303]

phase2_variant_tables = {}
phase2_variant_manifest_rows = []

for horizon_minutes in [15, 30, 60]:

    bundle = modelling_bundles[horizon_minutes]

    X_f3 = bundle["X"]["F3"].copy()
    X_f4 = bundle["X"]["F4"].copy()
    metadata = bundle["metadata"].copy()

    panchang_real = panchang_onehot_tables[horizon_minutes].copy()

    variants = {
        "F3": X_f3,
        "F4_real": X_f4,
    }

    # --------------------------------------------------------
    # Dimension-matched, temporally shifted structured nulls
    # --------------------------------------------------------
    for shift_days in SHIFT_DAYS:
        shifted_block = splitwise_circular_shift(
            panchang_real,
            metadata,
            shift_days=shift_days,
        )

        variants[f"F4_shift_{shift_days}d"] = pd.concat(
            [X_f3, shifted_block],
            axis=1,
        )

    # --------------------------------------------------------
    # Dimension-matched joint-row permutation nulls
    # --------------------------------------------------------
    for seed in PERMUTATION_SEEDS:
        permuted_block = splitwise_joint_permutation(
            panchang_real,
            metadata,
            seed=seed,
        )

        variants[f"F4_perm_{seed}"] = pd.concat(
            [X_f3, permuted_block],
            axis=1,
        )

    # --------------------------------------------------------
    # Reference-coded F4:
    # drop one canonical column from each categorical family.
    # This is a coding-sensitivity analysis, NOT a
    # dimension-matched null.
    # --------------------------------------------------------
    reference_drop_columns = [
        PANCHANG_GROUP_COLUMNS[family][0]
        for family in [
            "tithi",
            "paksha",
            "karana",
            "nakshatra",
            "yoga",
        ]
    ]

    panchang_reference = panchang_real.drop(
        columns=reference_drop_columns
    )

    assert panchang_reference.shape[1] == 92

    variants["F4_reference"] = pd.concat(
        [X_f3, panchang_reference],
        axis=1,
    )

    # --------------------------------------------------------
    # Add-one-family component screen
    # --------------------------------------------------------
    for family in [
        "tithi",
        "paksha",
        "karana",
        "nakshatra",
        "yoga",
    ]:
        family_block = panchang_real[
            PANCHANG_GROUP_COLUMNS[family]
        ]

        variants[f"F3_plus_{family}"] = pd.concat(
            [X_f3, family_block],
            axis=1,
        )

    phase2_variant_tables[horizon_minutes] = variants

    for variant_name, X_variant in variants.items():
        phase2_variant_manifest_rows.append(
            {
                "horizon_minutes": horizon_minutes,
                "variant": variant_name,
                "n_rows": len(X_variant),
                "n_features": X_variant.shape[1],
            }
        )


phase2_variant_manifest = pd.DataFrame(
    phase2_variant_manifest_rows
)

display(
    phase2_variant_manifest.sort_values(
        ["horizon_minutes", "variant"]
    )
)


,horizon_minutes,variant,n_rows,n_features
0,15,F3,131325,147
11,15,F3_plus_karana,131325,158
12,15,F3_plus_nakshatra,131325,174
10,15,F3_plus_paksha,131325,149
9,15,F3_plus_tithi,131325,177
13,15,F3_plus_yoga,131325,174
5,15,F4_perm_1301,131325,244
6,15,F4_perm_1302,131325,244
7,15,F4_perm_1303,131325,244
1,15,F4_real,131325,244


In [20]:
# ============================================================
# Step P2-4: Strong representation-control audit
# ============================================================

CORE_244_VARIANTS = [
    "F4_real",
    "F4_shift_17d",
    "F4_shift_37d",
    "F4_shift_61d",
    "F4_perm_1301",
    "F4_perm_1302",
    "F4_perm_1303",
]

EXPECTED_COMPONENT_COUNTS = {
    "F3_plus_tithi": 147 + 30,
    "F3_plus_paksha": 147 + 2,
    "F3_plus_karana": 147 + 11,
    "F3_plus_nakshatra": 147 + 27,
    "F3_plus_yoga": 147 + 27,
}

for horizon_minutes in [15, 30, 60]:

    variants = phase2_variant_tables[horizon_minutes]
    X_f3 = variants["F3"]

    assert X_f3.shape[1] == 147
    assert variants["F4_real"].shape[1] == 244
    assert variants["F4_reference"].shape[1] == 239

    # All dimension-matched core variants must preserve the
    # exact 147-column F3 prefix and the exact 97 F4 column names.
    for variant_name in CORE_244_VARIANTS:
        X = variants[variant_name]

        assert X.shape[1] == 244
        assert list(X.columns[:147]) == list(X_f3.columns)
        assert list(X.columns[147:]) == list(
            panchang_onehot_tables[horizon_minutes].columns
        )
        assert X.index.equals(X_f3.index)
        assert X.isna().sum().sum() == 0

        row_sums = X.iloc[:, 147:].sum(axis=1)
        assert (row_sums == 5).all(), (
            f"{horizon_minutes} / {variant_name}: "
            "expected exactly five active Panchang indicators per row."
        )

    # Null variants must not accidentally equal the real block.
    real_block = variants["F4_real"].iloc[:, 147:]

    for variant_name in CORE_244_VARIANTS:
        if variant_name == "F4_real":
            continue

        candidate_block = variants[variant_name].iloc[:, 147:]

        assert not candidate_block.equals(real_block), (
            f"{horizon_minutes} / {variant_name}: "
            "null block unexpectedly equals real Panchang block."
        )

    # Component-screen dimensions.
    for variant_name, expected_count in EXPECTED_COMPONENT_COUNTS.items():
        assert variants[variant_name].shape[1] == expected_count
        assert list(
            variants[variant_name].columns[:147]
        ) == list(X_f3.columns)

print("PHASE 2A REPRESENTATION CONTROL AUDIT: PASSED")


PHASE 2A REPRESENTATION CONTROL AUDIT: PASSED


In [21]:
# ============================================================
# Step P2-5: Persist the frozen variant manifest
# ============================================================

manifest_path = (
    PHASE2_RESULTS_DIR
    / "phase2_representation_variant_manifest.csv"
)

phase2_variant_manifest.to_csv(
    manifest_path,
    index=False,
)

print("Saved:", manifest_path)
print("\nNotebook 13 representation setup is frozen.")
print("STOP HERE before adding or modifying model experiments.")


Saved: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_representation_variant_manifest.csv

Notebook 13 representation setup is frozen.
STOP HERE before adding or modifying model experiments.


# Stop point

If all cells above run successfully and print:

`PHASE 2A REPRESENTATION CONTROL AUDIT: PASSED`

the representation protocol is frozen.

**Do not change the controls after looking at model results.**

The next section will fit the pre-specified architecture strata using the
same F4-selected model configuration across every Phase-2 variant.


# Phase 2A — Frozen model-configuration recovery

Before any Phase-2 model fitting, recover the exact Phase-1 validation-selected
configuration artifacts for the four pre-specified architecture strata:

- Ridge
- LightGBM
- LSTM
- TCN

## Why this checkpoint exists

Phase 1 allowed feature families to undergo their original validation-based
selection protocol. Phase 2 asks a different mechanistic question.

For Phase 2, each model/horizon will use the **F4-selected configuration
unchanged across F3, real F4, null controls, and component screens**. This
prevents representation-specific retuning from becoming a confound.

This cell performs **no training and no Phase-2 TEST evaluation**.


In [22]:
# ============================================================
# Step P2-6: Recover frozen Phase-1 configuration artifacts
# ============================================================

import json
from pathlib import Path

import pandas as pd

NB09_RESULTS = (
    PROJECT_ROOT
    / "results"
    / "notebook09_primary_experiment"
)

NB10_RESULTS = (
    PROJECT_ROOT
    / "results"
    / "notebook10_secondary_ml_models"
)

NB11_RESULTS = (
    PROJECT_ROOT
    / "results"
    / "notebook11_deep_learning_models"
)

PHASE2_CONFIG_ARTIFACTS = {
    "ridge_frozen_selection":
        NB09_RESULTS / "frozen_model_selection.csv",

    "lightgbm_frozen_selection":
        NB10_RESULTS / "lightgbm_frozen_selection.csv",

    "lightgbm_selected_configs":
        NB10_RESULTS / "lightgbm_selected_configs.json",

    "lstm_frozen_epochs":
        NB11_RESULTS / "lstm_frozen_epoch_selection.csv",

    # NOTE:
    # Notebook 11 froze LSTM architecture as RNN_C1 in-notebook,
    # but did not persist a separate lstm_selected_architecture.json.
    # We therefore reconstruct that already-frozen configuration
    # explicitly in the next cell from the canonical Notebook-11
    # architecture definition. This is NOT a new selection.

    "tcn_frozen_epochs":
        NB11_RESULTS / "tcn_frozen_epochs.json",

    "tcn_selected_architecture":
        NB11_RESULTS / "tcn_selected_architecture.json",
}

for artifact_name, artifact_path in PHASE2_CONFIG_ARTIFACTS.items():
    assert artifact_path.is_file(), (
        f"Missing required frozen Phase-1 artifact: "
        f"{artifact_name} -> {artifact_path}"
    )

print("Frozen Phase-1 configuration artifacts: ALL FOUND")
for artifact_name, artifact_path in PHASE2_CONFIG_ARTIFACTS.items():
    print(f"  {artifact_name}: {artifact_path.name}")

print(
    "\nLSTM architecture note: "
    "Notebook 11 froze RNN_C1 in-notebook; "
    "no separate lstm_selected_architecture.json was written."
)


Frozen Phase-1 configuration artifacts: ALL FOUND
  ridge_frozen_selection: frozen_model_selection.csv
  lightgbm_frozen_selection: lightgbm_frozen_selection.csv
  lightgbm_selected_configs: lightgbm_selected_configs.json
  lstm_frozen_epochs: lstm_frozen_epoch_selection.csv
  tcn_frozen_epochs: tcn_frozen_epochs.json
  tcn_selected_architecture: tcn_selected_architecture.json

LSTM architecture note: Notebook 11 froze RNN_C1 in-notebook; no separate lstm_selected_architecture.json was written.


In [23]:
# ============================================================
# Step P2-7: Load, isolate, and freeze F4-selected configurations
# ============================================================

ridge_frozen_selection = pd.read_csv(
    PHASE2_CONFIG_ARTIFACTS["ridge_frozen_selection"]
)

lightgbm_frozen_selection = pd.read_csv(
    PHASE2_CONFIG_ARTIFACTS["lightgbm_frozen_selection"]
)

lstm_frozen_epochs = pd.read_csv(
    PHASE2_CONFIG_ARTIFACTS["lstm_frozen_epochs"]
)

with open(
    PHASE2_CONFIG_ARTIFACTS["lightgbm_selected_configs"],
    "r",
    encoding="utf-8",
) as f:
    lightgbm_selected_configs = json.load(f)

with open(
    PHASE2_CONFIG_ARTIFACTS["tcn_frozen_epochs"],
    "r",
    encoding="utf-8",
) as f:
    tcn_frozen_epochs = json.load(f)

with open(
    PHASE2_CONFIG_ARTIFACTS["tcn_selected_architecture"],
    "r",
    encoding="utf-8",
) as f:
    tcn_selected_architecture = json.load(f)


# ------------------------------------------------------------
# Canonical frozen LSTM architecture from Notebook 11
# ------------------------------------------------------------
lstm_selected_architecture = {
    "candidate_id": "RNN_C1",
    "units": 32,
    "n_recurrent_layers": 1,
    "dropout": 0.0,
    "fusion_dense_units": 32,
}


# ------------------------------------------------------------
# Ridge: isolate ONLY Ridge + F4 rows
# ------------------------------------------------------------
ridge_f4_selection = (
    ridge_frozen_selection.loc[
        ridge_frozen_selection["model"].eq("Ridge")
        &
        ridge_frozen_selection["feature_family"].eq("F4")
    ]
    .copy()
    .sort_values("horizon_minutes")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# LightGBM: isolate F4 rows
# ------------------------------------------------------------
lightgbm_f4_selection = (
    lightgbm_frozen_selection.loc[
        lightgbm_frozen_selection["feature_family"].eq("F4")
    ]
    .copy()
    .sort_values("horizon_minutes")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# LSTM: isolate F4 frozen epochs
# ------------------------------------------------------------
lstm_f4_epochs = (
    lstm_frozen_epochs.loc[
        lstm_frozen_epochs["feature_family"].eq("F4")
    ]
    .copy()
    .sort_values("horizon_minutes")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# TCN: isolate F4 frozen epochs
# ------------------------------------------------------------
tcn_f4_epoch_map = {
    int(str(key).split("_")[0]): int(value)
    for key, value in tcn_frozen_epochs.items()
    if str(key).endswith("_F4")
}


# ------------------------------------------------------------
# Build one exact Phase-2 frozen configuration registry.
# This registry is written BEFORE any Phase-2 model fitting.
# ------------------------------------------------------------

PHASE2_FROZEN_MODEL_CONFIGS = {
    "Ridge": {},
    "LightGBM": {},
    "LSTM": {},
    "TCN": {},
}

for horizon in [15, 30, 60]:

    # Ridge
    ridge_row = ridge_f4_selection.loc[
        ridge_f4_selection["horizon_minutes"].eq(horizon)
    ].iloc[0]

    # The Phase-1 selected_parameters string is expected to be alpha=...
    ridge_parameter_text = str(
        ridge_row["selected_parameters"]
    ).strip()

    assert ridge_parameter_text.startswith("alpha=")

    ridge_alpha = float(
        ridge_parameter_text.split("=", 1)[1]
    )

    PHASE2_FROZEN_MODEL_CONFIGS["Ridge"][horizon] = {
        "alpha": ridge_alpha,
    }

    # LightGBM
    # JSON object keys become strings after json.load().
    lightgbm_horizon_key = str(horizon)

    lightgbm_f4_config = (
        lightgbm_selected_configs[
            lightgbm_horizon_key
        ][
            "F4"
        ]
        .copy()
    )

    PHASE2_FROZEN_MODEL_CONFIGS[
        "LightGBM"
    ][
        horizon
    ] = lightgbm_f4_config

    # LSTM
    lstm_epoch_row = lstm_f4_epochs.loc[
        lstm_f4_epochs["horizon_minutes"].eq(horizon)
    ].iloc[0]

    PHASE2_FROZEN_MODEL_CONFIGS["LSTM"][horizon] = {
        "architecture":
            lstm_selected_architecture.copy(),

        "epochs":
            int(
                lstm_epoch_row[
                    "best_epoch"
                ]
            ),
    }

    # TCN
    PHASE2_FROZEN_MODEL_CONFIGS["TCN"][horizon] = {
        "architecture":
            tcn_selected_architecture[
                "selected_architecture"
            ].copy(),

        "epochs":
            int(
                tcn_f4_epoch_map[
                    horizon
                ]
            ),
    }


print("=" * 90)
print("RIDGE — exact F4-selected Phase-1 rows")
print("=" * 90)
display(ridge_f4_selection)


print("\n" + "=" * 90)
print("LIGHTGBM — exact F4-selected Phase-1 rows")
print("=" * 90)
display(lightgbm_f4_selection)


print("\n" + "=" * 90)
print("LSTM — exact F4-selected frozen epochs")
print("=" * 90)
display(lstm_f4_epochs)


print("\n" + "=" * 90)
print("FROZEN PHASE-2 MODEL CONFIGURATION REGISTRY")
print("=" * 90)
print(
    json.dumps(
        PHASE2_FROZEN_MODEL_CONFIGS,
        indent=2,
    )
)


RIDGE — exact F4-selected Phase-1 rows


,model,horizon_minutes,feature_family,selected_parameters,validation_MAE_kW
0,Ridge,15,F4,alpha=0.01,0.306413
1,Ridge,30,F4,alpha=0.01,0.388180
2,Ridge,60,F4,alpha=0.01,0.509421



LIGHTGBM — exact F4-selected Phase-1 rows


,horizon_minutes,feature_family,n_features,n_estimators,learning_rate,num_leaves,subsample,subsample_freq,colsample_bytree,min_child_samples,validation_mae,validation_rmse
0,15,F4,244,300,0.03,15,0.8,1,0.8,20,0.269531,0.583488
1,30,F4,244,300,0.03,15,0.8,1,0.8,20,0.315966,0.645094
2,60,F4,244,300,0.03,15,0.8,1,0.8,20,0.381773,0.730913



LSTM — exact F4-selected frozen epochs


,horizon_minutes,feature_family,best_epoch,auxiliary_dim,trainable_parameter_count,validation_MAE_kW,validation_RMSE_kW,validation_R2,epoch_selection_source
0,15,F4,7,127,10529,0.293711,0.638679,0.891457,stage2_validation_run
1,30,F4,1,127,10529,0.382804,0.756728,0.847624,stage2_validation_run
2,60,F4,2,127,10529,0.464525,0.860281,0.803067,stage2_validation_run



FROZEN PHASE-2 MODEL CONFIGURATION REGISTRY
{
  "Ridge": {
    "15": {
      "alpha": 0.01
    },
    "30": {
      "alpha": 0.01
    },
    "60": {
      "alpha": 0.01
    }
  },
  "LightGBM": {
    "15": {
      "boosting_type": "gbdt",
      "colsample_bytree": 0.8,
      "deterministic": true,
      "force_col_wise": true,
      "learning_rate": 0.03,
      "max_depth": -1,
      "min_child_samples": 20,
      "n_estimators": 300,
      "n_jobs": -1,
      "num_leaves": 15,
      "objective": "regression",
      "random_state": 42,
      "reg_alpha": 0.0,
      "reg_lambda": 1.0,
      "subsample": 0.8,
      "subsample_freq": 1,
      "verbosity": -1
    },
    "30": {
      "boosting_type": "gbdt",
      "colsample_bytree": 0.8,
      "deterministic": true,
      "force_col_wise": true,
      "learning_rate": 0.03,
      "max_depth": -1,
      "min_child_samples": 20,
      "n_estimators": 300,
      "n_jobs": -1,
      "num_leaves": 15,
      "objective": "regression",
      "r

In [24]:
# ============================================================
# Step P2-8: Strong frozen-configuration audit + persistence
# ============================================================

EXPECTED_HORIZONS = {15, 30, 60}


# ------------------------------------------------------------
# 1. Exactly one F4 selection per horizon for each model
# ------------------------------------------------------------

assert len(ridge_f4_selection) == 3
assert set(
    ridge_f4_selection["horizon_minutes"].astype(int)
) == EXPECTED_HORIZONS

assert (
    ridge_f4_selection["model"]
    .eq("Ridge")
    .all()
)

assert (
    ridge_f4_selection["feature_family"]
    .eq("F4")
    .all()
)


assert len(lightgbm_f4_selection) == 3
assert set(
    lightgbm_f4_selection[
        "horizon_minutes"
    ].astype(int)
) == EXPECTED_HORIZONS

assert (
    lightgbm_f4_selection[
        "feature_family"
    ]
    .eq("F4")
    .all()
)


assert len(lstm_f4_epochs) == 3
assert set(
    lstm_f4_epochs[
        "horizon_minutes"
    ].astype(int)
) == EXPECTED_HORIZONS

assert (
    lstm_f4_epochs[
        "feature_family"
    ]
    .eq("F4")
    .all()
)


assert set(
    tcn_f4_epoch_map.keys()
) == EXPECTED_HORIZONS


# ------------------------------------------------------------
# 2. Frozen architecture identities
# ------------------------------------------------------------

assert lstm_selected_architecture == {
    "candidate_id": "RNN_C1",
    "units": 32,
    "n_recurrent_layers": 1,
    "dropout": 0.0,
    "fusion_dense_units": 32,
}

assert (
    tcn_selected_architecture[
        "selected_candidate_id"
    ]
    ==
    "TCN_C1"
)


# ------------------------------------------------------------
# 3. Cross-check LightGBM JSON against the human-readable
#    frozen-selection CSV for every F4 horizon
# ------------------------------------------------------------

for horizon in [15, 30, 60]:

    csv_row = lightgbm_f4_selection.loc[
        lightgbm_f4_selection[
            "horizon_minutes"
        ].eq(horizon)
    ].iloc[0]

    config = (
        PHASE2_FROZEN_MODEL_CONFIGS[
            "LightGBM"
        ][
            horizon
        ]
    )

    assert (
        int(config["n_estimators"])
        ==
        int(csv_row["n_estimators"])
    )

    assert np.isclose(
        float(config["learning_rate"]),
        float(csv_row["learning_rate"]),
    )

    assert (
        int(config["num_leaves"])
        ==
        int(csv_row["num_leaves"])
    )

    assert np.isclose(
        float(config["subsample"]),
        float(csv_row["subsample"]),
    )

    assert np.isclose(
        float(config["colsample_bytree"]),
        float(csv_row["colsample_bytree"]),
    )

    assert (
        int(config["min_child_samples"])
        ==
        int(csv_row["min_child_samples"])
    )


# ------------------------------------------------------------
# 4. Known Phase-1 F4 settings recovered exactly
# ------------------------------------------------------------

assert all(
    np.isclose(
        PHASE2_FROZEN_MODEL_CONFIGS[
            "Ridge"
        ][
            horizon
        ][
            "alpha"
        ],
        0.01,
    )
    for horizon in [15, 30, 60]
)

assert {
    horizon:
        PHASE2_FROZEN_MODEL_CONFIGS[
            "LSTM"
        ][
            horizon
        ][
            "epochs"
        ]
    for horizon in [15, 30, 60]
} == {
    15: 7,
    30: 1,
    60: 2,
}

assert {
    horizon:
        PHASE2_FROZEN_MODEL_CONFIGS[
            "TCN"
        ][
            horizon
        ][
            "epochs"
        ]
    for horizon in [15, 30, 60]
} == {
    15: 4,
    30: 1,
    60: 1,
}


# ------------------------------------------------------------
# 5. Persist the final pre-result model registry
# ------------------------------------------------------------

phase2_model_registry_path = (
    PHASE2_RESULTS_DIR
    / "phase2_frozen_model_config_registry.json"
)

# Convert integer horizon keys to strings automatically via JSON.
with open(
    phase2_model_registry_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        PHASE2_FROZEN_MODEL_CONFIGS,
        f,
        indent=2,
    )


print(
    "PHASE 2A FROZEN CONFIGURATION RECOVERY AUDIT: PASSED"
)

print(
    "Saved:",
    phase2_model_registry_path,
)

print(
    "\nExactly 3 F4-selected configurations recovered "
    "for each Phase-2 model family."
)

print(
    "\nSTOP HERE. No Phase-2 model has been trained yet."
)


PHASE 2A FROZEN CONFIGURATION RECOVERY AUDIT: PASSED
Saved: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_frozen_model_config_registry.json

Exactly 3 F4-selected configurations recovered for each Phase-2 model family.

STOP HERE. No Phase-2 model has been trained yet.


# Phase 2A — Stage 1: Classical-model mechanism experiment

This is the **first actual Phase-2 model-fitting stage**.

It evaluates two pre-specified architecture strata:

- **Ridge** — linear/regularized
- **LightGBM** — nonlinear boosted trees

The Stage-1 core representations are:

- `F3`
- `F4_real`
- `F4_reference`
- `F4_shift_17d`
- `F4_shift_37d`
- `F4_shift_61d`
- `F4_perm_1301`
- `F4_perm_1302`
- `F4_perm_1303`

For each model and horizon, the **same F4-selected Phase-1 configuration**
is used for every representation. No Phase-2 hyperparameter tuning occurs.

The five add-one-family component screens remain frozen but are deferred until
after the core mechanism controls have run successfully.


In [25]:
# ============================================================
# Step P2-9: Stage-1 imports and frozen execution set
# ============================================================

import time

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

try:
    from lightgbm import LGBMRegressor
except ImportError as exc:
    raise ImportError(
        "LightGBM is required for Phase 2A Stage 1. "
        "Install it in the active project .venv before continuing."
    ) from exc


PHASE2_STAGE1_MODELS = [
    "Ridge",
    "LightGBM",
]

PHASE2_STAGE1_VARIANTS = [
    "F3",
    "F4_real",
    "F4_reference",
    "F4_shift_17d",
    "F4_shift_37d",
    "F4_shift_61d",
    "F4_perm_1301",
    "F4_perm_1302",
    "F4_perm_1303",
]

PHASE2_STAGE1_HORIZONS = [
    15,
    30,
    60,
]

assert set(
    PHASE2_STAGE1_VARIANTS
).issubset(
    set(
        phase2_variant_tables[
            60
        ].keys()
    )
)

print("Phase 2A Stage-1 execution set:")
print("  Models:   ", PHASE2_STAGE1_MODELS)
print("  Horizons: ", PHASE2_STAGE1_HORIZONS)
print("  Variants: ", PHASE2_STAGE1_VARIANTS)
print(
    "  Planned fits:",
    len(PHASE2_STAGE1_MODELS)
    * len(PHASE2_STAGE1_HORIZONS)
    * len(PHASE2_STAGE1_VARIANTS),
)


Phase 2A Stage-1 execution set:
  Models:    ['Ridge', 'LightGBM']
  Horizons:  [15, 30, 60]
  Variants:  ['F3', 'F4_real', 'F4_reference', 'F4_shift_17d', 'F4_shift_37d', 'F4_shift_61d', 'F4_perm_1301', 'F4_perm_1302', 'F4_perm_1303']
  Planned fits: 54


In [26]:
# ============================================================
# Step P2-10: Shared Phase-2 metric helper
# ============================================================

PHASE2_PV_RATED_KW = 10.0


def phase2_regression_metrics(
    y_true,
    y_pred,
) -> dict:
    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float,
    )

    mae = mean_absolute_error(
        y_true,
        y_pred,
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred,
        )
    )

    r2 = r2_score(
        y_true,
        y_pred,
    )

    return {
        "MAE_kW": float(mae),
        "RMSE_kW": float(rmse),
        "nMAE_percent": float(
            100.0
            * mae
            / PHASE2_PV_RATED_KW
        ),
        "nRMSE_percent": float(
            100.0
            * rmse
            / PHASE2_PV_RATED_KW
        ),
        "R2": float(r2),
    }


In [27]:
# ============================================================
# Step P2-11: Pre-fit Stage-1 structural audit
# ============================================================

for horizon in PHASE2_STAGE1_HORIZONS:

    bundle = modelling_bundles[
        horizon
    ]

    variants = phase2_variant_tables[
        horizon
    ]

    y = bundle["y"]

    fit_mask = (
        bundle["train_mask"]
        | bundle["validation_mask"]
    )

    test_mask = bundle["test_mask"]

    daylight_test_mask = (
        test_mask
        & bundle["daylight_mask"]
    )

    assert int(
        test_mask.sum()
    ) == 26_496

    assert int(
        fit_mask.sum()
        +
        test_mask.sum()
    ) == len(y)

    for variant_name in PHASE2_STAGE1_VARIANTS:

        X = variants[
            variant_name
        ]

        assert X.index.equals(
            y.index
        )

        assert X.isna().sum().sum() == 0

        assert np.isfinite(
            X.to_numpy(
                dtype=float
            )
        ).all()

    # Dimension-matched mechanism-control invariant.
    for variant_name in [
        "F4_real",
        "F4_shift_17d",
        "F4_shift_37d",
        "F4_shift_61d",
        "F4_perm_1301",
        "F4_perm_1302",
        "F4_perm_1303",
    ]:
        assert (
            variants[
                variant_name
            ].shape[1]
            ==
            244
        )

    assert (
        variants[
            "F4_reference"
        ].shape[1]
        ==
        239
    )

    assert (
        variants[
            "F3"
        ].shape[1]
        ==
        147
    )


print(
    "PHASE 2A STAGE-1 PRE-FIT AUDIT: PASSED"
)
print(
    "No Stage-1 model has been fit yet."
)


PHASE 2A STAGE-1 PRE-FIT AUDIT: PASSED
No Stage-1 model has been fit yet.


In [28]:
# ============================================================
# Step P2-12: Fit Ridge + LightGBM across frozen core controls
# ============================================================
#
# IMPORTANT:
#
# - Train + validation are combined for the final refit.
# - TEST remains chronologically later.
# - For a given model/horizon, the same F4-selected
#   configuration is used across every Stage-1 representation.
# - No Phase-2 hyperparameter tuning occurs here.
# ============================================================

stage1_result_rows = []

stage1_prediction_tables = {
    horizon: pd.DataFrame()
    for horizon in PHASE2_STAGE1_HORIZONS
}

stage1_start_time = time.perf_counter()


for horizon in PHASE2_STAGE1_HORIZONS:

    print(
        "\n"
        + "=" * 92
    )
    print(
        f"PHASE 2A STAGE 1 — {horizon}-MINUTE HORIZON"
    )
    print(
        "=" * 92
    )

    bundle = modelling_bundles[
        horizon
    ]

    variants = phase2_variant_tables[
        horizon
    ]

    y = bundle["y"]

    fit_mask = (
        bundle["train_mask"]
        | bundle["validation_mask"]
    )

    test_mask = bundle["test_mask"]

    daylight_test_mask = (
        test_mask
        & bundle["daylight_mask"]
    )

    y_fit = y.loc[
        fit_mask
    ]

    y_test = y.loc[
        test_mask
    ]

    test_index = y_test.index

    daylight_on_test = (
        daylight_test_mask
        .loc[
            test_index
        ]
        .to_numpy(
            dtype=bool
        )
    )

    prediction_table = pd.DataFrame(
        index=test_index
    )

    prediction_table[
        "target_P_Solar_kW"
    ] = y_test.to_numpy(
        dtype=float
    )

    prediction_table[
        "daylight"
    ] = daylight_on_test


    for model_name in PHASE2_STAGE1_MODELS:

        print(
            f"\n--- {model_name} ---"
        )

        frozen_config = (
            PHASE2_FROZEN_MODEL_CONFIGS[
                model_name
            ][
                horizon
            ]
        )


        for variant_name in PHASE2_STAGE1_VARIANTS:

            X = variants[
                variant_name
            ]

            X_fit = X.loc[
                fit_mask
            ]

            X_test = X.loc[
                test_mask
            ]

            fit_predict_start = (
                time.perf_counter()
            )


            # =================================================
            # RIDGE
            # =================================================

            if model_name == "Ridge":

                alpha = float(
                    frozen_config[
                        "alpha"
                    ]
                )

                model = Pipeline(
                    steps=[
                        (
                            "scaler",
                            StandardScaler(),
                        ),
                        (
                            "ridge",
                            Ridge(
                                alpha=alpha
                            ),
                        ),
                    ]
                )

                model.fit(
                    X_fit,
                    y_fit,
                )

                test_predictions = (
                    model.predict(
                        X_test
                    )
                )

                config_string = (
                    f"alpha={alpha:g}"
                )


            # =================================================
            # LIGHTGBM
            # =================================================

            elif model_name == "LightGBM":

                lgbm_config = (
                    frozen_config.copy()
                )

                model = LGBMRegressor(
                    **lgbm_config
                )

                model.fit(
                    X_fit.to_numpy(
                        dtype=np.float32
                    ),
                    y_fit.to_numpy(
                        dtype=np.float32
                    ),
                )

                test_predictions = (
                    model.predict(
                        X_test.to_numpy(
                            dtype=np.float32
                        )
                    )
                )

                config_string = json.dumps(
                    lgbm_config,
                    sort_keys=True,
                )


            else:
                raise ValueError(
                    f"Unsupported Stage-1 model: "
                    f"{model_name}"
                )


            fit_predict_seconds = (
                time.perf_counter()
                -
                fit_predict_start
            )


            test_predictions = np.asarray(
                test_predictions,
                dtype=float,
            )

            assert (
                len(
                    test_predictions
                )
                ==
                len(
                    y_test
                )
            )

            assert np.isfinite(
                test_predictions
            ).all()


            # -------------------------------------------------
            # Persist timestamp-level prediction in memory
            # -------------------------------------------------

            prediction_column = (
                f"{model_name}__{variant_name}"
            )

            prediction_table[
                prediction_column
            ] = test_predictions


            # -------------------------------------------------
            # All + daylight metrics
            # -------------------------------------------------

            scope_masks = {
                "all":
                    np.ones(
                        len(y_test),
                        dtype=bool,
                    ),

                "daylight":
                    daylight_on_test,
            }


            for scope_name, scope_mask in scope_masks.items():

                y_scope = (
                    y_test.to_numpy(
                        dtype=float
                    )[
                        scope_mask
                    ]
                )

                pred_scope = (
                    test_predictions[
                        scope_mask
                    ]
                )

                metrics = (
                    phase2_regression_metrics(
                        y_scope,
                        pred_scope,
                    )
                )

                stage1_result_rows.append(
                    {
                        "model":
                            model_name,

                        "horizon_minutes":
                            horizon,

                        "variant":
                            variant_name,

                        "scope":
                            scope_name,

                        "n_rows":
                            int(
                                scope_mask.sum()
                            ),

                        "n_features":
                            int(
                                X.shape[1]
                            ),

                        "frozen_config":
                            config_string,

                        "fit_predict_seconds":
                            float(
                                fit_predict_seconds
                            ),

                        **metrics,
                    }
                )


            print(
                f"  {variant_name:<18} "
                f"features={X.shape[1]:>3} "
                f"time={fit_predict_seconds:>7.2f}s"
            )


    # ---------------------------------------------------------
    # Add timing metadata and save predictions for horizon
    # ---------------------------------------------------------

    metadata_test = (
        bundle[
            "metadata"
        ]
        .loc[
            test_mask
        ]
    )

    prediction_table.insert(
        0,
        "valid_time",
        metadata_test[
            "valid_time"
        ],
    )

    prediction_table.insert(
        0,
        "issue_time",
        metadata_test[
            "issue_time"
        ],
    )

    prediction_table.index.name = (
        "Timestamp"
    )

    stage1_prediction_tables[
        horizon
    ] = prediction_table


stage1_total_seconds = (
    time.perf_counter()
    -
    stage1_start_time
)


stage1_results = pd.DataFrame(
    stage1_result_rows
)


# 2 models × 3 horizons × 9 variants × 2 scopes = 108 rows
assert len(
    stage1_results
) == (
    2
    * 3
    * 9
    * 2
)


assert stage1_results[
    [
        "MAE_kW",
        "RMSE_kW",
        "nMAE_percent",
        "nRMSE_percent",
        "R2",
    ]
].notna().all().all()


assert np.isfinite(
    stage1_results[
        [
            "MAE_kW",
            "RMSE_kW",
            "nMAE_percent",
            "nRMSE_percent",
            "R2",
        ]
    ].to_numpy()
).all()


print(
    "\n"
    + "=" * 92
)

print(
    "PHASE 2A STAGE-1 MODEL FITTING: PASSED"
)

print(
    f"Total Stage-1 runtime: "
    f"{stage1_total_seconds:.2f} seconds"
)



PHASE 2A STAGE 1 — 15-MINUTE HORIZON

--- Ridge ---
  F3                 features=147 time=   0.23s
  F4_real            features=244 time=   0.36s
  F4_reference       features=239 time=   0.31s
  F4_shift_17d       features=244 time=   0.32s
  F4_shift_37d       features=244 time=   0.32s
  F4_shift_61d       features=244 time=   0.31s
  F4_perm_1301       features=244 time=   0.32s
  F4_perm_1302       features=244 time=   0.32s
  F4_perm_1303       features=244 time=   0.32s

--- LightGBM ---
  F3                 features=147 time=   6.62s
  F4_real            features=244 time=   6.57s
  F4_reference       features=239 time=   6.80s
  F4_shift_17d       features=244 time=   6.48s
  F4_shift_37d       features=244 time=   6.53s
  F4_shift_61d       features=244 time=   6.50s
  F4_perm_1301       features=244 time=   6.46s
  F4_perm_1302       features=244 time=   6.42s
  F4_perm_1303       features=244 time=   6.51s

PHASE 2A STAGE 1 — 30-MINUTE HORIZON

--- Ridge ---
  F3        

In [29]:
# ============================================================
# Step P2-13: Persist Stage-1 results before interpretation
# ============================================================

stage1_results_path = (
    PHASE2_RESULTS_DIR
    / "phase2_stage1_classical_core_results.csv"
)

stage1_results.to_csv(
    stage1_results_path,
    index=False,
)


stage1_prediction_paths = {}

for horizon, prediction_table in (
    stage1_prediction_tables.items()
):

    prediction_path = (
        PHASE2_RESULTS_DIR
        /
        (
            "phase2_stage1_classical_core_"
            f"predictions_{horizon}min.csv"
        )
    )

    prediction_table.to_csv(
        prediction_path,
        index=True,
    )

    stage1_prediction_paths[
        horizon
    ] = prediction_path


# ------------------------------------------------------------
# Compact all-row comparison table.
# No selection is performed here.
# ------------------------------------------------------------

stage1_all_mae = (
    stage1_results.loc[
        stage1_results[
            "scope"
        ].eq("all")
    ]
    .pivot_table(
        index=[
            "model",
            "horizon_minutes",
        ],
        columns="variant",
        values="MAE_kW",
        aggfunc="first",
    )
    .reset_index()
)


stage1_all_mae_path = (
    PHASE2_RESULTS_DIR
    / "phase2_stage1_allrow_mae_pivot.csv"
)

stage1_all_mae.to_csv(
    stage1_all_mae_path,
    index=False,
)


print(
    "Stage-1 raw results saved:"
)

print(
    " ",
    stage1_results_path,
)

print(
    " ",
    stage1_all_mae_path,
)

for horizon in PHASE2_STAGE1_HORIZONS:
    print(
        " ",
        stage1_prediction_paths[
            horizon
        ],
    )


print(
    "\nAll-row MAE table "
    "(descriptive only; no model selection):"
)

display(
    stage1_all_mae
)


print(
    "\nSTOP HERE after Stage 1."
)

print(
    "Do not add component-screen or neural runs "
    "until these raw outputs are reviewed."
)


Stage-1 raw results saved:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_stage1_classical_core_results.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_stage1_allrow_mae_pivot.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_stage1_classical_core_predictions_15min.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_stage1_classical_core_predictions_30min.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_stage1_classical_core_predictions_60min.csv

All-row MAE table (desc

variant,model,horizon_minutes,F3,F4_perm_1301,F4_perm_1302,F4_perm_1303,F4_real,F4_reference,F4_shift_17d,F4_shift_37d,F4_shift_61d
0,LightGBM,15,0.260922,0.261711,0.261819,0.261790,0.262057,0.262658,0.262169,0.262288,0.262970
1,LightGBM,30,0.302415,0.301292,0.301417,0.301311,0.300646,0.302067,0.302462,0.300689,0.302691
2,LightGBM,60,0.343349,0.346260,0.346281,0.346281,0.343123,0.341401,0.344719,0.343241,0.344973
3,Ridge,15,0.318521,0.318967,0.319017,0.318635,0.329466,0.329466,0.335008,0.331034,0.336704
4,Ridge,30,0.401663,0.402169,0.402150,0.401938,0.420881,0.420881,0.431241,0.423595,0.431668
5,Ridge,60,0.527476,0.528382,0.527570,0.527140,0.562776,0.562776,0.584348,0.565811,0.579642



STOP HERE after Stage 1.
Do not add component-screen or neural runs until these raw outputs are reviewed.


# Phase 2A — Stage 1B: Daylight + paired UTC-day bootstrap

This section performs **no model fitting**.

It uses the already-generated Stage-1 TEST predictions to quantify uncertainty
for the mechanism comparisons.

## Frozen comparison statistic

For every comparator \(C\):

\[
\Delta MAE = MAE_C - MAE_{F4_{real}}
\]

Therefore:

- **positive** \(\Delta MAE\): correctly aligned `F4_real` has lower MAE;
- **negative** \(\Delta MAE\): the comparator has lower MAE;
- **zero-crossing CI**: direction is unresolved at the 95% percentile interval.

Comparators:

- `F3`
- `F4_shift_17d`
- `F4_shift_37d`
- `F4_shift_61d`
- `F4_perm_1301`
- `F4_perm_1302`
- `F4_perm_1303`

Uncertainty protocol:

- forecast-valid **UTC calendar day** as the block;
- 92 held-out TEST days;
- 10,000 paired bootstrap replicates;
- seed 42;
- no model retraining during bootstrap;
- both `all` and `daylight` scopes.

This mirrors the frozen Phase-1 block-bootstrap philosophy while changing the
comparison target from only `F3 → F4` to the Phase-2 mechanism controls.


In [30]:
# ============================================================
# Step P2-14: Recover Stage-1 prediction tables robustly
# ============================================================

STAGE1_BOOTSTRAP_REPLICATES = 10_000
STAGE1_BOOTSTRAP_SEED = 42

STAGE1_COMPARATORS = [
    "F3",
    "F4_shift_17d",
    "F4_shift_37d",
    "F4_shift_61d",
    "F4_perm_1301",
    "F4_perm_1302",
    "F4_perm_1303",
]

STAGE1_BOOTSTRAP_MODELS = [
    "Ridge",
    "LightGBM",
]

STAGE1_BOOTSTRAP_HORIZONS = [
    15,
    30,
    60,
]


# If the current kernel still has the prediction tables produced
# by Stage 1, reuse them. Otherwise reload the persisted CSVs.
if (
    "stage1_prediction_tables"
    not in globals()
):
    stage1_prediction_tables = {}

    for horizon in STAGE1_BOOTSTRAP_HORIZONS:

        prediction_path = (
            PHASE2_RESULTS_DIR
            /
            (
                "phase2_stage1_classical_core_"
                f"predictions_{horizon}min.csv"
            )
        )

        assert prediction_path.is_file(), (
            f"Missing Stage-1 prediction file: {prediction_path}"
        )

        prediction_table = pd.read_csv(
            prediction_path,
            index_col=0,
        )

        prediction_table.index = pd.to_datetime(
            prediction_table.index,
            utc=True,
        )

        prediction_table["issue_time"] = pd.to_datetime(
            prediction_table["issue_time"],
            utc=True,
        )

        prediction_table["valid_time"] = pd.to_datetime(
            prediction_table["valid_time"],
            utc=True,
        )

        prediction_table["daylight"] = (
            prediction_table["daylight"]
            .astype(bool)
        )

        stage1_prediction_tables[
            horizon
        ] = prediction_table


for horizon in STAGE1_BOOTSTRAP_HORIZONS:

    table = stage1_prediction_tables[
        horizon
    ]

    assert len(table) == 26_496

    assert (
        table[
            "valid_time"
        ]
        .dt.floor("D")
        .nunique()
        ==
        92
    )

    required_prediction_columns = [
        f"{model_name}__F4_real"
        for model_name in STAGE1_BOOTSTRAP_MODELS
    ]

    for model_name in STAGE1_BOOTSTRAP_MODELS:
        for comparator in STAGE1_COMPARATORS:
            required_prediction_columns.append(
                f"{model_name}__{comparator}"
            )

    missing = [
        column
        for column in required_prediction_columns
        if column not in table.columns
    ]

    assert not missing, (
        f"{horizon}-min prediction table missing columns: {missing}"
    )


print(
    "PHASE 2A STAGE-1B PREDICTION RECOVERY AUDIT: PASSED"
)


PHASE 2A STAGE-1B PREDICTION RECOVERY AUDIT: PASSED


In [31]:
# ============================================================
# Step P2-15: Daylight descriptive MAE table
# ============================================================

stage1_daylight_rows = []

for horizon in STAGE1_BOOTSTRAP_HORIZONS:

    table = stage1_prediction_tables[
        horizon
    ]

    daylight_mask = (
        table[
            "daylight"
        ]
        .to_numpy(
            dtype=bool
        )
    )

    y_true = (
        table[
            "target_P_Solar_kW"
        ]
        .to_numpy(
            dtype=float
        )
    )

    for model_name in STAGE1_BOOTSTRAP_MODELS:

        for variant_name in [
            "F3",
            "F4_real",
            "F4_reference",
            "F4_shift_17d",
            "F4_shift_37d",
            "F4_shift_61d",
            "F4_perm_1301",
            "F4_perm_1302",
            "F4_perm_1303",
        ]:

            prediction_column = (
                f"{model_name}__{variant_name}"
            )

            prediction = (
                table[
                    prediction_column
                ]
                .to_numpy(
                    dtype=float
                )
            )

            daylight_mae = (
                np.mean(
                    np.abs(
                        y_true[
                            daylight_mask
                        ]
                        -
                        prediction[
                            daylight_mask
                        ]
                    )
                )
            )

            stage1_daylight_rows.append(
                {
                    "model":
                        model_name,

                    "horizon_minutes":
                        horizon,

                    "variant":
                        variant_name,

                    "daylight_MAE_kW":
                        float(
                            daylight_mae
                        ),

                    "daylight_rows":
                        int(
                            daylight_mask.sum()
                        ),
                }
            )


stage1_daylight_results = pd.DataFrame(
    stage1_daylight_rows
)

stage1_daylight_pivot = (
    stage1_daylight_results
    .pivot_table(
        index=[
            "model",
            "horizon_minutes",
        ],
        columns="variant",
        values="daylight_MAE_kW",
        aggfunc="first",
    )
    .reset_index()
)


stage1_daylight_path = (
    PHASE2_RESULTS_DIR
    / "phase2_stage1_daylight_mae_pivot.csv"
)

stage1_daylight_pivot.to_csv(
    stage1_daylight_path,
    index=False,
)


print(
    "Daylight MAE table:"
)

display(
    stage1_daylight_pivot
)

print(
    "Saved:",
    stage1_daylight_path,
)


Daylight MAE table:


variant,model,horizon_minutes,F3,F4_perm_1301,F4_perm_1302,F4_perm_1303,F4_real,F4_reference,F4_shift_17d,F4_shift_37d,F4_shift_61d
0,LightGBM,15,0.385134,0.386310,0.386518,0.386574,0.386933,0.387786,0.386941,0.387104,0.388051
1,LightGBM,30,0.446240,0.444630,0.444845,0.444615,0.443675,0.445424,0.446254,0.443712,0.446453
2,LightGBM,60,0.506980,0.510620,0.510651,0.510651,0.505740,0.504456,0.508240,0.506379,0.508676
3,Ridge,15,0.414016,0.414348,0.414392,0.414290,0.421066,0.421066,0.424680,0.421869,0.426427
4,Ridge,30,0.496434,0.496981,0.496913,0.496875,0.509375,0.509375,0.516614,0.510827,0.517359
5,Ridge,60,0.609240,0.610097,0.609429,0.609178,0.634608,0.634608,0.650134,0.635531,0.646641


Saved: /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_stage1_daylight_mae_pivot.csv


In [32]:
# ============================================================
# Step P2-16: Paired valid-UTC-day bootstrap helper
# ============================================================

def paired_day_block_bootstrap_f4_real(
    prediction_table: pd.DataFrame,
    comparator_prediction_column: str,
    real_f4_prediction_column: str,
    scope: str,
    bootstrap_draws: np.ndarray,
) -> dict:
    """
    Paired UTC-calendar-day block bootstrap for:

        MAE_comparator - MAE_F4_real

    Positive values mean correctly aligned F4_real has lower MAE.
    """

    if scope not in {
        "all",
        "daylight",
    }:
        raise ValueError(
            f"Unsupported scope: {scope}"
        )


    if scope == "all":
        scope_mask = np.ones(
            len(prediction_table),
            dtype=bool,
        )

    else:
        scope_mask = (
            prediction_table[
                "daylight"
            ]
            .to_numpy(
                dtype=bool
            )
        )


    scoped = (
        prediction_table
        .loc[
            scope_mask
        ]
        .copy()
    )


    scoped[
        "_valid_utc_day"
    ] = (
        scoped[
            "valid_time"
        ]
        .dt.floor("D")
    )


    y_true = (
        scoped[
            "target_P_Solar_kW"
        ]
        .to_numpy(
            dtype=float
        )
    )

    comparator_pred = (
        scoped[
            comparator_prediction_column
        ]
        .to_numpy(
            dtype=float
        )
    )

    real_pred = (
        scoped[
            real_f4_prediction_column
        ]
        .to_numpy(
            dtype=float
        )
    )


    comparator_abs_error = np.abs(
        y_true
        -
        comparator_pred
    )

    real_abs_error = np.abs(
        y_true
        -
        real_pred
    )


    working = pd.DataFrame(
        {
            "day":
                scoped[
                    "_valid_utc_day"
                ]
                .to_numpy(),

            "comparator_abs_error":
                comparator_abs_error,

            "real_abs_error":
                real_abs_error,
        }
    )


    daily = (
        working
        .groupby(
            "day",
            sort=True,
        )
        .agg(
            comparator_error_sum=(
                "comparator_abs_error",
                "sum",
            ),
            real_error_sum=(
                "real_abs_error",
                "sum",
            ),
            row_count=(
                "comparator_abs_error",
                "size",
            ),
        )
    )


    assert len(daily) == 92

    comparator_daily_sum = (
        daily[
            "comparator_error_sum"
        ]
        .to_numpy(
            dtype=float
        )
    )

    real_daily_sum = (
        daily[
            "real_error_sum"
        ]
        .to_numpy(
            dtype=float
        )
    )

    daily_count = (
        daily[
            "row_count"
        ]
        .to_numpy(
            dtype=float
        )
    )


    # --------------------------------------------------------
    # Point estimate
    # --------------------------------------------------------

    comparator_mae = float(
        comparator_abs_error.mean()
    )

    real_mae = float(
        real_abs_error.mean()
    )

    delta_mae = (
        comparator_mae
        -
        real_mae
    )

    improvement_percent = float(
        100.0
        *
        delta_mae
        /
        comparator_mae
    )


    # --------------------------------------------------------
    # Vectorized paired block bootstrap
    # --------------------------------------------------------

    sampled_counts = (
        daily_count[
            bootstrap_draws
        ]
        .sum(
            axis=1
        )
    )

    sampled_comparator_mae = (
        comparator_daily_sum[
            bootstrap_draws
        ]
        .sum(
            axis=1
        )
        /
        sampled_counts
    )

    sampled_real_mae = (
        real_daily_sum[
            bootstrap_draws
        ]
        .sum(
            axis=1
        )
        /
        sampled_counts
    )

    bootstrap_delta = (
        sampled_comparator_mae
        -
        sampled_real_mae
    )

    bootstrap_percent = (
        100.0
        *
        bootstrap_delta
        /
        sampled_comparator_mae
    )


    ci_low_kw, ci_high_kw = (
        np.percentile(
            bootstrap_delta,
            [
                2.5,
                97.5,
            ],
        )
    )

    ci_low_percent, ci_high_percent = (
        np.percentile(
            bootstrap_percent,
            [
                2.5,
                97.5,
            ],
        )
    )


    if ci_low_kw > 0:
        ci_zero_status = (
            "above_zero"
        )

    elif ci_high_kw < 0:
        ci_zero_status = (
            "below_zero"
        )

    else:
        ci_zero_status = (
            "crosses_zero"
        )


    return {
        "n_rows":
            int(
                len(scoped)
            ),

        "daily_blocks":
            int(
                len(daily)
            ),

        "comparator_MAE_kW":
            comparator_mae,

        "F4_real_MAE_kW":
            real_mae,

        "comparator_minus_F4_real_MAE_kW":
            float(
                delta_mae
            ),

        "F4_real_improvement_percent":
            improvement_percent,

        "CI95_low_kW":
            float(
                ci_low_kw
            ),

        "CI95_high_kW":
            float(
                ci_high_kw
            ),

        "CI95_low_percent":
            float(
                ci_low_percent
            ),

        "CI95_high_percent":
            float(
                ci_high_percent
            ),

        "CI_zero_status":
            ci_zero_status,
    }


In [33]:
# ============================================================
# Step P2-17: Execute the frozen Stage-1B bootstrap
# ============================================================

stage1_bootstrap_rows = []

scope_order = [
    "all",
    "daylight",
]

# Pre-generate one paired day-resampling matrix for each
# horizon × scope. Every model/control comparison inside the
# same condition therefore sees the same resampled UTC days.
bootstrap_draw_registry = {}


for horizon in STAGE1_BOOTSTRAP_HORIZONS:

    for scope_index, scope in enumerate(
        scope_order
    ):

        seed_sequence = np.random.SeedSequence(
            [
                STAGE1_BOOTSTRAP_SEED,
                int(horizon),
                int(scope_index),
            ]
        )

        rng = np.random.default_rng(
            seed_sequence
        )

        bootstrap_draw_registry[
            (
                horizon,
                scope,
            )
        ] = rng.integers(
            low=0,
            high=92,
            size=(
                STAGE1_BOOTSTRAP_REPLICATES,
                92,
            ),
            endpoint=False,
        )


for horizon in STAGE1_BOOTSTRAP_HORIZONS:

    table = stage1_prediction_tables[
        horizon
    ]

    for scope in scope_order:

        bootstrap_draws = (
            bootstrap_draw_registry[
                (
                    horizon,
                    scope,
                )
            ]
        )

        for model_name in STAGE1_BOOTSTRAP_MODELS:

            real_column = (
                f"{model_name}__F4_real"
            )

            for comparator in STAGE1_COMPARATORS:

                comparator_column = (
                    f"{model_name}__{comparator}"
                )

                result = (
                    paired_day_block_bootstrap_f4_real(
                        prediction_table=table,
                        comparator_prediction_column=comparator_column,
                        real_f4_prediction_column=real_column,
                        scope=scope,
                        bootstrap_draws=bootstrap_draws,
                    )
                )

                stage1_bootstrap_rows.append(
                    {
                        "model":
                            model_name,

                        "horizon_minutes":
                            horizon,

                        "scope":
                            scope,

                        "comparator":
                            comparator,

                        **result,
                    }
                )


stage1_bootstrap_summary = pd.DataFrame(
    stage1_bootstrap_rows
)


# 2 models × 3 horizons × 2 scopes × 7 comparators
assert len(
    stage1_bootstrap_summary
) == (
    2
    * 3
    * 2
    * 7
)


assert (
    stage1_bootstrap_summary[
        "daily_blocks"
    ]
    .eq(
        92
    )
    .all()
)


print(
    "PHASE 2A STAGE-1B BOOTSTRAP: PASSED"
)

print(
    "Comparisons:",
    len(
        stage1_bootstrap_summary
    ),
)

print(
    "Replicates per comparison:",
    STAGE1_BOOTSTRAP_REPLICATES,
)


PHASE 2A STAGE-1B BOOTSTRAP: PASSED
Comparisons: 84
Replicates per comparison: 10000


In [34]:
# ============================================================
# Step P2-18: Persist and display bootstrap summaries
# ============================================================

stage1_bootstrap_path = (
    PHASE2_RESULTS_DIR
    / "phase2_stage1_paired_day_bootstrap_summary.csv"
)

stage1_bootstrap_summary.to_csv(
    stage1_bootstrap_path,
    index=False,
)


# ------------------------------------------------------------
# A. Primary F3 versus correctly aligned F4_real
# ------------------------------------------------------------

stage1_f3_vs_real = (
    stage1_bootstrap_summary.loc[
        stage1_bootstrap_summary[
            "comparator"
        ].eq(
            "F3"
        )
    ]
    .copy()
    .sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "=" * 100
)

print(
    "PRIMARY MECHANISM COMPARISON:"
)

print(
    "F3 MAE - F4_real MAE"
)

print(
    "Positive = correctly aligned F4_real lower MAE"
)

print(
    "=" * 100
)

display(
    stage1_f3_vs_real[
        [
            "model",
            "horizon_minutes",
            "scope",
            "comparator_MAE_kW",
            "F4_real_MAE_kW",
            "comparator_minus_F4_real_MAE_kW",
            "F4_real_improvement_percent",
            "CI95_low_percent",
            "CI95_high_percent",
            "CI_zero_status",
        ]
    ]
)


# ------------------------------------------------------------
# B. Real F4 versus the dimension-matched null controls
# ------------------------------------------------------------

stage1_real_vs_null = (
    stage1_bootstrap_summary.loc[
        ~stage1_bootstrap_summary[
            "comparator"
        ].eq(
            "F3"
        )
    ]
    .copy()
    .sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
            "comparator",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "DIMENSION-MATCHED NULL COMPARISONS:"
)

print(
    "Null-control MAE - F4_real MAE"
)

print(
    "Positive = correctly aligned F4_real lower MAE"
)

print(
    "=" * 100
)

display(
    stage1_real_vs_null[
        [
            "model",
            "horizon_minutes",
            "scope",
            "comparator",
            "comparator_minus_F4_real_MAE_kW",
            "F4_real_improvement_percent",
            "CI95_low_percent",
            "CI95_high_percent",
            "CI_zero_status",
        ]
    ]
)


# ------------------------------------------------------------
# C. Compact evidence-count audit
# ------------------------------------------------------------

stage1_bootstrap_evidence_counts = (
    stage1_bootstrap_summary
    .groupby(
        [
            "model",
            "scope",
            "CI_zero_status",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)


print(
    "\nEvidence-count summary:"
)

display(
    stage1_bootstrap_evidence_counts
)


evidence_count_path = (
    PHASE2_RESULTS_DIR
    / "phase2_stage1_bootstrap_evidence_counts.csv"
)

stage1_bootstrap_evidence_counts.to_csv(
    evidence_count_path,
    index=False,
)


print(
    "\nSaved:"
)

print(
    " ",
    stage1_bootstrap_path,
)

print(
    " ",
    evidence_count_path,
)


print(
    "\nSTOP HERE."
)

print(
    "No LSTM/TCN Phase-2 model has been trained yet."
)


PRIMARY MECHANISM COMPARISON:
F3 MAE - F4_real MAE
Positive = correctly aligned F4_real lower MAE


,model,horizon_minutes,scope,comparator_MAE_kW,F4_real_MAE_kW,comparator_minus_F4_real_MAE_kW,F4_real_improvement_percent,CI95_low_percent,CI95_high_percent,CI_zero_status
0,LightGBM,15,all,0.260922,0.262057,-0.001135,-0.435003,-0.809260,-0.105090,below_zero
1,LightGBM,15,daylight,0.385134,0.386933,-0.001799,-0.467191,-0.849248,-0.133525,below_zero
2,LightGBM,30,all,0.302415,0.300646,0.001769,0.584866,0.173550,1.064364,above_zero
3,LightGBM,30,daylight,0.446240,0.443675,0.002565,0.574744,0.159423,1.061117,above_zero
4,LightGBM,60,all,0.343349,0.343123,0.000226,0.065940,-0.555602,0.743921,crosses_zero
5,LightGBM,60,daylight,0.506980,0.505740,0.001240,0.244620,-0.391883,0.950502,crosses_zero
6,Ridge,15,all,0.318521,0.329466,-0.010945,-3.436065,-4.763754,-2.163911,below_zero
7,Ridge,15,daylight,0.414016,0.421066,-0.007050,-1.702771,-2.437786,-1.020227,below_zero
8,Ridge,30,all,0.401663,0.420881,-0.019218,-4.784612,-6.693354,-3.006604,below_zero
9,Ridge,30,daylight,0.496434,0.509375,-0.012941,-2.606784,-3.738024,-1.582670,below_zero



DIMENSION-MATCHED NULL COMPARISONS:
Null-control MAE - F4_real MAE
Positive = correctly aligned F4_real lower MAE


,model,horizon_minutes,scope,comparator,comparator_minus_F4_real_MAE_kW,F4_real_improvement_percent,CI95_low_percent,CI95_high_percent,CI_zero_status
0,LightGBM,15,all,F4_perm_1301,-0.000346,-0.132106,-0.520176,0.224427,crosses_zero
1,LightGBM,15,all,F4_perm_1302,-0.000238,-0.090821,-0.420112,0.196407,crosses_zero
2,LightGBM,15,all,F4_perm_1303,-0.000267,-0.101985,-0.454429,0.213354,crosses_zero
3,LightGBM,15,all,F4_shift_17d,0.000112,0.042607,-0.295972,0.397190,crosses_zero
4,LightGBM,15,all,F4_shift_37d,0.000231,0.088257,-0.206167,0.387309,crosses_zero
...,...,...,...,...,...,...,...,...,...
67,Ridge,60,daylight,F4_perm_1302,-0.025180,-4.131706,-5.865441,-2.486154,below_zero
68,Ridge,60,daylight,F4_perm_1303,-0.025430,-4.174496,-5.932805,-2.509338,below_zero
69,Ridge,60,daylight,F4_shift_17d,0.015526,2.388131,0.975643,3.883126,above_zero
70,Ridge,60,daylight,F4_shift_37d,0.000923,0.145178,-1.130839,1.346537,crosses_zero



Evidence-count summary:


CI_zero_status,model,scope,above_zero,below_zero,crosses_zero
0,LightGBM,all,7,1,13
1,LightGBM,daylight,7,1,13
2,Ridge,all,4,12,5
3,Ridge,daylight,4,12,5



Saved:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_stage1_paired_day_bootstrap_summary.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_stage1_bootstrap_evidence_counts.csv

STOP HERE.
No LSTM/TCN Phase-2 model has been trained yet.


In [121]:
# ============================================================
# Step P2-19: Build Phase-2 neural inputs using the frozen
#             Notebook-11 representation/scaling protocol
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Frozen neural protocol
# ------------------------------------------------------------

PHASE2_NEURAL_HORIZONS = [15, 30, 60]

PHASE2_NEURAL_VARIANTS = [
    "F3",
    "F4_real",
    "F4_reference",
    "F4_shift_17d",
    "F4_shift_37d",
    "F4_shift_61d",
    "F4_perm_1301",
    "F4_perm_1302",
    "F4_perm_1303",
]

PHASE2_SEQUENCE_LENGTH = 13

PHASE2_SEQUENCE_BASE_NAMES = list(
    F0_HISTORY_SOURCES.keys()
)

assert len(
    PHASE2_SEQUENCE_BASE_NAMES
) == 9


# ------------------------------------------------------------
# 2. Exact recent-history columns used by the sequence branch
#
# Sequence order:
#
#   t-60, t-55, ..., t-5, t
#
# so lag order is:
#
#   12, 11, ..., 1, 0
# ------------------------------------------------------------

PHASE2_RECENT_HISTORY_COLUMNS = [
    f"{feature_name}_lag_{lag}"
    for feature_name in PHASE2_SEQUENCE_BASE_NAMES
    for lag in RECENT_LAGS
]

assert len(
    PHASE2_RECENT_HISTORY_COLUMNS
) == 117


# ------------------------------------------------------------
# 3. Binary auxiliary feature rules
#
# Notebook 11 leaves:
#
#   - weekend indicator
#   - daylight indicator
#   - Panchang one-hot indicators
#
# exactly as 0/1.
# ------------------------------------------------------------

PHASE2_BINARY_BASE_COLUMNS = {
    "valid_utc_is_weekend",
    "valid_solar_daylight",
}

PHASE2_PANCHANG_PREFIXES = (
    "valid_tithi_",
    "valid_paksha_",
    "valid_karana_",
    "valid_nakshatra_",
    "valid_yoga_",
)


def is_phase2_binary_aux_column(column_name: str) -> bool:

    if column_name in PHASE2_BINARY_BASE_COLUMNS:
        return True

    return column_name.startswith(
        PHASE2_PANCHANG_PREFIXES
    )


# ------------------------------------------------------------
# 4. Output containers
# ------------------------------------------------------------

phase2_neural_data = {}

phase2_neural_scaling_statistics = {}


# ------------------------------------------------------------
# 5. Build each horizon independently
# ------------------------------------------------------------

for horizon in PHASE2_NEURAL_HORIZONS:

    bundle = modelling_bundles[
        horizon
    ]

    variants = phase2_variant_tables[
        horizon
    ]

    X_f3 = variants[
        "F3"
    ]

    y = bundle[
        "y"
    ]


    # --------------------------------------------------------
    # Split masks
    # --------------------------------------------------------

    train_mask = (
        bundle[
            "train_mask"
        ]
        .to_numpy(
            dtype=bool
        )
    )

    validation_mask = (
        bundle[
            "validation_mask"
        ]
        .to_numpy(
            dtype=bool
        )
    )

    test_mask = (
        bundle[
            "test_mask"
        ]
        .to_numpy(
            dtype=bool
        )
    )


    assert (
        train_mask.sum()
        +
        validation_mask.sum()
        +
        test_mask.sum()
        ==
        len(y)
    )

    assert (
        test_mask.sum()
        ==
        26_496
    )


    # ========================================================
    # A. RAW HISTORICAL SEQUENCE
    # ========================================================

    sequence_timesteps = []

    for lag in range(
        12,
        -1,
        -1,
    ):

        timestep_columns = [
            f"{feature_name}_lag_{lag}"
            for feature_name
            in PHASE2_SEQUENCE_BASE_NAMES
        ]

        timestep_values = (
            X_f3[
                timestep_columns
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        sequence_timesteps.append(
            timestep_values
        )


    raw_sequence = np.stack(
        sequence_timesteps,
        axis=1,
    )


    assert raw_sequence.shape == (
        len(y),
        13,
        9,
    )

    assert np.isfinite(
        raw_sequence
    ).all()


    # ========================================================
    # B. TRAIN-ONLY SEQUENCE SCALING
    # ========================================================

    train_sequence_64 = (
        raw_sequence[
            train_mask
        ]
        .astype(
            np.float64,
            copy=False,
        )
    )

    sequence_mean = (
        train_sequence_64
        .mean(
            axis=(0, 1)
        )
    )

    sequence_std = (
        train_sequence_64
        .std(
            axis=(0, 1),
            ddof=0,
        )
    )

    assert (
        sequence_std
        >
        0
    ).all()


    scaled_sequence = (
        (
            raw_sequence.astype(
                np.float64
            )
            -
            sequence_mean
        )
        /
        sequence_std
    ).astype(
        np.float32
    )


    # ========================================================
    # C. TRAIN-ONLY TARGET SCALING
    # ========================================================

    raw_target = y.to_numpy(
        dtype=np.float32
    )

    train_target_64 = (
        raw_target[
            train_mask
        ]
        .astype(
            np.float64
        )
    )

    target_mean = float(
        train_target_64.mean()
    )

    target_std = float(
        train_target_64.std(
            ddof=0
        )
    )

    assert target_std > 0


    scaled_target = (
        (
            raw_target.astype(
                np.float64
            )
            -
            target_mean
        )
        /
        target_std
    ).astype(
        np.float32
    )


    # ========================================================
    # D. SCALE EACH PHASE-2 AUXILIARY REPRESENTATION
    # ========================================================

    variant_data = {}

    for variant_name in PHASE2_NEURAL_VARIANTS:

        X_variant = variants[
            variant_name
        ]


        # ----------------------------------------------------
        # Auxiliary branch =
        #
        #   24-hour PV lag
        #   +
        #   all deterministic F1/F2/F3/F4 features
        #
        # The 117 recent-history features are represented
        # exclusively by the sequence branch.
        # ----------------------------------------------------

        auxiliary_columns = [
            column
            for column in X_variant.columns
            if column
            not in PHASE2_RECENT_HISTORY_COLUMNS
        ]


        assert (
            "P_Solar_kw_lag_288"
            in auxiliary_columns
        )


        raw_auxiliary_df = (
            X_variant[
                auxiliary_columns
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Identify binary vs continuous columns
        # ----------------------------------------------------

        binary_columns = [
            column
            for column in auxiliary_columns
            if is_phase2_binary_aux_column(
                column
            )
        ]

        continuous_columns = [
            column
            for column in auxiliary_columns
            if column
            not in binary_columns
        ]


        # ----------------------------------------------------
        # Expected auxiliary widths
        # ----------------------------------------------------

        expected_widths = {
            "F3": 30,
            "F4_real": 127,
            "F4_reference": 122,
            "F4_shift_17d": 127,
            "F4_shift_37d": 127,
            "F4_shift_61d": 127,
            "F4_perm_1301": 127,
            "F4_perm_1302": 127,
            "F4_perm_1303": 127,
        }

        assert (
            len(
                auxiliary_columns
            )
            ==
            expected_widths[
                variant_name
            ]
        )


        raw_auxiliary = (
            raw_auxiliary_df
            .to_numpy(
                dtype=np.float32
            )
        )


        scaled_auxiliary = (
            raw_auxiliary.copy()
        )


        # ----------------------------------------------------
        # Train-only continuous auxiliary scaling
        # ----------------------------------------------------

        continuous_positions = np.array(
            [
                auxiliary_columns.index(
                    column
                )
                for column
                in continuous_columns
            ],
            dtype=np.int64,
        )


        binary_positions = np.array(
            [
                auxiliary_columns.index(
                    column
                )
                for column
                in binary_columns
            ],
            dtype=np.int64,
        )


        train_continuous = (
            raw_auxiliary[
                train_mask
            ][
                :,
                continuous_positions,
            ]
            .astype(
                np.float64
            )
        )


        auxiliary_mean = (
            train_continuous
            .mean(
                axis=0
            )
        )


        auxiliary_std = (
            train_continuous
            .std(
                axis=0,
                ddof=0,
            )
        )


        assert (
            auxiliary_std
            >
            0
        ).all()


        scaled_continuous = (
            (
                raw_auxiliary[
                    :,
                    continuous_positions,
                ]
                .astype(
                    np.float64
                )
                -
                auxiliary_mean
            )
            /
            auxiliary_std
        ).astype(
            np.float32
        )


        scaled_auxiliary[
            :,
            continuous_positions,
        ] = scaled_continuous


        # ----------------------------------------------------
        # Binary columns must remain EXACTLY 0/1
        # ----------------------------------------------------

        if len(
            binary_positions
        ) > 0:

            assert np.array_equal(
                scaled_auxiliary[
                    :,
                    binary_positions,
                ],
                raw_auxiliary[
                    :,
                    binary_positions,
                ],
            )

            observed_binary_values = np.unique(
                scaled_auxiliary[
                    :,
                    binary_positions,
                ]
            )

            assert np.all(
                np.isin(
                    observed_binary_values,
                    [
                        0.0,
                        1.0,
                    ],
                )
            )


        # ----------------------------------------------------
        # Store variant
        # ----------------------------------------------------

        variant_data[
            variant_name
        ] = {
            "auxiliary":
                np.ascontiguousarray(
                    scaled_auxiliary,
                    dtype=np.float32,
                ),

            "auxiliary_columns":
                tuple(
                    auxiliary_columns
                ),

            "continuous_columns":
                tuple(
                    continuous_columns
                ),

            "binary_columns":
                tuple(
                    binary_columns
                ),

            "auxiliary_mean":
                auxiliary_mean.copy(),

            "auxiliary_std":
                auxiliary_std.copy(),
        }


    # ========================================================
    # E. STORE HORIZON
    # ========================================================

    phase2_neural_data[
        horizon
    ] = {
        "sequence":
            np.ascontiguousarray(
                scaled_sequence,
                dtype=np.float32,
            ),

        "target_scaled":
            np.ascontiguousarray(
                scaled_target,
                dtype=np.float32,
            ),

        "target_kw":
            np.ascontiguousarray(
                raw_target,
                dtype=np.float32,
            ),

        "train_mask":
            train_mask.copy(),

        "validation_mask":
            validation_mask.copy(),

        "test_mask":
            test_mask.copy(),

        "variants":
            variant_data,
    }


    phase2_neural_scaling_statistics[
        horizon
    ] = {
        "sequence_mean":
            sequence_mean.copy(),

        "sequence_std":
            sequence_std.copy(),

        "target_mean":
            target_mean,

        "target_std":
            target_std,
    }


# ============================================================
# 6. Strong audits
# ============================================================

for horizon in PHASE2_NEURAL_HORIZONS:

    data = phase2_neural_data[
        horizon
    ]

    train_mask = data[
        "train_mask"
    ]


    # --------------------------------------------------------
    # Sequence training population should be standardized
    # --------------------------------------------------------

    train_sequence = (
        data[
            "sequence"
        ][
            train_mask
        ]
        .astype(
            np.float64
        )
    )

    sequence_train_mean = (
        train_sequence
        .mean(
            axis=(0, 1)
        )
    )

    sequence_train_std = (
        train_sequence
        .std(
            axis=(0, 1),
            ddof=0,
        )
    )


    assert np.allclose(
        sequence_train_mean,
        0.0,
        atol=1e-6,
        rtol=0.0,
    )

    assert np.allclose(
        sequence_train_std,
        1.0,
        atol=1e-6,
        rtol=0.0,
    )


    # --------------------------------------------------------
    # Target training population should be standardized
    # --------------------------------------------------------

    train_target = (
        data[
            "target_scaled"
        ][
            train_mask
        ]
        .astype(
            np.float64
        )
    )

    assert np.isclose(
        train_target.mean(),
        0.0,
        atol=1e-6,
        rtol=0.0,
    )

    assert np.isclose(
        train_target.std(
            ddof=0
        ),
        1.0,
        atol=1e-6,
        rtol=0.0,
    )


    print(
        f"\n{horizon}-minute horizon"
    )

    print(
        "  sequence:",
        data[
            "sequence"
        ].shape,
    )

    print(
        "  target:",
        data[
            "target_scaled"
        ].shape,
    )

    for variant_name in PHASE2_NEURAL_VARIANTS:

        aux = (
            data[
                "variants"
            ][
                variant_name
            ][
                "auxiliary"
            ]
        )

        print(
            f"  {variant_name:<18}",
            aux.shape,
        )


print(
    "\nPHASE 2A NEURAL INPUT + SCALING AUDIT: PASSED"
)


15-minute horizon
  sequence: (131325, 13, 9)
  target: (131325,)
  F3                 (131325, 30)
  F4_real            (131325, 127)
  F4_reference       (131325, 122)
  F4_shift_17d       (131325, 127)
  F4_shift_37d       (131325, 127)
  F4_shift_61d       (131325, 127)
  F4_perm_1301       (131325, 127)
  F4_perm_1302       (131325, 127)
  F4_perm_1303       (131325, 127)

30-minute horizon
  sequence: (131322, 13, 9)
  target: (131322,)
  F3                 (131322, 30)
  F4_real            (131322, 127)
  F4_reference       (131322, 122)
  F4_shift_17d       (131322, 127)
  F4_shift_37d       (131322, 127)
  F4_shift_61d       (131322, 127)
  F4_perm_1301       (131322, 127)
  F4_perm_1302       (131322, 127)
  F4_perm_1303       (131322, 127)

60-minute horizon
  sequence: (131316, 13, 9)
  target: (131316,)
  F3                 (131316, 30)
  F4_real            (131316, 127)
  F4_reference       (131316, 122)
  F4_shift_17d       (131316, 127)
  F4_shift_37d       (131316, 12

In [122]:
# ============================================================
# Step P2-20: Frozen Phase-2 LSTM / TCN model factory
#             + UNTRAINED structural smoke test
# ============================================================

import random
import numpy as np
import tensorflow as tf


# ------------------------------------------------------------
# 1. Frozen neural training constants from Notebook 11
# ------------------------------------------------------------

PHASE2_NEURAL_SEED = 42
PHASE2_NEURAL_BATCH_SIZE = 256
PHASE2_NEURAL_LEARNING_RATE = 1e-3
PHASE2_NEURAL_CLIPNORM = 1.0
PHASE2_NEURAL_LOSS = "mae"


# ------------------------------------------------------------
# 2. Verify the frozen Phase-1 F4-selected epoch registry
# ------------------------------------------------------------

assert {
    horizon:
        int(
            PHASE2_FROZEN_MODEL_CONFIGS[
                "LSTM"
            ][
                horizon
            ][
                "epochs"
            ]
        )
    for horizon in [
        15,
        30,
        60,
    ]
} == {
    15: 7,
    30: 1,
    60: 2,
}


assert {
    horizon:
        int(
            PHASE2_FROZEN_MODEL_CONFIGS[
                "TCN"
            ][
                horizon
            ][
                "epochs"
            ]
        )
    for horizon in [
        15,
        30,
        60,
    ]
} == {
    15: 4,
    30: 1,
    60: 1,
}


print(
    "Frozen neural epoch audit: PASSED"
)


# ------------------------------------------------------------
# 3. Reproducible fresh-model reset
# ------------------------------------------------------------

def reset_phase2_neural_state(
    seed: int = PHASE2_NEURAL_SEED,
):

    tf.keras.backend.clear_session()

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    tf.keras.utils.set_random_seed(
        seed
    )

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


# ============================================================
# 4. LSTM sequence branch
# ============================================================

def build_phase2_lstm_sequence_branch(
    sequence_input,
    architecture: dict,
):

    units = int(
        architecture[
            "units"
        ]
    )

    n_layers = int(
        architecture[
            "n_recurrent_layers"
        ]
    )

    dropout = float(
        architecture[
            "dropout"
        ]
    )


    assert n_layers in {
        1,
        2,
    }


    x = sequence_input


    for layer_index in range(
        n_layers
    ):

        is_last = (
            layer_index
            ==
            n_layers - 1
        )

        x = tf.keras.layers.LSTM(
            units=units,
            return_sequences=(
                not is_last
            ),
            dropout=dropout,
            recurrent_dropout=0.0,
            name=(
                f"lstm_"
                f"{layer_index + 1}"
            ),
        )(
            x
        )


    return x


# ============================================================
# 5. One causal residual TCN block
# ============================================================

def phase2_tcn_residual_block(
    x,
    filters: int,
    kernel_size: int,
    dilation_rate: int,
    dropout: float,
    convolutions_per_block: int,
    block_index: int,
):

    residual = x


    assert (
        convolutions_per_block
        >= 1
    )


    for convolution_index in range(
        convolutions_per_block
    ):

        x = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            dilation_rate=dilation_rate,
            padding="causal",
            activation="relu",
            name=(
                f"tcn_block_{block_index}_"
                f"conv_{convolution_index + 1}_"
                f"d{dilation_rate}"
            ),
        )(
            x
        )


        if dropout > 0.0:

            x = tf.keras.layers.Dropout(
                rate=dropout,
                name=(
                    f"tcn_block_{block_index}_"
                    f"dropout_{convolution_index + 1}"
                ),
            )(
                x
            )


    # --------------------------------------------------------
    # Residual projection when channel widths differ
    # --------------------------------------------------------

    residual_channels = (
        residual.shape[-1]
    )


    if (
        residual_channels is None
        or
        int(
            residual_channels
        )
        != filters
    ):

        residual = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=1,
            padding="same",
            name=(
                f"tcn_block_{block_index}_"
                "residual_projection"
            ),
        )(
            residual
        )


    x = tf.keras.layers.Add(
        name=(
            f"tcn_block_{block_index}_"
            "residual_add"
        )
    )(
        [
            x,
            residual,
        ]
    )


    x = tf.keras.layers.Activation(
        "relu",
        name=(
            f"tcn_block_{block_index}_"
            "output_relu"
        ),
    )(
        x
    )


    return x


# ============================================================
# 6. TCN sequence branch
# ============================================================

def build_phase2_tcn_sequence_branch(
    sequence_input,
    architecture: dict,
):

    filters = int(
        architecture[
            "filters"
        ]
    )

    kernel_size = int(
        architecture[
            "kernel_size"
        ]
    )

    dropout = float(
        architecture[
            "dropout"
        ]
    )

    dilations = tuple(
        int(value)
        for value in architecture[
            "dilations"
        ]
    )

    convolutions_per_block = int(
        architecture.get(
            "convolutions_per_block",
            2,
        )
    )


    x = sequence_input


    for block_index, dilation_rate in enumerate(
        dilations,
        start=1,
    ):

        x = phase2_tcn_residual_block(
            x=x,
            filters=filters,
            kernel_size=kernel_size,
            dilation_rate=dilation_rate,
            dropout=dropout,
            convolutions_per_block=(
                convolutions_per_block
            ),
            block_index=block_index,
        )


    # Same Notebook-11 temporal summarization
    x = tf.keras.layers.GlobalAveragePooling1D(
        name="tcn_global_average_pool"
    )(
        x
    )


    return x


# ============================================================
# 7. Generic frozen Phase-2 neural model factory
# ============================================================

def build_phase2_neural_model(
    model_name: str,
    auxiliary_dim: int,
    architecture: dict,
    seed: int = PHASE2_NEURAL_SEED,
):

    assert model_name in {
        "LSTM",
        "TCN",
    }

    assert auxiliary_dim > 0


    reset_phase2_neural_state(
        seed=seed
    )


    # ========================================================
    # INPUT 1 — historical sequence
    # ========================================================

    sequence_input = (
        tf.keras.layers.Input(
            shape=(
                13,
                9,
            ),
            dtype=tf.float32,
            name="sequence_input",
        )
    )


    if model_name == "LSTM":

        sequence_embedding = (
            build_phase2_lstm_sequence_branch(
                sequence_input=sequence_input,
                architecture=architecture,
            )
        )


    else:

        sequence_embedding = (
            build_phase2_tcn_sequence_branch(
                sequence_input=sequence_input,
                architecture=architecture,
            )
        )


    # ========================================================
    # INPUT 2 — deterministic auxiliary representation
    # ========================================================

    auxiliary_input = (
        tf.keras.layers.Input(
            shape=(
                auxiliary_dim,
            ),
            dtype=tf.float32,
            name="auxiliary_input",
        )
    )


    # ========================================================
    # Fusion
    # ========================================================

    fused = tf.keras.layers.Concatenate(
        name="sequence_auxiliary_fusion"
    )(
        [
            sequence_embedding,
            auxiliary_input,
        ]
    )


    fusion_dense_units = int(
        architecture[
            "fusion_dense_units"
        ]
    )


    fused = tf.keras.layers.Dense(
        units=fusion_dense_units,
        activation="relu",
        name="fusion_dense",
    )(
        fused
    )


    output = tf.keras.layers.Dense(
        units=1,
        activation="linear",
        name="scaled_p_solar_output",
    )(
        fused
    )


    model = tf.keras.Model(
        inputs=[
            sequence_input,
            auxiliary_input,
        ],
        outputs=output,
        name=(
            f"phase2_"
            f"{model_name.lower()}"
        ),
    )


    optimizer = (
        tf.keras.optimizers.Adam(
            learning_rate=(
                PHASE2_NEURAL_LEARNING_RATE
            ),
            clipnorm=(
                PHASE2_NEURAL_CLIPNORM
            ),
        )
    )


    model.compile(
        optimizer=optimizer,
        loss=PHASE2_NEURAL_LOSS,
    )


    return model


# ============================================================
# 8. UNTRAINED smoke test
#
# Use:
#
#   horizon = 60 min
#   variants = F3 and F4_real
#   models   = LSTM and TCN
#
# NO TRAINING occurs.
# ============================================================

SMOKE_HORIZON = 60
SMOKE_BATCH_SIZE = 8

smoke_data = (
    phase2_neural_data[
        SMOKE_HORIZON
    ]
)

test_positions = np.flatnonzero(
    smoke_data[
        "test_mask"
    ]
)

assert len(
    test_positions
) == 26_496


smoke_positions = (
    test_positions[
        :SMOKE_BATCH_SIZE
    ]
)


smoke_sequence = (
    smoke_data[
        "sequence"
    ][
        smoke_positions
    ]
)


assert smoke_sequence.shape == (
    8,
    13,
    9,
)


smoke_rows = []


for model_name in [
    "LSTM",
    "TCN",
]:

    frozen_config = (
        PHASE2_FROZEN_MODEL_CONFIGS[
            model_name
        ][
            SMOKE_HORIZON
        ]
    )

    architecture = (
        frozen_config[
            "architecture"
        ]
        .copy()
    )


    for variant_name in [
        "F3",
        "F4_real",
    ]:

        smoke_auxiliary = (
            smoke_data[
                "variants"
            ][
                variant_name
            ][
                "auxiliary"
            ][
                smoke_positions
            ]
        )


        auxiliary_dim = int(
            smoke_auxiliary.shape[
                1
            ]
        )


        model = (
            build_phase2_neural_model(
                model_name=model_name,
                auxiliary_dim=auxiliary_dim,
                architecture=architecture,
                seed=42,
            )
        )


        smoke_prediction = (
            model(
                [
                    smoke_sequence,
                    smoke_auxiliary,
                ],
                training=False,
            )
            .numpy()
            .reshape(
                -1
            )
        )


        assert smoke_prediction.shape == (
            SMOKE_BATCH_SIZE,
        )

        assert np.isfinite(
            smoke_prediction
        ).all()


        smoke_rows.append(
            {
                "model":
                    model_name,

                "variant":
                    variant_name,

                "auxiliary_dim":
                    auxiliary_dim,

                "parameters":
                    int(
                        model.count_params()
                    ),

                "output_shape":
                    tuple(
                        model.output.shape
                    ),

                "all_finite":
                    bool(
                        np.isfinite(
                            smoke_prediction
                        ).all()
                    ),

                "frozen_epochs":
                    int(
                        frozen_config[
                            "epochs"
                        ]
                    ),
            }
        )


phase2_neural_smoke_audit = (
    pd.DataFrame(
        smoke_rows
    )
)


display(
    phase2_neural_smoke_audit
)


# ============================================================
# 9. Frozen architecture report
# ============================================================

print(
    "\nFrozen LSTM architecture:"
)

print(
    PHASE2_FROZEN_MODEL_CONFIGS[
        "LSTM"
    ][
        60
    ][
        "architecture"
    ]
)


print(
    "\nFrozen TCN architecture:"
)

print(
    PHASE2_FROZEN_MODEL_CONFIGS[
        "TCN"
    ][
        60
    ][
        "architecture"
    ]
)


print(
    "\nPHASE 2A NEURAL MODEL FACTORY + SMOKE TEST: PASSED"
)

print(
    "No neural model has been trained yet."
)

Frozen neural epoch audit: PASSED


2026-09-29 20:31:57.467423: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-09-29 20:31:57.467484: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-09-29 20:31:57.467497: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1790694117.467559 51828129 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1790694117.467629 51828129 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


,model,variant,auxiliary_dim,parameters,output_shape,all_finite,frozen_epochs
0,LSTM,F3,30,7425,"(None, 1)",True,2
1,LSTM,F4_real,127,10529,"(None, 1)",True,2
2,TCN,F3,30,18785,"(None, 1)",True,1
3,TCN,F4_real,127,21889,"(None, 1)",True,1



Frozen LSTM architecture:
{'candidate_id': 'RNN_C1', 'units': 32, 'n_recurrent_layers': 1, 'dropout': 0.0, 'fusion_dense_units': 32}

Frozen TCN architecture:
{'candidate_id': 'TCN_C1', 'filters': 32, 'kernel_size': 3, 'dilations': [1, 2, 4], 'convolutions_per_block': 2, 'dropout': 0.0, 'fusion_dense_units': 32, 'receptive_field_timesteps': 29}

PHASE 2A NEURAL MODEL FACTORY + SMOKE TEST: PASSED
No neural model has been trained yet.


In [128]:
# ============================================================
# Step P2-21B: FINAL corrected Phase-2 neural refit helper
#
# Replaces the previous P2-21 function.
#
# LSTM:
#   exact Notebook-11 generic recurrent architecture.
#
# TCN:
#   corrected Notebook-11 FINAL TCN architecture:
#       causal convolutions
#       He-normal initialization
#       causal residual projection
#       LAST temporal position
#
# Scientific policy:
#   every Phase-2 representation is trained under the SAME
#   current TensorFlow environment and frozen configuration.
#
# THIS CELL PERFORMS NO MODEL TRAINING.
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


# ============================================================
# 1. Frozen current-environment protocol identifier
# ============================================================

PHASE2_NEURAL_EXECUTION_PROTOCOL = (
    "phase2_common_current_environment_v1"
)


# ============================================================
# 2. Final corrected helper
# ============================================================

def run_phase2_neural_final_refit(
    model_name: str,
    horizon_minutes: int,
    variant_name: str,
    verbose: int = 2,
):
    """
    Run one frozen Phase-2 neural representation experiment.

    Important
    ---------
    Every representation uses:

        architecture:
            frozen Phase-1 F4-selected architecture

        epochs:
            frozen Phase-1 F4-selected epoch count

        initialization seed:
            42

        preprocessing:
            original TRAIN-only scalers

        final fitting population:
            TRAIN followed by VALIDATION

        batch size:
            256

        shuffle:
            True

        validation during final refit:
            none

        early stopping:
            none

        prediction clipping:
            none

    TEST is evaluated only after the exact frozen refit.
    """

    # --------------------------------------------------------
    # A. Normalize / validate experiment identity
    # --------------------------------------------------------

    model_name = str(
        model_name
    )

    horizon_minutes = int(
        horizon_minutes
    )

    variant_name = str(
        variant_name
    )


    assert model_name in {
        "LSTM",
        "TCN",
    }


    assert horizon_minutes in {
        15,
        30,
        60,
    }


    assert (
        variant_name
        in PHASE2_NEURAL_VARIANTS
    )


    # ========================================================
    # B. Retrieve frozen tensors
    # ========================================================

    data = (
        phase2_neural_data[
            horizon_minutes
        ]
    )


    variant_data = (
        data[
            "variants"
        ][
            variant_name
        ]
    )


    sequence_all = np.asarray(
        data[
            "sequence"
        ],
        dtype=np.float32,
    )


    auxiliary_all = np.asarray(
        variant_data[
            "auxiliary"
        ],
        dtype=np.float32,
    )


    target_scaled_all = np.asarray(
        data[
            "target_scaled"
        ],
        dtype=np.float32,
    )


    target_kw_all = np.asarray(
        data[
            "target_kw"
        ],
        dtype=np.float32,
    )


    train_mask = np.asarray(
        data[
            "train_mask"
        ],
        dtype=bool,
    )


    validation_mask = np.asarray(
        data[
            "validation_mask"
        ],
        dtype=bool,
    )


    test_mask = np.asarray(
        data[
            "test_mask"
        ],
        dtype=bool,
    )


    # ========================================================
    # C. Population audit
    # ========================================================

    expected_train_counts = {
        15: 87_261,
        30: 87_258,
        60: 87_252,
    }


    n_train = int(
        train_mask.sum()
    )


    n_validation = int(
        validation_mask.sum()
    )


    n_test = int(
        test_mask.sum()
    )


    assert (
        n_train
        ==
        expected_train_counts[
            horizon_minutes
        ]
    )


    assert (
        n_validation
        ==
        17_568
    )


    assert (
        n_test
        ==
        26_496
    )


    n_trainval = (
        n_train
        +
        n_validation
    )


    # ========================================================
    # D. Retrieve TRAIN
    # ========================================================

    train_sequence = np.asarray(
        sequence_all[
            train_mask
        ],
        dtype=np.float32,
    )


    train_auxiliary = np.asarray(
        auxiliary_all[
            train_mask
        ],
        dtype=np.float32,
    )


    train_target_scaled = np.asarray(
        target_scaled_all[
            train_mask
        ],
        dtype=np.float32,
    )


    # ========================================================
    # E. Retrieve VALIDATION
    # ========================================================

    validation_sequence = np.asarray(
        sequence_all[
            validation_mask
        ],
        dtype=np.float32,
    )


    validation_auxiliary = np.asarray(
        auxiliary_all[
            validation_mask
        ],
        dtype=np.float32,
    )


    validation_target_scaled = np.asarray(
        target_scaled_all[
            validation_mask
        ],
        dtype=np.float32,
    )


    # ========================================================
    # F. Retrieve TEST
    # ========================================================

    test_sequence = np.asarray(
        sequence_all[
            test_mask
        ],
        dtype=np.float32,
    )


    test_auxiliary = np.asarray(
        auxiliary_all[
            test_mask
        ],
        dtype=np.float32,
    )


    test_target_kw = np.asarray(
        target_kw_all[
            test_mask
        ],
        dtype=np.float64,
    ).reshape(
        -1
    )


    auxiliary_dim = int(
        auxiliary_all.shape[
            1
        ]
    )


    # ========================================================
    # G. Strong split-shape audit
    # ========================================================

    assert train_sequence.shape == (
        n_train,
        13,
        9,
    )


    assert validation_sequence.shape == (
        n_validation,
        13,
        9,
    )


    assert test_sequence.shape == (
        n_test,
        13,
        9,
    )


    assert train_auxiliary.shape == (
        n_train,
        auxiliary_dim,
    )


    assert validation_auxiliary.shape == (
        n_validation,
        auxiliary_dim,
    )


    assert test_auxiliary.shape == (
        n_test,
        auxiliary_dim,
    )


    # ========================================================
    # H. Concatenate TRAIN then VALIDATION
    #
    # Explicit concatenation mirrors Notebook 11 exactly.
    # ========================================================

    trainval_sequence = np.concatenate(
        [
            train_sequence,
            validation_sequence,
        ],
        axis=0,
    )


    trainval_auxiliary = np.concatenate(
        [
            train_auxiliary,
            validation_auxiliary,
        ],
        axis=0,
    )


    trainval_target_scaled = np.concatenate(
        [
            train_target_scaled,
            validation_target_scaled,
        ],
        axis=0,
    )


    assert trainval_sequence.shape == (
        n_trainval,
        13,
        9,
    )


    assert trainval_auxiliary.shape == (
        n_trainval,
        auxiliary_dim,
    )


    # ========================================================
    # I. Preserve the model-specific Notebook-11 target shape
    #
    # LSTM final helper:
    #       target shape (n,)
    #
    # TCN final helper:
    #       target shape (n, 1)
    # ========================================================

    if model_name == "LSTM":

        final_fit_target = (
            trainval_target_scaled
            .reshape(
                -1
            )
        )


        assert final_fit_target.shape == (
            n_trainval,
        )


    else:

        final_fit_target = (
            trainval_target_scaled
            .reshape(
                -1,
                1,
            )
        )


        assert final_fit_target.shape == (
            n_trainval,
            1,
        )


    # ========================================================
    # J. Frozen architecture + epoch
    # ========================================================

    frozen_config = (
        PHASE2_FROZEN_MODEL_CONFIGS[
            model_name
        ][
            horizon_minutes
        ]
    )


    architecture = (
        frozen_config[
            "architecture"
        ]
        .copy()
    )


    frozen_epochs = int(
        frozen_config[
            "epochs"
        ]
    )


    expected_epoch_registry = {

        "LSTM": {
            15: 7,
            30: 1,
            60: 2,
        },

        "TCN": {
            15: 4,
            30: 1,
            60: 1,
        },
    }


    assert (
        frozen_epochs
        ==
        expected_epoch_registry[
            model_name
        ][
            horizon_minutes
        ]
    )


    # ========================================================
    # K. Fresh deterministic initialization
    # ========================================================

    model = (
        build_phase2_neural_model(

            model_name=model_name,

            auxiliary_dim=auxiliary_dim,

            architecture=architecture,

            seed=PHASE2_NEURAL_SEED,
        )
    )


    # --------------------------------------------------------
    # Parameter-count audit
    # --------------------------------------------------------

    expected_parameters = (
        expected_phase2_neural_parameter_count(

            model_name=model_name,

            auxiliary_dim=auxiliary_dim,
        )
    )


    actual_parameters = int(
        model.count_params()
    )


    assert (
        actual_parameters
        ==
        expected_parameters
    )


    # ========================================================
    # L. Architecture-specific audit
    # ========================================================

    if model_name == "LSTM":

        lstm_layers = [
            layer
            for layer
            in model.layers
            if isinstance(
                layer,
                tf.keras.layers.LSTM,
            )
        ]


        assert (
            len(
                lstm_layers
            )
            ==
            1
        )


        assert (
            int(
                lstm_layers[
                    0
                ].units
            )
            ==
            32
        )


        assert np.isclose(
            float(
                lstm_layers[
                    0
                ].dropout
            ),
            0.0,
        )


        assert np.isclose(
            float(
                lstm_layers[
                    0
                ].recurrent_dropout
            ),
            0.0,
        )


    else:

        tcn_conv_layers = [
            layer
            for layer
            in model.layers
            if isinstance(
                layer,
                tf.keras.layers.Conv1D,
            )
        ]


        assert (
            len(
                tcn_conv_layers
            )
            ==
            7
        )


        assert all(
            layer.padding
            ==
            "causal"

            for layer
            in tcn_conv_layers
        )


        assert all(
            tuple(
                layer.strides
            )
            ==
            (
                1,
            )

            for layer
            in tcn_conv_layers
        )


        assert any(
            layer.name
            ==
            "tcn_last_position"

            for layer
            in model.layers
        )


        assert not any(
            isinstance(
                layer,
                tf.keras.layers.GlobalAveragePooling1D,
            )

            for layer
            in model.layers
        )


    # ========================================================
    # M. Exact frozen final refit
    # ========================================================

    fit_start = (
        time.perf_counter()
    )


    history_object = model.fit(

        x=[
            trainval_sequence,
            trainval_auxiliary,
        ],

        y=final_fit_target,

        epochs=frozen_epochs,

        batch_size=(
            PHASE2_NEURAL_BATCH_SIZE
        ),

        shuffle=True,

        callbacks=None,

        validation_data=None,

        verbose=verbose,
    )


    fit_seconds = float(
        time.perf_counter()
        -
        fit_start
    )


    # ========================================================
    # N. History audit
    # ========================================================

    history_df = pd.DataFrame(
        history_object.history
    )


    history_df.insert(
        0,
        "epoch",
        np.arange(
            1,
            len(
                history_df
            )
            +
            1,
            dtype=int,
        ),
    )


    assert (
        len(
            history_df
        )
        ==
        frozen_epochs
    )


    assert (
        "loss"
        in
        history_df.columns
    )


    assert (
        "val_loss"
        not in
        history_df.columns
    )


    # ========================================================
    # O. Held-out TEST prediction
    # ========================================================

    predict_start = (
        time.perf_counter()
    )


    prediction_scaled = np.asarray(

        model.predict(

            [
                test_sequence,
                test_auxiliary,
            ],

            batch_size=(
                PHASE2_NEURAL_BATCH_SIZE
            ),

            verbose=0,
        ),

        dtype=np.float64,

    ).reshape(
        -1
    )


    predict_seconds = float(
        time.perf_counter()
        -
        predict_start
    )


    assert prediction_scaled.shape == (
        n_test,
    )


    assert np.isfinite(
        prediction_scaled
    ).all()


    # ========================================================
    # P. Inverse target transform
    # ========================================================

    scaling_stats = (
        phase2_neural_scaling_statistics[
            horizon_minutes
        ]
    )


    target_mean = float(
        scaling_stats[
            "target_mean"
        ]
    )


    target_std = float(
        scaling_stats[
            "target_std"
        ]
    )


    prediction_kw = (
        target_mean
        +
        target_std
        *
        prediction_scaled
    )


    assert np.isfinite(
        prediction_kw
    ).all()


    # ========================================================
    # Q. All-row metrics
    # ========================================================

    all_mae = float(
        mean_absolute_error(
            test_target_kw,
            prediction_kw,
        )
    )


    all_rmse = float(
        np.sqrt(
            mean_squared_error(
                test_target_kw,
                prediction_kw,
            )
        )
    )


    all_r2 = float(
        r2_score(
            test_target_kw,
            prediction_kw,
        )
    )


    # ========================================================
    # R. TEST metadata / daylight
    # ========================================================

    bundle = (
        modelling_bundles[
            horizon_minutes
        ]
    )


    metadata_test = (
        bundle[
            "metadata"
        ]
        .loc[
            bundle[
                "test_mask"
            ]
        ]
        .copy()
    )


    daylight_test = (
        bundle[
            "daylight_mask"
        ]
        .loc[
            bundle[
                "test_mask"
            ]
        ]
        .to_numpy(
            dtype=bool,
        )
    )


    assert len(
        metadata_test
    ) == n_test


    assert daylight_test.shape == (
        n_test,
    )


    # ========================================================
    # S. Daylight metrics
    # ========================================================

    daylight_mae = float(
        mean_absolute_error(

            test_target_kw[
                daylight_test
            ],

            prediction_kw[
                daylight_test
            ],
        )
    )


    daylight_rmse = float(
        np.sqrt(
            mean_squared_error(

                test_target_kw[
                    daylight_test
                ],

                prediction_kw[
                    daylight_test
                ],
            )
        )
    )


    daylight_r2 = float(
        r2_score(

            test_target_kw[
                daylight_test
            ],

            prediction_kw[
                daylight_test
            ],
        )
    )


    # ========================================================
    # T. Prediction table
    # ========================================================

    prediction_table = pd.DataFrame(
        {
            "issue_time":
                metadata_test[
                    "issue_time"
                ]
                .to_numpy(),

            "valid_time":
                metadata_test[
                    "valid_time"
                ]
                .to_numpy(),

            "daylight":
                daylight_test,

            "target_P_Solar_kW":
                test_target_kw,

            "prediction_kW":
                prediction_kw,

            "prediction_scaled":
                prediction_scaled,
        },
        index=metadata_test.index,
    )


    prediction_table.index.name = (
        "Timestamp"
    )


    # ========================================================
    # U. Result record
    # ========================================================

    result = {

        "execution_protocol":
            PHASE2_NEURAL_EXECUTION_PROTOCOL,

        "model":
            model_name,

        "horizon_minutes":
            horizon_minutes,

        "variant":
            variant_name,

        "auxiliary_dim":
            auxiliary_dim,

        "trainable_parameter_count":
            actual_parameters,

        "n_train":
            n_train,

        "n_validation":
            n_validation,

        "n_trainval":
            n_trainval,

        "n_test":
            n_test,

        "frozen_epochs":
            frozen_epochs,

        "batch_size":
            PHASE2_NEURAL_BATCH_SIZE,

        "seed":
            PHASE2_NEURAL_SEED,

        "shuffle":
            True,

        "validation_data_used":
            False,

        "callbacks_used":
            False,

        "prediction_clipping":
            False,

        "scaler_policy":
            "original_train_only_scalers",

        "refit_population":
            "train_plus_validation",

        "MAE_kW":
            all_mae,

        "RMSE_kW":
            all_rmse,

        "R2":
            all_r2,

        "daylight_rows":
            int(
                daylight_test.sum()
            ),

        "daylight_MAE_kW":
            daylight_mae,

        "daylight_RMSE_kW":
            daylight_rmse,

        "daylight_R2":
            daylight_r2,

        "fit_seconds":
            fit_seconds,

        "predict_seconds":
            predict_seconds,
    }


    return (
        result,
        history_df,
        prediction_table,
    )


# ============================================================
# 3. Definition-only checkpoint
# ============================================================

print(
    "PHASE 2A FINAL CORRECTED NEURAL REFIT HELPER: PASSED"
)

print(
    "Execution protocol:",
    PHASE2_NEURAL_EXECUTION_PROTOCOL,
)

print(
    "LSTM factory: verified against Notebook 11"
)

print(
    "TCN factory: corrected final Notebook-11 implementation"
)

print(
    "No model was trained."
)

PHASE 2A FINAL CORRECTED NEURAL REFIT HELPER: PASSED
Execution protocol: phase2_common_current_environment_v1
LSTM factory: verified against Notebook 11
TCN factory: corrected final Notebook-11 implementation
No model was trained.


In [129]:
# ============================================================
# Step P2-22C: Exact Notebook-11 preprocessing reproduction
#              audit
#
# NO MODEL TRAINING occurs here.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Exact Notebook-11 target-scaler references
# ============================================================

NOTEBOOK11_TARGET_REFERENCES = {

    15: {
        "mean": 0.838792218,
        "std":  1.587853895,
    },

    30: {
        "mean": 0.838821062,
        "std":  1.587873570,
    },

    60: {
        "mean": 0.838878755,
        "std":  1.587912924,
    },
}


target_audit_rows = []


for horizon in [15, 30, 60]:

    current_mean = float(
        phase2_neural_scaling_statistics[
            horizon
        ][
            "target_mean"
        ]
    )

    current_std = float(
        phase2_neural_scaling_statistics[
            horizon
        ][
            "target_std"
        ]
    )

    reference_mean = (
        NOTEBOOK11_TARGET_REFERENCES[
            horizon
        ][
            "mean"
        ]
    )

    reference_std = (
        NOTEBOOK11_TARGET_REFERENCES[
            horizon
        ][
            "std"
        ]
    )


    target_audit_rows.append(
        {
            "horizon_minutes":
                horizon,

            "Notebook11_mean":
                reference_mean,

            "Phase2_mean":
                current_mean,

            "mean_abs_diff":
                abs(
                    current_mean
                    -
                    reference_mean
                ),

            "Notebook11_std":
                reference_std,

            "Phase2_std":
                current_std,

            "std_abs_diff":
                abs(
                    current_std
                    -
                    reference_std
                ),
        }
    )


target_scaling_reproduction = pd.DataFrame(
    target_audit_rows
)


print(
    "=" * 90
)

print(
    "TARGET SCALING REPRODUCTION"
)

print(
    "=" * 90
)

display(
    target_scaling_reproduction
)


# ============================================================
# 2. Exact Notebook-11 60-minute sequence references
#
# Channel order:
#
#   P_Solar
#   GHI
#   POA
#   Temperature
#   Humidity
#   Pressure
#   Wind speed
#   Wind-direction sin
#   Wind-direction cos
# ============================================================

NOTEBOOK11_SEQUENCE_CHANNELS = [
    "P_Solar[kW]",
    "GHI[kW1m2]",
    "POA Irr[kW1m2]",
    "TEMPERATURE[degC]",
    "Humidity_clean",
    "Pressure_clean[mbar]",
    "WIND_SPEED[m1s]",
    "wind_direction_sin",
    "wind_direction_cos",
]


NOTEBOOK11_SEQUENCE_MEAN_60 = np.array(
    [
        0.838879,
        0.102510,
        0.119594,
        10.488891,
        0.794023,
        1008.836182,
        3.773284,
        -0.293050,
        -0.312508,
    ],
    dtype=float,
)


NOTEBOOK11_SEQUENCE_STD_60 = np.array(
    [
        1.587913,
        0.182455,
        0.228358,
        7.479582,
        0.146613,
        10.187528,
        2.179404,
        0.707097,
        0.562560,
    ],
    dtype=float,
)


phase2_sequence_mean_60 = np.asarray(
    phase2_neural_scaling_statistics[
        60
    ][
        "sequence_mean"
    ],
    dtype=float,
)


phase2_sequence_std_60 = np.asarray(
    phase2_neural_scaling_statistics[
        60
    ][
        "sequence_std"
    ],
    dtype=float,
)


sequence_scaling_reproduction = pd.DataFrame(
    {
        "channel":
            NOTEBOOK11_SEQUENCE_CHANNELS,

        "Notebook11_mean":
            NOTEBOOK11_SEQUENCE_MEAN_60,

        "Phase2_mean":
            phase2_sequence_mean_60,

        "mean_abs_diff":
            np.abs(
                phase2_sequence_mean_60
                -
                NOTEBOOK11_SEQUENCE_MEAN_60
            ),

        "Notebook11_std":
            NOTEBOOK11_SEQUENCE_STD_60,

        "Phase2_std":
            phase2_sequence_std_60,

        "std_abs_diff":
            np.abs(
                phase2_sequence_std_60
                -
                NOTEBOOK11_SEQUENCE_STD_60
            ),
    }
)


print(
    "\n"
    + "=" * 90
)

print(
    "60-MINUTE SEQUENCE SCALING REPRODUCTION"
)

print(
    "=" * 90
)

display(
    sequence_scaling_reproduction
)


# ============================================================
# 3. F4 auxiliary partition audit
#
# Notebook 11:
#
#   continuous = 28
#
#   binary =
#       weekend      1
#       daylight     1
#       Panchang    97
#       ----------------
#                   99
#
#   total = 127
# ============================================================

f4_variant_info = (
    phase2_neural_data[
        60
    ][
        "variants"
    ][
        "F4_real"
    ]
)


f4_continuous_columns = list(
    f4_variant_info[
        "continuous_columns"
    ]
)


f4_binary_columns = list(
    f4_variant_info[
        "binary_columns"
    ]
)


print(
    "\n"
    + "=" * 90
)

print(
    "F4 AUXILIARY PARTITION"
)

print(
    "=" * 90
)


print(
    "Continuous columns:",
    len(
        f4_continuous_columns
    )
)

print(
    "Binary columns:",
    len(
        f4_binary_columns
    )
)

print(
    "Total:",
    len(
        f4_continuous_columns
    )
    +
    len(
        f4_binary_columns
    )
)


assert (
    len(
        f4_continuous_columns
    )
    ==
    28
)

assert (
    len(
        f4_binary_columns
    )
    ==
    99
)

assert (
    len(
        f4_variant_info[
            "auxiliary_columns"
        ]
    )
    ==
    127
)


# ============================================================
# 4. Continuous F4 auxiliary scaling statistics
#
# Notebook 11 60-minute reference:
#
# smallest std:
#   moon_illumination_fraction
#   0.34829363296764254
#
# largest std:
#   earth_moon_distance_km
#   15701.457440019216
# ============================================================

phase2_aux_std_60 = np.asarray(
    f4_variant_info[
        "auxiliary_std"
    ],
    dtype=float,
)


min_position = int(
    np.argmin(
        phase2_aux_std_60
    )
)


max_position = int(
    np.argmax(
        phase2_aux_std_60
    )
)


phase2_min_column = (
    f4_continuous_columns[
        min_position
    ]
)


phase2_max_column = (
    f4_continuous_columns[
        max_position
    ]
)


phase2_min_std = float(
    phase2_aux_std_60[
        min_position
    ]
)


phase2_max_std = float(
    phase2_aux_std_60[
        max_position
    ]
)


print(
    "\n"
    + "=" * 90
)

print(
    "60-MINUTE CONTINUOUS AUXILIARY SCALING"
)

print(
    "=" * 90
)


print(
    "Smallest std:"
)

print(
    " ",
    phase2_min_column,
    phase2_min_std,
)


print(
    "\nLargest std:"
)

print(
    " ",
    phase2_max_column,
    phase2_max_std,
)


# ============================================================
# 5. Panchang column-order audit
# ============================================================

phase2_panchang_columns = [
    column
    for column
    in f4_variant_info[
        "auxiliary_columns"
    ]
    if column.startswith(
        (
            "valid_tithi_",
            "valid_paksha_",
            "valid_karana_",
            "valid_nakshatra_",
            "valid_yoga_",
        )
    )
]


assert len(
    phase2_panchang_columns
) == 97


print(
    "\n"
    + "=" * 90
)

print(
    "PANCHANG COLUMN ORDER"
)

print(
    "=" * 90
)


print(
    "First 10:"
)

for column in (
    phase2_panchang_columns[
        :10
    ]
):
    print(
        " ",
        column
    )


print(
    "\nLast 10:"
)

for column in (
    phase2_panchang_columns[
        -10:
    ]
):
    print(
        " ",
        column
    )


# Expected canonical category ordering.

EXPECTED_FIRST_10 = [
    f"valid_tithi_{i}"
    for i in range(
        1,
        11,
    )
]


EXPECTED_LAST_10 = [
    "valid_yoga_Variyana",
    "valid_yoga_Parigha",
    "valid_yoga_Shiva",
    "valid_yoga_Siddha",
    "valid_yoga_Sadhya",
    "valid_yoga_Shubha",
    "valid_yoga_Shukla",
    "valid_yoga_Brahma",
    "valid_yoga_Indra",
    "valid_yoga_Vaidhriti",
]


assert (
    phase2_panchang_columns[
        :10
    ]
    ==
    EXPECTED_FIRST_10
)


assert (
    phase2_panchang_columns[
        -10:
    ]
    ==
    EXPECTED_LAST_10
)


# ============================================================
# 6. Raw first-row F4 identity audit
#
# Notebook-11 first 60-minute valid row:
#
# tithi      = 19
# paksha     = Krishna
# karana     = Bava
# nakshatra  = Purva Ashadha
# yoga       = Shukla
# ============================================================

first_f4_row = (
    phase2_variant_tables[
        60
    ][
        "F4_real"
    ]
    .iloc[
        0
    ]
)


first_active_panchang = [
    column
    for column
    in phase2_panchang_columns
    if float(
        first_f4_row[
            column
        ]
    )
    ==
    1.0
]


EXPECTED_FIRST_ACTIVE = [
    "valid_tithi_19",
    "valid_paksha_Krishna",
    "valid_karana_Bava",
    "valid_nakshatra_Purva Ashadha",
    "valid_yoga_Shukla",
]


print(
    "\nFirst 60-minute active Panchang indicators:"
)

for column in first_active_panchang:
    print(
        " ",
        column
    )


assert (
    first_active_panchang
    ==
    EXPECTED_FIRST_ACTIVE
)


# ============================================================
# 7. Numerical reproduction checks
# ============================================================

# Target transforms should essentially match exactly.

assert (
    target_scaling_reproduction[
        "mean_abs_diff"
    ].max()
    <
    1e-6
)

assert (
    target_scaling_reproduction[
        "std_abs_diff"
    ].max()
    <
    1e-6
)


# Sequence references printed in Notebook 11 were rounded
# to six decimals, so use a small tolerance.

assert (
    sequence_scaling_reproduction[
        "mean_abs_diff"
    ].max()
    <
    2e-6
)

assert (
    sequence_scaling_reproduction[
        "std_abs_diff"
    ].max()
    <
    2e-6
)


# Auxiliary extrema should reproduce the canonical features.

assert (
    phase2_min_column
    ==
    "valid_moon_illumination_fraction"
    or
    phase2_min_column
    ==
    "moon_illumination_fraction"
)


assert (
    abs(
        phase2_min_std
        -
        0.34829363296764254
    )
    <
    1e-6
)


assert (
    phase2_max_column
    ==
    "valid_earth_moon_distance_km"
    or
    phase2_max_column
    ==
    "earth_moon_distance_km"
)


assert (
    abs(
        phase2_max_std
        -
        15701.457440019216
    )
    <
    1e-3
)


print(
    "\n"
    + "=" * 90
)

print(
    "PHASE 2A NOTEBOOK-11 PREPROCESSING REPRODUCTION AUDIT: PASSED"
)

print(
    "=" * 90
)

print(
    "No model was trained."
)

TARGET SCALING REPRODUCTION


,horizon_minutes,Notebook11_mean,Phase2_mean,mean_abs_diff,Notebook11_std,Phase2_std,std_abs_diff
0,15,0.838792,0.838792,9.780282e-09,1.587854,1.587854,5.369781e-09
1,30,0.838821,0.838821,4.129584e-09,1.587874,1.587874,1.914082e-09
2,60,0.838879,0.838879,6.222491e-09,1.587913,1.587913,2.753748e-09



60-MINUTE SEQUENCE SCALING REPRODUCTION


,channel,Notebook11_mean,Phase2_mean,mean_abs_diff,Notebook11_std,Phase2_std,std_abs_diff
0,P_Solar[kW],0.838879,0.838879,2.512225e-07,1.587913,1.587913,7.324715e-08
1,GHI[kW1m2],0.102510,0.102510,4.645050e-07,0.182455,0.182455,6.623460e-08
2,POA Irr[kW1m2],0.119594,0.119594,1.982695e-07,0.228358,0.228358,3.277317e-07
3,TEMPERATURE[degC],10.488891,10.488891,2.640267e-07,7.479582,7.479582,4.151652e-07
4,Humidity_clean,0.794023,0.794023,1.319034e-07,0.146613,0.146613,1.237983e-07
5,Pressure_clean[mbar],1008.836182,1008.836182,2.366761e-07,10.187528,10.187528,4.788541e-07
6,WIND_SPEED[m1s],3.773284,3.773284,3.579449e-07,2.179404,2.179404,3.406264e-07
7,wind_direction_sin,-0.293050,-0.293050,4.493288e-07,0.707097,0.707097,3.454965e-07
8,wind_direction_cos,-0.312508,-0.312508,6.636115e-08,0.562560,0.562560,2.371122e-07



F4 AUXILIARY PARTITION
Continuous columns: 28
Binary columns: 99
Total: 127

60-MINUTE CONTINUOUS AUXILIARY SCALING
Smallest std:
  valid_moon_illumination_fraction 0.34829363296764254

Largest std:
  valid_earth_moon_distance_km 15701.457440019216

PANCHANG COLUMN ORDER
First 10:
  valid_tithi_1
  valid_tithi_2
  valid_tithi_3
  valid_tithi_4
  valid_tithi_5
  valid_tithi_6
  valid_tithi_7
  valid_tithi_8
  valid_tithi_9
  valid_tithi_10

Last 10:
  valid_yoga_Variyana
  valid_yoga_Parigha
  valid_yoga_Shiva
  valid_yoga_Siddha
  valid_yoga_Sadhya
  valid_yoga_Shubha
  valid_yoga_Shukla
  valid_yoga_Brahma
  valid_yoga_Indra
  valid_yoga_Vaidhriti

First 60-minute active Panchang indicators:
  valid_tithi_19
  valid_paksha_Krishna
  valid_karana_Bava
  valid_nakshatra_Purva Ashadha
  valid_yoga_Shukla

PHASE 2A NOTEBOOK-11 PREPROCESSING REPRODUCTION AUDIT: PASSED
No model was trained.


In [130]:
# ============================================================
# Step P2-23: RESUMABLE Phase-2 neural core grid
#
# Models:
#   LSTM
#   TCN
#
# Horizons:
#   15, 30, 60 minutes
#
# Representations:
#   F3
#   F4_real
#   F4_reference
#   F4_shift_17d
#   F4_shift_37d
#   F4_shift_61d
#   F4_perm_1301
#   F4_perm_1302
#   F4_perm_1303
#
# Total:
#   2 × 3 × 9 = 54 final-refit experiments
#
# RESUMABILITY:
#   Every completed experiment is persisted immediately.
#   Re-running this cell skips already-complete valid runs.
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd


# ============================================================
# 1. Frozen execution grid
# ============================================================

PHASE2_NEURAL_GRID_MODELS = [
    "LSTM",
    "TCN",
]


PHASE2_NEURAL_GRID_HORIZONS = [
    15,
    30,
    60,
]


PHASE2_NEURAL_GRID_VARIANTS = [
    "F3",
    "F4_real",
    "F4_reference",
    "F4_shift_17d",
    "F4_shift_37d",
    "F4_shift_61d",
    "F4_perm_1301",
    "F4_perm_1302",
    "F4_perm_1303",
]


assert (
    PHASE2_NEURAL_GRID_VARIANTS
    ==
    PHASE2_NEURAL_VARIANTS
)


PHASE2_NEURAL_TOTAL_RUNS = (
    len(
        PHASE2_NEURAL_GRID_MODELS
    )
    *
    len(
        PHASE2_NEURAL_GRID_HORIZONS
    )
    *
    len(
        PHASE2_NEURAL_GRID_VARIANTS
    )
)


assert (
    PHASE2_NEURAL_TOTAL_RUNS
    ==
    54
)


# ============================================================
# 2. Dedicated output directory
#
# Keep this separate from the earlier diagnostic attempts.
# ============================================================

PHASE2_NEURAL_GRID_DIR = (
    PHASE2_RESULTS_DIR
    /
    "neural_core_grid_v1"
)


PHASE2_NEURAL_GRID_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Neural grid directory:"
)

print(
    " ",
    PHASE2_NEURAL_GRID_DIR
)


# ============================================================
# 3. Run identifier helpers
# ============================================================

def phase2_neural_run_id(
    model_name,
    horizon_minutes,
    variant_name,
):

    return (
        f"{model_name.lower()}"
        f"__{int(horizon_minutes)}min"
        f"__{variant_name}"
    )


def phase2_neural_run_paths(
    model_name,
    horizon_minutes,
    variant_name,
):

    run_id = phase2_neural_run_id(
        model_name,
        horizon_minutes,
        variant_name,
    )


    run_dir = (
        PHASE2_NEURAL_GRID_DIR
        /
        run_id
    )


    return {

        "run_id":
            run_id,

        "run_dir":
            run_dir,

        "result":
            run_dir
            /
            "result.csv",

        "history":
            run_dir
            /
            "history.csv",

        "predictions":
            run_dir
            /
            "predictions.csv",

        "metadata":
            run_dir
            /
            "metadata.json",

        "failure":
            run_dir
            /
            "failure.json",
    }


# ============================================================
# 4. Atomic persistence helpers
#
# Write temporary file first, then replace.
# This prevents an interrupted write from masquerading as
# a completed experiment.
# ============================================================

def atomic_dataframe_to_csv(
    dataframe,
    path,
    index,
):

    path = Path(
        path
    )

    temporary_path = (
        path.parent
        /
        (
            path.name
            +
            ".tmp"
        )
    )


    dataframe.to_csv(
        temporary_path,
        index=index,
    )


    temporary_path.replace(
        path
    )


def atomic_json_dump(
    payload,
    path,
):

    path = Path(
        path
    )

    temporary_path = (
        path.parent
        /
        (
            path.name
            +
            ".tmp"
        )
    )


    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            default=str,
        )


    temporary_path.replace(
        path
    )


# ============================================================
# 5. Validate an already-saved run
# ============================================================

def saved_phase2_neural_run_is_complete(
    model_name,
    horizon_minutes,
    variant_name,
):

    paths = (
        phase2_neural_run_paths(
            model_name,
            horizon_minutes,
            variant_name,
        )
    )


    required_paths = [
        paths[
            "result"
        ],
        paths[
            "history"
        ],
        paths[
            "predictions"
        ],
        paths[
            "metadata"
        ],
    ]


    if not all(
        path.is_file()
        for path
        in required_paths
    ):
        return False


    try:

        result_df = pd.read_csv(
            paths[
                "result"
            ]
        )


        if len(
            result_df
        ) != 1:

            return False


        result_row = (
            result_df.iloc[
                0
            ]
        )


        if (
            str(
                result_row[
                    "execution_protocol"
                ]
            )
            !=
            PHASE2_NEURAL_EXECUTION_PROTOCOL
        ):
            return False


        if (
            str(
                result_row[
                    "model"
                ]
            )
            !=
            str(
                model_name
            )
        ):
            return False


        if (
            int(
                result_row[
                    "horizon_minutes"
                ]
            )
            !=
            int(
                horizon_minutes
            )
        ):
            return False


        if (
            str(
                result_row[
                    "variant"
                ]
            )
            !=
            str(
                variant_name
            )
        ):
            return False


        if (
            int(
                result_row[
                    "n_test"
                ]
            )
            !=
            26_496
        ):
            return False


        prediction_df = pd.read_csv(
            paths[
                "predictions"
            ]
        )


        if len(
            prediction_df
        ) != 26_496:

            return False


        required_prediction_columns = {
            "issue_time",
            "valid_time",
            "daylight",
            "target_P_Solar_kW",
            "prediction_kW",
            "prediction_scaled",
        }


        if not (
            required_prediction_columns
            .issubset(
                prediction_df.columns
            )
        ):
            return False


        if not np.isfinite(
            prediction_df[
                [
                    "target_P_Solar_kW",
                    "prediction_kW",
                    "prediction_scaled",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        ).all():

            return False


        return True


    except Exception:

        return False


# ============================================================
# 6. Build complete frozen manifest BEFORE training
# ============================================================

phase2_neural_grid_manifest_rows = []


for model_name in (
    PHASE2_NEURAL_GRID_MODELS
):

    for horizon_minutes in (
        PHASE2_NEURAL_GRID_HORIZONS
    ):

        frozen_config = (
            PHASE2_FROZEN_MODEL_CONFIGS[
                model_name
            ][
                horizon_minutes
            ]
        )


        for variant_name in (
            PHASE2_NEURAL_GRID_VARIANTS
        ):

            auxiliary_dim = int(
                phase2_neural_data[
                    horizon_minutes
                ][
                    "variants"
                ][
                    variant_name
                ][
                    "auxiliary"
                ]
                .shape[
                    1
                ]
            )


            expected_parameters = (
                expected_phase2_neural_parameter_count(

                    model_name=
                        model_name,

                    auxiliary_dim=
                        auxiliary_dim,
                )
            )


            phase2_neural_grid_manifest_rows.append(
                {
                    "execution_protocol":
                        PHASE2_NEURAL_EXECUTION_PROTOCOL,

                    "model":
                        model_name,

                    "horizon_minutes":
                        horizon_minutes,

                    "variant":
                        variant_name,

                    "auxiliary_dim":
                        auxiliary_dim,

                    "expected_parameters":
                        expected_parameters,

                    "frozen_epochs":
                        int(
                            frozen_config[
                                "epochs"
                            ]
                        ),

                    "seed":
                        PHASE2_NEURAL_SEED,

                    "batch_size":
                        PHASE2_NEURAL_BATCH_SIZE,

                    "shuffle":
                        True,

                    "scaler_policy":
                        "original_train_only_scalers",

                    "refit_population":
                        "train_plus_validation",

                    "validation_data_used":
                        False,

                    "callbacks_used":
                        False,

                    "prediction_clipping":
                        False,
                }
            )


phase2_neural_grid_manifest = (
    pd.DataFrame(
        phase2_neural_grid_manifest_rows
    )
)


assert len(
    phase2_neural_grid_manifest
) == 54


assert (
    phase2_neural_grid_manifest[
        [
            "model",
            "horizon_minutes",
            "variant",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
)


manifest_path = (
    PHASE2_NEURAL_GRID_DIR
    /
    "phase2_neural_core_grid_manifest.csv"
)


atomic_dataframe_to_csv(
    phase2_neural_grid_manifest,
    manifest_path,
    index=False,
)


print(
    "\nFrozen neural grid manifest: PASSED"
)

print(
    "Planned runs:",
    len(
        phase2_neural_grid_manifest
    )
)


# ============================================================
# 7. Determine resume state
# ============================================================

completed_before_start = []


pending_before_start = []


for row in (
    phase2_neural_grid_manifest
    .itertuples(
        index=False
    )
):

    key = (
        row.model,
        int(
            row.horizon_minutes
        ),
        row.variant,
    )


    if saved_phase2_neural_run_is_complete(
        *key
    ):

        completed_before_start.append(
            key
        )

    else:

        pending_before_start.append(
            key
        )


print(
    "\nResume audit"
)

print(
    "------------"
)

print(
    "Already complete:",
    len(
        completed_before_start
    )
)

print(
    "Pending:",
    len(
        pending_before_start
    )
)

print(
    "Total:",
    (
        len(
            completed_before_start
        )
        +
        len(
            pending_before_start
        )
    )
)


assert (
    len(
        completed_before_start
    )
    +
    len(
        pending_before_start
    )
    ==
    54
)


# ============================================================
# 8. Execute pending grid
#
# Each model is reset to seed 42 inside the frozen helper.
# Run order therefore does NOT alter model initialization.
# ============================================================

grid_start_time = (
    time.perf_counter()
)


newly_completed = []


for grid_position, row in enumerate(
    phase2_neural_grid_manifest.itertuples(
        index=False
    ),
    start=1,
):

    model_name = (
        row.model
    )

    horizon_minutes = int(
        row.horizon_minutes
    )

    variant_name = (
        row.variant
    )


    paths = (
        phase2_neural_run_paths(
            model_name,
            horizon_minutes,
            variant_name,
        )
    )


    # --------------------------------------------------------
    # Skip valid completed run
    # --------------------------------------------------------

    if saved_phase2_neural_run_is_complete(
        model_name,
        horizon_minutes,
        variant_name,
    ):

        print(
            f"[{grid_position:02d}/54] "
            f"SKIP  "
            f"{model_name:<4} | "
            f"{horizon_minutes:>2} min | "
            f"{variant_name}"
        )

        continue


    paths[
        "run_dir"
    ].mkdir(
        parents=True,
        exist_ok=True,
    )


    # Delete stale failure record before retrying.
    if paths[
        "failure"
    ].is_file():

        paths[
            "failure"
        ].unlink()


    print(
        "\n"
        +
        "=" * 90
    )

    print(
        f"[{grid_position:02d}/54] "
        f"RUN   "
        f"{model_name} | "
        f"{horizon_minutes} min | "
        f"{variant_name}"
    )

    print(
        "=" * 90
    )


    run_start_time = (
        time.perf_counter()
    )


    try:

        # ====================================================
        # ACTUAL FINAL REFIT + TEST
        # ====================================================

        (
            result,
            history_df,
            prediction_df,
        ) = run_phase2_neural_final_refit(

            model_name=
                model_name,

            horizon_minutes=
                horizon_minutes,

            variant_name=
                variant_name,

            verbose=0,
        )


        # ====================================================
        # Run-specific validation before saving
        # ====================================================

        assert (
            result[
                "execution_protocol"
            ]
            ==
            PHASE2_NEURAL_EXECUTION_PROTOCOL
        )


        assert (
            result[
                "model"
            ]
            ==
            model_name
        )


        assert (
            int(
                result[
                    "horizon_minutes"
                ]
            )
            ==
            horizon_minutes
        )


        assert (
            result[
                "variant"
            ]
            ==
            variant_name
        )


        assert (
            int(
                result[
                    "auxiliary_dim"
                ]
            )
            ==
            int(
                row.auxiliary_dim
            )
        )


        assert (
            int(
                result[
                    "trainable_parameter_count"
                ]
            )
            ==
            int(
                row.expected_parameters
            )
        )


        assert (
            int(
                result[
                    "frozen_epochs"
                ]
            )
            ==
            int(
                row.frozen_epochs
            )
        )


        assert len(
            prediction_df
        ) == 26_496


        assert np.isfinite(
            [
                result[
                    "MAE_kW"
                ],
                result[
                    "RMSE_kW"
                ],
                result[
                    "R2"
                ],
                result[
                    "daylight_MAE_kW"
                ],
                result[
                    "daylight_RMSE_kW"
                ],
                result[
                    "daylight_R2"
                ],
            ]
        ).all()


        # ====================================================
        # Persist result only after all validation passes
        # ====================================================

        result_df = pd.DataFrame(
            [
                result
            ]
        )


        atomic_dataframe_to_csv(
            result_df,
            paths[
                "result"
            ],
            index=False,
        )


        atomic_dataframe_to_csv(
            history_df,
            paths[
                "history"
            ],
            index=False,
        )


        atomic_dataframe_to_csv(
            prediction_df,
            paths[
                "predictions"
            ],
            index=True,
        )


        metadata_record = {

            "run_id":
                paths[
                    "run_id"
                ],

            "execution_protocol":
                PHASE2_NEURAL_EXECUTION_PROTOCOL,

            "model":
                model_name,

            "horizon_minutes":
                horizon_minutes,

            "variant":
                variant_name,

            "architecture":
                PHASE2_FROZEN_MODEL_CONFIGS[
                    model_name
                ][
                    horizon_minutes
                ][
                    "architecture"
                ],

            "frozen_epochs":
                int(
                    row.frozen_epochs
                ),

            "seed":
                int(
                    PHASE2_NEURAL_SEED
                ),

            "batch_size":
                int(
                    PHASE2_NEURAL_BATCH_SIZE
                ),

            "tensorflow_version":
                str(
                    tf.__version__
                ),

            "completed_successfully":
                True,
        }


        atomic_json_dump(
            metadata_record,
            paths[
                "metadata"
            ],
        )


        # ====================================================
        # Verify persisted artifacts before marking complete
        # ====================================================

        assert saved_phase2_neural_run_is_complete(

            model_name,

            horizon_minutes,

            variant_name,
        )


        newly_completed.append(
            (
                model_name,
                horizon_minutes,
                variant_name,
            )
        )


        run_seconds = (
            time.perf_counter()
            -
            run_start_time
        )


        print(
            (
                f"PASS | "
                f"MAE={result['MAE_kW']:.6f} kW | "
                f"Daylight MAE="
                f"{result['daylight_MAE_kW']:.6f} kW | "
                f"epochs="
                f"{result['frozen_epochs']} | "
                f"time="
                f"{run_seconds:.1f}s"
            )
        )


    except Exception as exc:

        # ----------------------------------------------------
        # Save failure metadata, then stop immediately.
        #
        # Re-running the cell will retry this experiment.
        # ----------------------------------------------------

        failure_record = {

            "run_id":
                paths[
                    "run_id"
                ],

            "model":
                model_name,

            "horizon_minutes":
                horizon_minutes,

            "variant":
                variant_name,

            "exception_type":
                type(
                    exc
                ).__name__,

            "exception_message":
                str(
                    exc
                ),

            "completed_successfully":
                False,
        }


        atomic_json_dump(
            failure_record,
            paths[
                "failure"
            ],
        )


        print(
            "\nFAILED RUN:"
        )

        print(
            failure_record
        )


        raise


# ============================================================
# 9. Recover every completed result from disk
# ============================================================

completed_result_frames = []


final_completed_keys = []


final_missing_keys = []


for row in (
    phase2_neural_grid_manifest
    .itertuples(
        index=False
    )
):

    key = (
        row.model,
        int(
            row.horizon_minutes
        ),
        row.variant,
    )


    paths = (
        phase2_neural_run_paths(
            *key
        )
    )


    if saved_phase2_neural_run_is_complete(
        *key
    ):

        completed_result_frames.append(
            pd.read_csv(
                paths[
                    "result"
                ]
            )
        )

        final_completed_keys.append(
            key
        )

    else:

        final_missing_keys.append(
            key
        )


if completed_result_frames:

    phase2_neural_core_results = pd.concat(
        completed_result_frames,
        ignore_index=True,
    )

else:

    phase2_neural_core_results = pd.DataFrame()


# ============================================================
# 10. Save partial/final aggregate after this execution
# ============================================================

partial_summary_path = (
    PHASE2_NEURAL_GRID_DIR
    /
    "phase2_neural_core_results_partial.csv"
)


atomic_dataframe_to_csv(
    phase2_neural_core_results,
    partial_summary_path,
    index=False,
)


grid_total_seconds = (
    time.perf_counter()
    -
    grid_start_time
)


print(
    "\n"
    +
    "=" * 90
)

print(
    "PHASE 2A NEURAL CORE GRID STATUS"
)

print(
    "=" * 90
)


print(
    "Completed:",
    len(
        final_completed_keys
    ),
    "/ 54",
)


print(
    "Remaining:",
    len(
        final_missing_keys
    ),
)


print(
    "Newly completed this execution:",
    len(
        newly_completed
    ),
)


print(
    "Execution time:",
    f"{grid_total_seconds:.1f} seconds",
)


# ============================================================
# 11. If complete, freeze final aggregate
# ============================================================

if (
    len(
        final_missing_keys
    )
    ==
    0
):

    assert len(
        phase2_neural_core_results
    ) == 54


    assert (
        phase2_neural_core_results[
            [
                "model",
                "horizon_minutes",
                "variant",
            ]
        ]
        .duplicated()
        .sum()
        ==
        0
    )


    final_summary_path = (
        PHASE2_NEURAL_GRID_DIR
        /
        "phase2_neural_core_results.csv"
    )


    atomic_dataframe_to_csv(
        phase2_neural_core_results,
        final_summary_path,
        index=False,
    )


    # --------------------------------------------------------
    # All-row MAE pivot
    # --------------------------------------------------------

    neural_all_mae_pivot = (
        phase2_neural_core_results
        .pivot_table(
            index=[
                "model",
                "horizon_minutes",
            ],
            columns="variant",
            values="MAE_kW",
            aggfunc="first",
        )
        .reset_index()
    )


    # --------------------------------------------------------
    # Daylight MAE pivot
    # --------------------------------------------------------

    neural_daylight_mae_pivot = (
        phase2_neural_core_results
        .pivot_table(
            index=[
                "model",
                "horizon_minutes",
            ],
            columns="variant",
            values="daylight_MAE_kW",
            aggfunc="first",
        )
        .reset_index()
    )


    all_mae_path = (
        PHASE2_NEURAL_GRID_DIR
        /
        "phase2_neural_core_all_mae_pivot.csv"
    )


    daylight_mae_path = (
        PHASE2_NEURAL_GRID_DIR
        /
        "phase2_neural_core_daylight_mae_pivot.csv"
    )


    atomic_dataframe_to_csv(
        neural_all_mae_pivot,
        all_mae_path,
        index=False,
    )


    atomic_dataframe_to_csv(
        neural_daylight_mae_pivot,
        daylight_mae_path,
        index=False,
    )


    print(
        "\nPHASE 2A NEURAL CORE GRID: PASSED"
    )


    print(
        "\nAll-row MAE:"
    )

    display(
        neural_all_mae_pivot
    )


    print(
        "\nDaylight MAE:"
    )

    display(
        neural_daylight_mae_pivot
    )


    print(
        "\nSaved final aggregate:"
    )

    print(
        " ",
        final_summary_path
    )


else:

    print(
        "\nGRID NOT YET COMPLETE."
    )

    print(
        "Re-run this same cell to resume."
    )


    print(
        "\nRemaining experiments:"
    )


    for key in (
        final_missing_keys[
            :20
        ]
    ):

        print(
            " ",
            key
        )


    if len(
        final_missing_keys
    ) > 20:

        print(
            "  ..."
        )

Neural grid directory:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/neural_core_grid_v1

Frozen neural grid manifest: PASSED
Planned runs: 54

Resume audit
------------
Already complete: 0
Pending: 54
Total: 54

[01/54] RUN   LSTM | 15 min | F3


2026-09-29 21:08:54.985812: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-29 21:08:54.986313: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

PASS | MAE=0.325792 kW | Daylight MAE=0.451053 kW | epochs=7 | time=45.8s

[02/54] RUN   LSTM | 15 min | F4_real


2026-09-29 21:10:28.206224: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-29 21:10:28.206616: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

PASS | MAE=0.327740 kW | Daylight MAE=0.439672 kW | epochs=7 | time=49.4s

[03/54] RUN   LSTM | 15 min | F4_reference


...

PASS | MAE=0.546463 kW | Daylight MAE=0.673122 kW | epochs=1 | time=25.3s

PHASE 2A NEURAL CORE GRID STATUS
Completed: 54 / 54
Remaining: 0
Newly completed this execution: 54
Execution time: 1932.9 seconds

PHASE 2A NEURAL CORE GRID: PASSED

All-row MAE:


variant,model,horizon_minutes,F3,F4_perm_1301,F4_perm_1302,F4_perm_1303,F4_real,F4_reference,F4_shift_17d,F4_shift_37d,F4_shift_61d
0,LSTM,15,0.325792,0.316964,0.293895,0.298361,0.327740,0.293925,0.299665,0.313815,0.338067
1,LSTM,30,0.385862,0.371731,0.373672,0.378820,0.370956,0.366807,0.367525,0.369617,0.373111
2,LSTM,60,0.468807,0.476224,0.473874,0.513100,0.469986,0.508542,0.474369,0.489899,0.500634
3,TCN,15,0.438522,0.409863,0.338749,0.470011,0.407552,0.304935,0.358663,0.541416,0.416502
4,TCN,30,0.542479,0.547307,0.451644,0.450810,0.601631,0.763561,0.422762,0.516632,0.509037
5,TCN,60,0.687491,0.531814,0.498446,0.546463,0.602103,0.824788,0.628169,0.591566,0.527436



Daylight MAE:


variant,model,horizon_minutes,F3,F4_perm_1301,F4_perm_1302,F4_perm_1303,F4_real,F4_reference,F4_shift_17d,F4_shift_37d,F4_shift_61d
0,LSTM,15,0.451053,0.427694,0.413646,0.418061,0.439672,0.411820,0.413622,0.434100,0.449853
1,LSTM,30,0.525864,0.523978,0.513206,0.524790,0.532072,0.516502,0.520468,0.519832,0.513930
2,LSTM,60,0.651638,0.680014,0.673260,0.702476,0.649063,0.685946,0.642756,0.669855,0.679042
3,TCN,15,0.539464,0.541513,0.459790,0.538664,0.546965,0.422388,0.492698,0.687898,0.490808
4,TCN,30,0.678117,0.742069,0.576667,0.562769,0.679732,0.731767,0.549663,0.650006,0.594563
5,TCN,60,0.761989,0.688063,0.647557,0.673122,0.728352,1.037667,0.750006,0.713823,0.693560



Saved final aggregate:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/neural_core_grid_v1/phase2_neural_core_results.csv


In [131]:
# ============================================================
# Step P2-24: Neural paired UTC-calendar-day bootstrap
#
# Models:
#   LSTM
#   TCN
#
# Horizons:
#   15, 30, 60 minutes
#
# Scopes:
#   all
#   daylight
#
# Comparator set:
#   F3
#   F4_reference
#   F4_shift_17d
#   F4_shift_37d
#   F4_shift_61d
#   F4_perm_1301
#   F4_perm_1302
#   F4_perm_1303
#
# Statistic:
#
#   delta_MAE =
#       MAE_comparator
#       -
#       MAE_F4_real
#
# Positive:
#   aligned F4_real has LOWER MAE.
#
# Negative:
#   comparator has LOWER MAE.
#
# Bootstrap:
#   paired UTC calendar-day blocks
#   92 TEST days
#   10,000 replicates
#   seed = 42
#
# NO MODEL TRAINING occurs in this cell.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Frozen bootstrap protocol
# ============================================================

PHASE2_BOOTSTRAP_REPLICATES = 10_000

PHASE2_BOOTSTRAP_SEED = 42

PHASE2_BOOTSTRAP_ALPHA = 0.05


PHASE2_NEURAL_BOOTSTRAP_MODELS = [
    "LSTM",
    "TCN",
]


PHASE2_NEURAL_BOOTSTRAP_HORIZONS = [
    15,
    30,
    60,
]


PHASE2_NEURAL_BOOTSTRAP_SCOPES = [
    "all",
    "daylight",
]


PHASE2_NEURAL_BOOTSTRAP_COMPARATORS = [
    "F3",
    "F4_reference",
    "F4_shift_17d",
    "F4_shift_37d",
    "F4_shift_61d",
    "F4_perm_1301",
    "F4_perm_1302",
    "F4_perm_1303",
]


assert len(
    PHASE2_NEURAL_BOOTSTRAP_COMPARATORS
) == 8


# 2 models × 3 horizons × 2 scopes × 8 comparators

EXPECTED_BOOTSTRAP_COMPARISONS = (
    2
    *
    3
    *
    2
    *
    8
)

assert (
    EXPECTED_BOOTSTRAP_COMPARISONS
    ==
    96
)


# ============================================================
# 2. Comparator classification
# ============================================================

def phase2_comparator_type(
    variant_name,
):

    if variant_name == "F3":

        return "baseline_F3"

    if variant_name == "F4_reference":

        return "coding_reference"

    if variant_name.startswith(
        "F4_shift_"
    ):

        return "circular_shift_null"

    if variant_name.startswith(
        "F4_perm_"
    ):

        return "joint_row_permutation_null"

    raise ValueError(
        variant_name
    )


# ============================================================
# 3. Load one saved prediction table
# ============================================================

def load_phase2_neural_prediction_table(
    model_name,
    horizon_minutes,
    variant_name,
):

    paths = phase2_neural_run_paths(
        model_name,
        horizon_minutes,
        variant_name,
    )


    assert saved_phase2_neural_run_is_complete(

        model_name,
        horizon_minutes,
        variant_name,

    ), (
        "Incomplete saved run: "
        f"{model_name}, "
        f"{horizon_minutes}, "
        f"{variant_name}"
    )


    prediction_table = pd.read_csv(
        paths[
            "predictions"
        ]
    )


    required_columns = {
        "issue_time",
        "valid_time",
        "daylight",
        "target_P_Solar_kW",
        "prediction_kW",
        "prediction_scaled",
    }


    assert (
        required_columns
        .issubset(
            prediction_table.columns
        )
    )


    assert len(
        prediction_table
    ) == 26_496


    # --------------------------------------------------------
    # Parse timestamps explicitly as UTC
    # --------------------------------------------------------

    prediction_table[
        "issue_time"
    ] = pd.to_datetime(
        prediction_table[
            "issue_time"
        ],
        utc=True,
    )


    prediction_table[
        "valid_time"
    ] = pd.to_datetime(
        prediction_table[
            "valid_time"
        ],
        utc=True,
    )


    # --------------------------------------------------------
    # Robust daylight conversion
    # --------------------------------------------------------

    if (
        prediction_table[
            "daylight"
        ].dtype
        != bool
    ):

        prediction_table[
            "daylight"
        ] = (
            prediction_table[
                "daylight"
            ]
            .astype(
                str
            )
            .str.lower()
            .map(
                {
                    "true": True,
                    "false": False,
                    "1": True,
                    "0": False,
                }
            )
        )


    assert (
        prediction_table[
            "daylight"
        ]
        .notna()
        .all()
    )


    prediction_table[
        "daylight"
    ] = (
        prediction_table[
            "daylight"
        ]
        .astype(
            bool
        )
    )


    assert np.isfinite(
        prediction_table[
            [
                "target_P_Solar_kW",
                "prediction_kW",
            ]
        ]
        .to_numpy(
            dtype=float
        )
    ).all()


    return prediction_table


# ============================================================
# 4. Recover ALL neural prediction tables
# ============================================================

phase2_neural_prediction_tables = {}


for model_name in (
    PHASE2_NEURAL_BOOTSTRAP_MODELS
):

    for horizon_minutes in (
        PHASE2_NEURAL_BOOTSTRAP_HORIZONS
    ):

        phase2_neural_prediction_tables[
            (
                model_name,
                horizon_minutes,
            )
        ] = {}


        for variant_name in (
            [
                "F4_real"
            ]
            +
            PHASE2_NEURAL_BOOTSTRAP_COMPARATORS
        ):

            phase2_neural_prediction_tables[
                (
                    model_name,
                    horizon_minutes,
                )
            ][
                variant_name
            ] = (
                load_phase2_neural_prediction_table(

                    model_name,
                    horizon_minutes,
                    variant_name,
                )
            )


print(
    "Neural prediction recovery: PASSED"
)


# ============================================================
# 5. Cross-variant alignment audit
#
# Every representation for a model+horizon must have:
#
#   identical issue times
#   identical valid times
#   identical targets
#   identical daylight mask
# ============================================================

for model_name in (
    PHASE2_NEURAL_BOOTSTRAP_MODELS
):

    for horizon_minutes in (
        PHASE2_NEURAL_BOOTSTRAP_HORIZONS
    ):

        tables = (
            phase2_neural_prediction_tables[
                (
                    model_name,
                    horizon_minutes,
                )
            ]
        )


        reference = tables[
            "F4_real"
        ]


        for variant_name, table in (
            tables.items()
        ):

            assert np.array_equal(

                table[
                    "issue_time"
                ].to_numpy(),

                reference[
                    "issue_time"
                ].to_numpy(),
            )


            assert np.array_equal(

                table[
                    "valid_time"
                ].to_numpy(),

                reference[
                    "valid_time"
                ].to_numpy(),
            )


            assert np.allclose(

                table[
                    "target_P_Solar_kW"
                ].to_numpy(
                    dtype=float
                ),

                reference[
                    "target_P_Solar_kW"
                ].to_numpy(
                    dtype=float
                ),

                atol=0.0,
                rtol=0.0,
            )


            assert np.array_equal(

                table[
                    "daylight"
                ].to_numpy(
                    dtype=bool
                ),

                reference[
                    "daylight"
                ].to_numpy(
                    dtype=bool
                ),
            )


print(
    "Cross-variant TEST alignment: PASSED"
)


# ============================================================
# 6. Canonical TEST-day audit
# ============================================================

reference_test_table = (
    phase2_neural_prediction_tables[
        (
            "LSTM",
            15,
        )
    ][
        "F4_real"
    ]
)


canonical_test_days = pd.DatetimeIndex(
    reference_test_table[
        "valid_time"
    ]
    .dt
    .floor(
        "D"
    )
    .unique()
).sort_values()


assert (
    len(
        canonical_test_days
    )
    ==
    92
)


assert (
    canonical_test_days[
        0
    ]
    ==
    pd.Timestamp(
        "2019-06-01 00:00:00+00:00"
    )
)


assert (
    canonical_test_days[
        -1
    ]
    ==
    pd.Timestamp(
        "2019-08-31 00:00:00+00:00"
    )
)


print(
    "UTC calendar-day audit: PASSED"
)

print(
    "TEST days:",
    len(
        canonical_test_days
    )
)


# ============================================================
# 7. ONE canonical bootstrap draw matrix
#
# Frozen seed = 42.
#
# Because all conditions use the same 92 TEST calendar days,
# the SAME resampled-day indices are reused everywhere.
# This strengthens paired comparability.
# ============================================================

bootstrap_rng = (
    np.random.default_rng(
        PHASE2_BOOTSTRAP_SEED
    )
)


canonical_bootstrap_draws = (
    bootstrap_rng.integers(

        low=0,

        high=92,

        size=(
            PHASE2_BOOTSTRAP_REPLICATES,
            92,
        ),

        dtype=np.int16,
    )
)


assert canonical_bootstrap_draws.shape == (
    10_000,
    92,
)


print(
    "Canonical bootstrap draws: PASSED"
)


# ============================================================
# 8. Daily aggregate helper
# ============================================================

def build_daily_absolute_error_aggregate(
    prediction_table,
    scope,
):

    assert scope in {
        "all",
        "daylight",
    }


    working = (
        prediction_table[
            [
                "valid_time",
                "daylight",
                "target_P_Solar_kW",
                "prediction_kW",
            ]
        ]
        .copy()
    )


    if scope == "daylight":

        working = (
            working.loc[
                working[
                    "daylight"
                ]
            ]
            .copy()
        )


    working[
        "utc_day"
    ] = (
        working[
            "valid_time"
        ]
        .dt
        .floor(
            "D"
        )
    )


    working[
        "absolute_error"
    ] = np.abs(

        working[
            "target_P_Solar_kW"
        ].to_numpy(
            dtype=float
        )

        -

        working[
            "prediction_kW"
        ].to_numpy(
            dtype=float
        )
    )


    daily = (
        working
        .groupby(
            "utc_day",
            sort=True,
        )
        .agg(
            error_sum=(
                "absolute_error",
                "sum",
            ),
            row_count=(
                "absolute_error",
                "size",
            ),
        )
        .reindex(
            canonical_test_days
        )
    )


    # All 92 days must contribute observations.
    assert (
        daily[
            "row_count"
        ]
        .notna()
        .all()
    )


    assert (
        daily[
            "row_count"
        ]
        >
        0
    ).all()


    assert (
        daily[
            "error_sum"
        ]
        .notna()
        .all()
    )


    return daily


# ============================================================
# 9. Paired day-block bootstrap helper
# ============================================================

def paired_day_bootstrap_delta_mae(
    real_prediction_table,
    comparator_prediction_table,
    scope,
):

    real_daily = (
        build_daily_absolute_error_aggregate(
            real_prediction_table,
            scope,
        )
    )


    comparator_daily = (
        build_daily_absolute_error_aggregate(
            comparator_prediction_table,
            scope,
        )
    )


    # Exact same daily population.
    assert real_daily.index.equals(
        comparator_daily.index
    )


    assert np.array_equal(

        real_daily[
            "row_count"
        ].to_numpy(),

        comparator_daily[
            "row_count"
        ].to_numpy(),
    )


    counts = (
        real_daily[
            "row_count"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    real_error_sum = (
        real_daily[
            "error_sum"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    comparator_error_sum = (
        comparator_daily[
            "error_sum"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    # --------------------------------------------------------
    # Point estimates
    # --------------------------------------------------------

    total_rows = float(
        counts.sum()
    )


    real_mae = float(
        real_error_sum.sum()
        /
        total_rows
    )


    comparator_mae = float(
        comparator_error_sum.sum()
        /
        total_rows
    )


    point_delta = float(
        comparator_mae
        -
        real_mae
    )


    # --------------------------------------------------------
    # Bootstrap resampling
    # --------------------------------------------------------

    sampled_counts = (
        counts[
            canonical_bootstrap_draws
        ]
        .sum(
            axis=1
        )
    )


    sampled_real_errors = (
        real_error_sum[
            canonical_bootstrap_draws
        ]
        .sum(
            axis=1
        )
    )


    sampled_comparator_errors = (
        comparator_error_sum[
            canonical_bootstrap_draws
        ]
        .sum(
            axis=1
        )
    )


    bootstrap_real_mae = (
        sampled_real_errors
        /
        sampled_counts
    )


    bootstrap_comparator_mae = (
        sampled_comparator_errors
        /
        sampled_counts
    )


    bootstrap_delta = (
        bootstrap_comparator_mae
        -
        bootstrap_real_mae
    )


    assert bootstrap_delta.shape == (
        10_000,
    )


    assert np.isfinite(
        bootstrap_delta
    ).all()


    ci_low = float(
        np.percentile(
            bootstrap_delta,
            2.5,
        )
    )


    ci_high = float(
        np.percentile(
            bootstrap_delta,
            97.5,
        )
    )


    if ci_low > 0.0:

        evidence_class = (
            "F4_real_lower_MAE"
        )

    elif ci_high < 0.0:

        evidence_class = (
            "comparator_lower_MAE"
        )

    else:

        evidence_class = (
            "CI_crosses_zero"
        )


    relative_change_percent = float(
        100.0
        *
        point_delta
        /
        comparator_mae
    )


    return {

        "scope":
            scope,

        "n_rows":
            int(
                total_rows
            ),

        "n_utc_days":
            92,

        "F4_real_MAE_kW":
            real_mae,

        "comparator_MAE_kW":
            comparator_mae,

        "delta_MAE_comparator_minus_F4_real_kW":
            point_delta,

        "relative_F4_real_MAE_reduction_vs_comparator_percent":
            relative_change_percent,

        "ci95_low_kW":
            ci_low,

        "ci95_high_kW":
            ci_high,

        "evidence_class":
            evidence_class,

        "bootstrap_replicates":
            PHASE2_BOOTSTRAP_REPLICATES,

        "bootstrap_seed":
            PHASE2_BOOTSTRAP_SEED,
    }


# ============================================================
# 10. Execute all 96 neural comparisons
# ============================================================

phase2_neural_bootstrap_rows = []


for model_name in (
    PHASE2_NEURAL_BOOTSTRAP_MODELS
):

    for horizon_minutes in (
        PHASE2_NEURAL_BOOTSTRAP_HORIZONS
    ):

        tables = (
            phase2_neural_prediction_tables[
                (
                    model_name,
                    horizon_minutes,
                )
            ]
        )


        real_table = tables[
            "F4_real"
        ]


        for scope in (
            PHASE2_NEURAL_BOOTSTRAP_SCOPES
        ):

            for comparator in (
                PHASE2_NEURAL_BOOTSTRAP_COMPARATORS
            ):

                bootstrap_result = (
                    paired_day_bootstrap_delta_mae(

                        real_prediction_table=
                            real_table,

                        comparator_prediction_table=
                            tables[
                                comparator
                            ],

                        scope=
                            scope,
                    )
                )


                phase2_neural_bootstrap_rows.append(
                    {
                        "model":
                            model_name,

                        "horizon_minutes":
                            horizon_minutes,

                        "comparator":
                            comparator,

                        "comparator_type":
                            phase2_comparator_type(
                                comparator
                            ),

                        **bootstrap_result,
                    }
                )


phase2_neural_bootstrap_summary = (
    pd.DataFrame(
        phase2_neural_bootstrap_rows
    )
)


assert len(
    phase2_neural_bootstrap_summary
) == 96


assert (
    phase2_neural_bootstrap_summary[
        [
            "model",
            "horizon_minutes",
            "scope",
            "comparator",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
)


print(
    "\nPHASE 2A NEURAL PAIRED-DAY BOOTSTRAP: PASSED"
)

print(
    "Comparisons:",
    len(
        phase2_neural_bootstrap_summary
    )
)

print(
    "Bootstrap replicates per comparison:",
    PHASE2_BOOTSTRAP_REPLICATES
)

print(
    "UTC day blocks:",
    92
)

print(
    "Seed:",
    PHASE2_BOOTSTRAP_SEED
)


# ============================================================
# 11. Save complete bootstrap table
# ============================================================

neural_bootstrap_path = (
    PHASE2_NEURAL_GRID_DIR
    /
    "phase2_neural_paired_day_bootstrap_summary.csv"
)


phase2_neural_bootstrap_summary.to_csv(
    neural_bootstrap_path,
    index=False,
)


# ============================================================
# 12. Primary F3 ↔ aligned-F4 comparison
# ============================================================

neural_f3_vs_real = (
    phase2_neural_bootstrap_summary
    .loc[
        phase2_neural_bootstrap_summary[
            "comparator"
        ].eq(
            "F3"
        )
    ]
    .copy()
    .sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "PRIMARY NEURAL CONTRAST: F3 vs aligned F4_real"
)

print(
    "Positive delta = F4_real lower MAE"
)

print(
    "=" * 100
)


display(
    neural_f3_vs_real[
        [
            "model",
            "horizon_minutes",
            "scope",
            "F4_real_MAE_kW",
            "comparator_MAE_kW",
            "delta_MAE_comparator_minus_F4_real_kW",
            "ci95_low_kW",
            "ci95_high_kW",
            "evidence_class",
        ]
    ]
)


# ============================================================
# 13. Coding sensitivity:
#     F4_reference vs F4_real
# ============================================================

neural_reference_vs_real = (
    phase2_neural_bootstrap_summary
    .loc[
        phase2_neural_bootstrap_summary[
            "comparator"
        ].eq(
            "F4_reference"
        )
    ]
    .copy()
    .sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "CODING-SENSITIVITY CONTRAST: F4_reference vs F4_real"
)

print(
    "Positive delta = full F4_real coding lower MAE"
)

print(
    "=" * 100
)


display(
    neural_reference_vs_real[
        [
            "model",
            "horizon_minutes",
            "scope",
            "F4_real_MAE_kW",
            "comparator_MAE_kW",
            "delta_MAE_comparator_minus_F4_real_kW",
            "ci95_low_kW",
            "ci95_high_kW",
            "evidence_class",
        ]
    ]
)


# ============================================================
# 14. Null-control evidence counts
# ============================================================

neural_null_only = (
    phase2_neural_bootstrap_summary
    .loc[
        phase2_neural_bootstrap_summary[
            "comparator_type"
        ].isin(
            [
                "circular_shift_null",
                "joint_row_permutation_null",
            ]
        )
    ]
    .copy()
)


neural_null_evidence_counts = (
    neural_null_only
    .groupby(
        [
            "model",
            "comparator_type",
            "evidence_class",
        ],
        dropna=False,
    )
    .size()
    .rename(
        "comparison_count"
    )
    .reset_index()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "NULL-CONTROL EVIDENCE COUNTS"
)

print(
    "=" * 100
)


display(
    neural_null_evidence_counts
)


# ============================================================
# 15. Save focused summaries
# ============================================================

neural_f3_vs_real.to_csv(

    PHASE2_NEURAL_GRID_DIR
    /
    "phase2_neural_F3_vs_F4_real_bootstrap.csv",

    index=False,
)


neural_reference_vs_real.to_csv(

    PHASE2_NEURAL_GRID_DIR
    /
    "phase2_neural_F4_reference_vs_real_bootstrap.csv",

    index=False,
)


neural_null_evidence_counts.to_csv(

    PHASE2_NEURAL_GRID_DIR
    /
    "phase2_neural_null_evidence_counts.csv",

    index=False,
)


print(
    "\nSaved:"
)

print(
    " ",
    neural_bootstrap_path
)


print(
    "\nSTOP HERE."
)

print(
    "No additional model was trained."
)

Neural prediction recovery: PASSED
Cross-variant TEST alignment: PASSED
UTC calendar-day audit: PASSED
TEST days: 92
Canonical bootstrap draws: PASSED

PHASE 2A NEURAL PAIRED-DAY BOOTSTRAP: PASSED
Comparisons: 96
Bootstrap replicates per comparison: 10000
UTC day blocks: 92
Seed: 42

PRIMARY NEURAL CONTRAST: F3 vs aligned F4_real
Positive delta = F4_real lower MAE


,model,horizon_minutes,scope,F4_real_MAE_kW,comparator_MAE_kW,delta_MAE_comparator_minus_F4_real_kW,ci95_low_kW,ci95_high_kW,evidence_class
0,LSTM,15,all,0.327740,0.325792,-0.001948,-0.009222,0.005539,CI_crosses_zero
1,LSTM,15,daylight,0.439672,0.451053,0.011381,0.001520,0.021544,F4_real_lower_MAE
2,LSTM,30,all,0.370956,0.385862,0.014906,0.009770,0.019980,F4_real_lower_MAE
3,LSTM,30,daylight,0.532072,0.525864,-0.006208,-0.011577,-0.001022,comparator_lower_MAE
4,LSTM,60,all,0.469986,0.468807,-0.001180,-0.009381,0.007295,CI_crosses_zero
5,LSTM,60,daylight,0.649063,0.651638,0.002576,-0.008825,0.014131,CI_crosses_zero
6,TCN,15,all,0.407552,0.438522,0.030970,0.010855,0.050991,F4_real_lower_MAE
7,TCN,15,daylight,0.546965,0.539464,-0.007501,-0.031139,0.015825,CI_crosses_zero
8,TCN,30,all,0.601631,0.542479,-0.059152,-0.089339,-0.029783,comparator_lower_MAE
9,TCN,30,daylight,0.679732,0.678117,-0.001615,-0.030651,0.026416,CI_crosses_zero



CODING-SENSITIVITY CONTRAST: F4_reference vs F4_real
Positive delta = full F4_real coding lower MAE


,model,horizon_minutes,scope,F4_real_MAE_kW,comparator_MAE_kW,delta_MAE_comparator_minus_F4_real_kW,ci95_low_kW,ci95_high_kW,evidence_class
0,LSTM,15,all,0.327740,0.293925,-0.033815,-0.039830,-0.027991,comparator_lower_MAE
1,LSTM,15,daylight,0.439672,0.411820,-0.027852,-0.034679,-0.021068,comparator_lower_MAE
2,LSTM,30,all,0.370956,0.366807,-0.004149,-0.007728,-0.000575,comparator_lower_MAE
3,LSTM,30,daylight,0.532072,0.516502,-0.015570,-0.020194,-0.011117,comparator_lower_MAE
4,LSTM,60,all,0.469986,0.508542,0.038556,0.027196,0.050664,F4_real_lower_MAE
5,LSTM,60,daylight,0.649063,0.685946,0.036883,0.021073,0.054105,F4_real_lower_MAE
6,TCN,15,all,0.407552,0.304935,-0.102617,-0.120046,-0.086224,comparator_lower_MAE
7,TCN,15,daylight,0.546965,0.422388,-0.124578,-0.148747,-0.101964,comparator_lower_MAE
8,TCN,30,all,0.601631,0.763561,0.161930,0.120993,0.203193,F4_real_lower_MAE
9,TCN,30,daylight,0.679732,0.731767,0.052035,0.022660,0.081391,F4_real_lower_MAE



NULL-CONTROL EVIDENCE COUNTS


,model,comparator_type,evidence_class,comparison_count
0,LSTM,circular_shift_null,CI_crosses_zero,5
1,LSTM,circular_shift_null,F4_real_lower_MAE,6
2,LSTM,circular_shift_null,comparator_lower_MAE,7
3,LSTM,joint_row_permutation_null,CI_crosses_zero,4
4,LSTM,joint_row_permutation_null,F4_real_lower_MAE,5
5,LSTM,joint_row_permutation_null,comparator_lower_MAE,9
6,TCN,circular_shift_null,CI_crosses_zero,5
7,TCN,circular_shift_null,F4_real_lower_MAE,2
8,TCN,circular_shift_null,comparator_lower_MAE,11
9,TCN,joint_row_permutation_null,CI_crosses_zero,3



Saved:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/neural_core_grid_v1/phase2_neural_paired_day_bootstrap_summary.csv

STOP HERE.
No additional model was trained.


In [132]:
# ============================================================
# Step P2-25A:
# Add the previously omitted F4_reference coding-sensitivity
# bootstrap for Ridge + LightGBM.
#
# IMPORTANT
# ---------
# - NO model training.
# - Uses the already-saved Stage-1 TEST predictions.
# - Uses the SAME original Stage-1B bootstrap construction:
#
#       SeedSequence([42, horizon, scope_index])
#
# - Does NOT modify the original 84 comparisons.
# - Adds exactly 12 F4_reference comparisons:
#
#       2 models
#       × 3 horizons
#       × 2 scopes
#       = 12
#
# Final classical table:
#
#       84 + 12 = 96 comparisons
#
# This makes the classical and neural core analyses symmetric:
# 8 comparators per model/horizon/scope.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Frozen classical conditions
# ============================================================

CLASSICAL_REFERENCE_MODELS = [
    "Ridge",
    "LightGBM",
]


CLASSICAL_REFERENCE_HORIZONS = [
    15,
    30,
    60,
]


CLASSICAL_REFERENCE_SCOPES = [
    "all",
    "daylight",
]


CLASSICAL_REFERENCE_BOOTSTRAP_REPLICATES = 10_000

CLASSICAL_REFERENCE_BOOTSTRAP_SEED = 42


# ============================================================
# 2. Recover Stage-1 prediction tables if necessary
# ============================================================

if (
    "stage1_prediction_tables"
    not in globals()
):

    stage1_prediction_tables = {}


    for horizon in (
        CLASSICAL_REFERENCE_HORIZONS
    ):

        prediction_path = (
            PHASE2_RESULTS_DIR
            /
            (
                "phase2_stage1_classical_core_"
                f"predictions_{horizon}min.csv"
            )
        )


        assert prediction_path.is_file(), (
            f"Missing Stage-1 prediction file: "
            f"{prediction_path}"
        )


        table = pd.read_csv(
            prediction_path,
            index_col=0,
        )


        table.index = pd.to_datetime(
            table.index,
            utc=True,
        )


        table[
            "issue_time"
        ] = pd.to_datetime(
            table[
                "issue_time"
            ],
            utc=True,
        )


        table[
            "valid_time"
        ] = pd.to_datetime(
            table[
                "valid_time"
            ],
            utc=True,
        )


        # Robust daylight parsing.
        if (
            table[
                "daylight"
            ].dtype
            != bool
        ):

            daylight_as_text = (
                table[
                    "daylight"
                ]
                .astype(
                    str
                )
                .str.lower()
            )


            table[
                "daylight"
            ] = (
                daylight_as_text
                .map(
                    {
                        "true": True,
                        "false": False,
                        "1": True,
                        "0": False,
                    }
                )
            )


        assert (
            table[
                "daylight"
            ]
            .notna()
            .all()
        )


        table[
            "daylight"
        ] = (
            table[
                "daylight"
            ]
            .astype(
                bool
            )
        )


        stage1_prediction_tables[
            horizon
        ] = table


# ============================================================
# 3. Stage-1 prediction-table audit
# ============================================================

for horizon in (
    CLASSICAL_REFERENCE_HORIZONS
):

    table = (
        stage1_prediction_tables[
            horizon
        ]
    )


    assert len(
        table
    ) == 26_496


    assert (
        table[
            "valid_time"
        ]
        .dt
        .floor(
            "D"
        )
        .nunique()
        ==
        92
    )


    for model_name in (
        CLASSICAL_REFERENCE_MODELS
    ):

        real_column = (
            f"{model_name}__F4_real"
        )


        reference_column = (
            f"{model_name}__F4_reference"
        )


        assert (
            real_column
            in table.columns
        )


        assert (
            reference_column
            in table.columns
        )


        assert np.isfinite(
            table[
                real_column
            ]
            .to_numpy(
                dtype=float
            )
        ).all()


        assert np.isfinite(
            table[
                reference_column
            ]
            .to_numpy(
                dtype=float
            )
        ).all()


print(
    "Classical F4_reference prediction recovery: PASSED"
)


# ============================================================
# 4. Independent helper
#
# Statistic:
#
#   delta =
#       MAE_F4_reference
#       -
#       MAE_F4_real
#
# Positive:
#       full F4_real coding lower MAE
#
# Negative:
#       reference coding lower MAE
# ============================================================

def classical_reference_day_bootstrap(
    prediction_table,
    comparator_prediction_column,
    real_prediction_column,
    scope,
    bootstrap_draws,
):

    assert scope in {
        "all",
        "daylight",
    }


    if scope == "all":

        scoped = (
            prediction_table.copy()
        )

    else:

        scoped = (
            prediction_table
            .loc[
                prediction_table[
                    "daylight"
                ]
            ]
            .copy()
        )


    scoped[
        "_utc_day"
    ] = (
        scoped[
            "valid_time"
        ]
        .dt
        .floor(
            "D"
        )
    )


    y_true = (
        scoped[
            "target_P_Solar_kW"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    comparator_prediction = (
        scoped[
            comparator_prediction_column
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    real_prediction = (
        scoped[
            real_prediction_column
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    comparator_absolute_error = np.abs(
        y_true
        -
        comparator_prediction
    )


    real_absolute_error = np.abs(
        y_true
        -
        real_prediction
    )


    daily_table = pd.DataFrame(
        {
            "utc_day":
                scoped[
                    "_utc_day"
                ]
                .to_numpy(),

            "comparator_error":
                comparator_absolute_error,

            "real_error":
                real_absolute_error,
        }
    )


    daily = (
        daily_table
        .groupby(
            "utc_day",
            sort=True,
        )
        .agg(
            comparator_error_sum=(
                "comparator_error",
                "sum",
            ),

            real_error_sum=(
                "real_error",
                "sum",
            ),

            row_count=(
                "comparator_error",
                "size",
            ),
        )
    )


    assert len(
        daily
    ) == 92


    comparator_daily_sum = (
        daily[
            "comparator_error_sum"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    real_daily_sum = (
        daily[
            "real_error_sum"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    daily_count = (
        daily[
            "row_count"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    # --------------------------------------------------------
    # Point estimates
    # --------------------------------------------------------

    comparator_mae = float(
        comparator_absolute_error.mean()
    )


    real_mae = float(
        real_absolute_error.mean()
    )


    delta_mae = float(
        comparator_mae
        -
        real_mae
    )


    improvement_percent = float(
        100.0
        *
        delta_mae
        /
        comparator_mae
    )


    # --------------------------------------------------------
    # Paired UTC-day bootstrap
    # --------------------------------------------------------

    sampled_count = (
        daily_count[
            bootstrap_draws
        ]
        .sum(
            axis=1
        )
    )


    sampled_comparator_mae = (
        comparator_daily_sum[
            bootstrap_draws
        ]
        .sum(
            axis=1
        )
        /
        sampled_count
    )


    sampled_real_mae = (
        real_daily_sum[
            bootstrap_draws
        ]
        .sum(
            axis=1
        )
        /
        sampled_count
    )


    bootstrap_delta = (
        sampled_comparator_mae
        -
        sampled_real_mae
    )


    bootstrap_percent = (
        100.0
        *
        bootstrap_delta
        /
        sampled_comparator_mae
    )


    assert bootstrap_delta.shape == (
        10_000,
    )


    assert np.isfinite(
        bootstrap_delta
    ).all()


    ci_low_kw, ci_high_kw = (
        np.percentile(
            bootstrap_delta,
            [
                2.5,
                97.5,
            ],
        )
    )


    ci_low_percent, ci_high_percent = (
        np.percentile(
            bootstrap_percent,
            [
                2.5,
                97.5,
            ],
        )
    )


    if ci_low_kw > 0.0:

        ci_zero_status = (
            "above_zero"
        )

    elif ci_high_kw < 0.0:

        ci_zero_status = (
            "below_zero"
        )

    else:

        ci_zero_status = (
            "crosses_zero"
        )


    return {

        "n_rows":
            int(
                len(
                    scoped
                )
            ),

        "daily_blocks":
            92,

        "comparator_MAE_kW":
            comparator_mae,

        "F4_real_MAE_kW":
            real_mae,

        "comparator_minus_F4_real_MAE_kW":
            delta_mae,

        "F4_real_improvement_percent":
            improvement_percent,

        "CI95_low_kW":
            float(
                ci_low_kw
            ),

        "CI95_high_kW":
            float(
                ci_high_kw
            ),

        "CI95_low_percent":
            float(
                ci_low_percent
            ),

        "CI95_high_percent":
            float(
                ci_high_percent
            ),

        "CI_zero_status":
            ci_zero_status,
    }


# ============================================================
# 5. Reproduce the ORIGINAL Stage-1 bootstrap draw policy
#
# This deliberately matches P2-17:
#
# SeedSequence(
#     [
#         42,
#         horizon,
#         scope_index,
#     ]
# )
# ============================================================

classical_reference_draws = {}


for horizon in (
    CLASSICAL_REFERENCE_HORIZONS
):

    for scope_index, scope in enumerate(
        CLASSICAL_REFERENCE_SCOPES
    ):

        seed_sequence = (
            np.random.SeedSequence(
                [
                    CLASSICAL_REFERENCE_BOOTSTRAP_SEED,
                    int(
                        horizon
                    ),
                    int(
                        scope_index
                    ),
                ]
            )
        )


        rng = np.random.default_rng(
            seed_sequence
        )


        classical_reference_draws[
            (
                horizon,
                scope,
            )
        ] = rng.integers(

            low=0,

            high=92,

            size=(
                CLASSICAL_REFERENCE_BOOTSTRAP_REPLICATES,
                92,
            ),

            endpoint=False,
        )


# ============================================================
# 6. Execute the 12 missing comparisons
# ============================================================

classical_reference_rows = []


for horizon in (
    CLASSICAL_REFERENCE_HORIZONS
):

    table = (
        stage1_prediction_tables[
            horizon
        ]
    )


    for scope in (
        CLASSICAL_REFERENCE_SCOPES
    ):

        draws = (
            classical_reference_draws[
                (
                    horizon,
                    scope,
                )
            ]
        )


        for model_name in (
            CLASSICAL_REFERENCE_MODELS
        ):

            result = (
                classical_reference_day_bootstrap(

                    prediction_table=
                        table,

                    comparator_prediction_column=
                        (
                            f"{model_name}"
                            "__F4_reference"
                        ),

                    real_prediction_column=
                        (
                            f"{model_name}"
                            "__F4_real"
                        ),

                    scope=
                        scope,

                    bootstrap_draws=
                        draws,
                )
            )


            classical_reference_rows.append(
                {
                    "model":
                        model_name,

                    "horizon_minutes":
                        horizon,

                    "scope":
                        scope,

                    "comparator":
                        "F4_reference",

                    **result,
                }
            )


classical_reference_bootstrap = (
    pd.DataFrame(
        classical_reference_rows
    )
)


assert len(
    classical_reference_bootstrap
) == 12


assert (
    classical_reference_bootstrap[
        [
            "model",
            "horizon_minutes",
            "scope",
            "comparator",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
)


print(
    "\nCLASSICAL F4_REFERENCE BOOTSTRAP: PASSED"
)

print(
    "Comparisons:",
    len(
        classical_reference_bootstrap
    )
)


# ============================================================
# 7. Load original 84-comparison classical bootstrap
# ============================================================

original_classical_bootstrap_path = (
    PHASE2_RESULTS_DIR
    /
    "phase2_stage1_paired_day_bootstrap_summary.csv"
)


assert (
    original_classical_bootstrap_path
    .is_file()
)


original_classical_bootstrap = (
    pd.read_csv(
        original_classical_bootstrap_path
    )
)


assert len(
    original_classical_bootstrap
) == 84


assert (
    "F4_reference"
    not in
    set(
        original_classical_bootstrap[
            "comparator"
        ]
    )
)


# ============================================================
# 8. Create complete 96-comparison classical table
# ============================================================

stage1_bootstrap_summary_with_reference = (
    pd.concat(
        [
            original_classical_bootstrap,
            classical_reference_bootstrap,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
            "comparator",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    stage1_bootstrap_summary_with_reference
) == 96


assert (
    stage1_bootstrap_summary_with_reference[
        [
            "model",
            "horizon_minutes",
            "scope",
            "comparator",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
)


assert set(
    stage1_bootstrap_summary_with_reference[
        "comparator"
    ]
) == {
    "F3",
    "F4_reference",
    "F4_shift_17d",
    "F4_shift_37d",
    "F4_shift_61d",
    "F4_perm_1301",
    "F4_perm_1302",
    "F4_perm_1303",
}


# ============================================================
# 9. Save complete classical bootstrap
# ============================================================

complete_classical_bootstrap_path = (
    PHASE2_RESULTS_DIR
    /
    (
        "phase2_stage1_paired_day_bootstrap_"
        "summary_with_reference.csv"
    )
)


stage1_bootstrap_summary_with_reference.to_csv(
    complete_classical_bootstrap_path,
    index=False,
)


# ============================================================
# 10. Display coding-sensitivity result
# ============================================================

classical_reference_display = (
    classical_reference_bootstrap[
        [
            "model",
            "horizon_minutes",
            "scope",
            "F4_real_MAE_kW",
            "comparator_MAE_kW",
            "comparator_minus_F4_real_MAE_kW",
            "CI95_low_kW",
            "CI95_high_kW",
            "CI_zero_status",
        ]
    ]
    .sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "CLASSICAL CODING-SENSITIVITY CONTRAST:"
)

print(
    "F4_reference vs F4_real"
)

print(
    "Positive delta = full F4_real coding lower MAE"
)

print(
    "=" * 100
)


display(
    classical_reference_display
)


print(
    "\nComplete classical comparison count:",
    len(
        stage1_bootstrap_summary_with_reference
    )
)


print(
    "\nSaved:"
)

print(
    " ",
    complete_classical_bootstrap_path
)


print(
    "\nSTOP HERE."
)

print(
    "No model was trained."
)

Classical F4_reference prediction recovery: PASSED

CLASSICAL F4_REFERENCE BOOTSTRAP: PASSED
Comparisons: 12

CLASSICAL CODING-SENSITIVITY CONTRAST:
F4_reference vs F4_real
Positive delta = full F4_real coding lower MAE


,model,horizon_minutes,scope,F4_real_MAE_kW,comparator_MAE_kW,comparator_minus_F4_real_MAE_kW,CI95_low_kW,CI95_high_kW,CI_zero_status
0,LightGBM,15,all,0.262057,0.262658,6.011631e-04,-2.041450e-04,1.433976e-03,crosses_zero
1,LightGBM,15,daylight,0.386933,0.387786,8.527254e-04,-3.453151e-04,2.054061e-03,crosses_zero
2,LightGBM,30,all,0.300646,0.302067,1.420501e-03,3.850983e-04,2.493285e-03,above_zero
3,LightGBM,30,daylight,0.443675,0.445424,1.748467e-03,2.047872e-04,3.366158e-03,above_zero
4,LightGBM,60,all,0.343123,0.341401,-1.721925e-03,-3.393677e-03,-1.031215e-04,below_zero
5,LightGBM,60,daylight,0.505740,0.504456,-1.283827e-03,-3.764434e-03,1.056924e-03,crosses_zero
6,Ridge,15,all,0.329466,0.329466,3.428228e-09,-6.514217e-09,1.361829e-08,crosses_zero
7,Ridge,15,daylight,0.421066,0.421066,-4.919057e-09,-1.518519e-08,4.949272e-09,crosses_zero
8,Ridge,30,all,0.420881,0.420881,2.964584e-09,-1.176503e-08,1.738539e-08,crosses_zero
9,Ridge,30,daylight,0.509375,0.509375,-1.027437e-08,-2.537148e-08,4.180457e-09,crosses_zero



Complete classical comparison count: 96

Saved:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_stage1_paired_day_bootstrap_summary_with_reference.csv

STOP HERE.
No model was trained.


In [133]:
# ============================================================
# Step P2-25B:
# FOUR-ARCHITECTURE Phase-2A mechanism synthesis
#
# Combines:
#
#   Classical:
#       Ridge
#       LightGBM
#       96 bootstrap comparisons
#
#   Neural:
#       LSTM
#       TCN
#       96 bootstrap comparisons
#
# Total:
#       192 paired-bootstrap comparisons
#
# NO MODEL TRAINING occurs here.
#
# Outputs:
#
#   1. Unified 192-row inference table
#   2. Primary F3 vs F4_real table (24 rows)
#   3. Coding-sensitivity table (24 rows)
#   4. Null-control evidence counts
#   5. Per-condition mechanism summary (24 rows)
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Recover complete classical bootstrap if required
# ============================================================

if (
    "stage1_bootstrap_summary_with_reference"
    not in globals()
):

    classical_path = (
        PHASE2_RESULTS_DIR
        /
        (
            "phase2_stage1_paired_day_bootstrap_"
            "summary_with_reference.csv"
        )
    )


    assert classical_path.is_file(), (
        f"Missing classical bootstrap file: "
        f"{classical_path}"
    )


    stage1_bootstrap_summary_with_reference = (
        pd.read_csv(
            classical_path
        )
    )


assert len(
    stage1_bootstrap_summary_with_reference
) == 96


# ============================================================
# 2. Recover neural bootstrap if required
# ============================================================

if (
    "phase2_neural_bootstrap_summary"
    not in globals()
):

    neural_path = (
        PHASE2_NEURAL_GRID_DIR
        /
        "phase2_neural_paired_day_bootstrap_summary.csv"
    )


    assert neural_path.is_file(), (
        f"Missing neural bootstrap file: "
        f"{neural_path}"
    )


    phase2_neural_bootstrap_summary = (
        pd.read_csv(
            neural_path
        )
    )


assert len(
    phase2_neural_bootstrap_summary
) == 96


# ============================================================
# 3. Common comparator classification
# ============================================================

def common_comparator_type(
    comparator,
):

    comparator = str(
        comparator
    )


    if comparator == "F3":

        return "baseline_F3"


    if comparator == "F4_reference":

        return "coding_reference"


    if comparator.startswith(
        "F4_shift_"
    ):

        return "circular_shift_null"


    if comparator.startswith(
        "F4_perm_"
    ):

        return "joint_row_permutation_null"


    raise ValueError(
        f"Unknown comparator: {comparator}"
    )


# ============================================================
# 4. Normalize classical table into common schema
# ============================================================

classical_common = pd.DataFrame(
    {
        "model":
            stage1_bootstrap_summary_with_reference[
                "model"
            ]
            .astype(
                str
            ),

        "model_family":
            "classical",

        "horizon_minutes":
            stage1_bootstrap_summary_with_reference[
                "horizon_minutes"
            ]
            .astype(
                int
            ),

        "scope":
            stage1_bootstrap_summary_with_reference[
                "scope"
            ]
            .astype(
                str
            ),

        "comparator":
            stage1_bootstrap_summary_with_reference[
                "comparator"
            ]
            .astype(
                str
            ),

        "n_rows":
            stage1_bootstrap_summary_with_reference[
                "n_rows"
            ]
            .astype(
                int
            ),

        "n_utc_days":
            stage1_bootstrap_summary_with_reference[
                "daily_blocks"
            ]
            .astype(
                int
            ),

        "F4_real_MAE_kW":
            stage1_bootstrap_summary_with_reference[
                "F4_real_MAE_kW"
            ]
            .astype(
                float
            ),

        "comparator_MAE_kW":
            stage1_bootstrap_summary_with_reference[
                "comparator_MAE_kW"
            ]
            .astype(
                float
            ),

        "delta_MAE_comparator_minus_F4_real_kW":
            stage1_bootstrap_summary_with_reference[
                "comparator_minus_F4_real_MAE_kW"
            ]
            .astype(
                float
            ),

        "relative_F4_real_MAE_reduction_vs_comparator_percent":
            stage1_bootstrap_summary_with_reference[
                "F4_real_improvement_percent"
            ]
            .astype(
                float
            ),

        "ci95_low_kW":
            stage1_bootstrap_summary_with_reference[
                "CI95_low_kW"
            ]
            .astype(
                float
            ),

        "ci95_high_kW":
            stage1_bootstrap_summary_with_reference[
                "CI95_high_kW"
            ]
            .astype(
                float
            ),
    }
)


classical_status_map = {
    "above_zero":
        "F4_real_lower_MAE",

    "below_zero":
        "comparator_lower_MAE",

    "crosses_zero":
        "CI_crosses_zero",
}


classical_common[
    "evidence_class"
] = (
    stage1_bootstrap_summary_with_reference[
        "CI_zero_status"
    ]
    .map(
        classical_status_map
    )
)


assert (
    classical_common[
        "evidence_class"
    ]
    .notna()
    .all()
)


classical_common[
    "comparator_type"
] = (
    classical_common[
        "comparator"
    ]
    .map(
        common_comparator_type
    )
)


classical_common[
    "bootstrap_replicates"
] = 10_000


classical_common[
    "bootstrap_base_seed"
] = 42


classical_common[
    "bootstrap_draw_policy"
] = (
    "SeedSequence([42,horizon,scope_index])"
)


# ============================================================
# 5. Normalize neural table into common schema
# ============================================================

neural_common = pd.DataFrame(
    {
        "model":
            phase2_neural_bootstrap_summary[
                "model"
            ]
            .astype(
                str
            ),

        "model_family":
            "neural",

        "horizon_minutes":
            phase2_neural_bootstrap_summary[
                "horizon_minutes"
            ]
            .astype(
                int
            ),

        "scope":
            phase2_neural_bootstrap_summary[
                "scope"
            ]
            .astype(
                str
            ),

        "comparator":
            phase2_neural_bootstrap_summary[
                "comparator"
            ]
            .astype(
                str
            ),

        "n_rows":
            phase2_neural_bootstrap_summary[
                "n_rows"
            ]
            .astype(
                int
            ),

        "n_utc_days":
            phase2_neural_bootstrap_summary[
                "n_utc_days"
            ]
            .astype(
                int
            ),

        "F4_real_MAE_kW":
            phase2_neural_bootstrap_summary[
                "F4_real_MAE_kW"
            ]
            .astype(
                float
            ),

        "comparator_MAE_kW":
            phase2_neural_bootstrap_summary[
                "comparator_MAE_kW"
            ]
            .astype(
                float
            ),

        "delta_MAE_comparator_minus_F4_real_kW":
            phase2_neural_bootstrap_summary[
                "delta_MAE_comparator_minus_F4_real_kW"
            ]
            .astype(
                float
            ),

        "relative_F4_real_MAE_reduction_vs_comparator_percent":
            phase2_neural_bootstrap_summary[
                "relative_F4_real_MAE_reduction_vs_comparator_percent"
            ]
            .astype(
                float
            ),

        "ci95_low_kW":
            phase2_neural_bootstrap_summary[
                "ci95_low_kW"
            ]
            .astype(
                float
            ),

        "ci95_high_kW":
            phase2_neural_bootstrap_summary[
                "ci95_high_kW"
            ]
            .astype(
                float
            ),

        "evidence_class":
            phase2_neural_bootstrap_summary[
                "evidence_class"
            ]
            .astype(
                str
            ),
    }
)


neural_common[
    "comparator_type"
] = (
    neural_common[
        "comparator"
    ]
    .map(
        common_comparator_type
    )
)


neural_common[
    "bootstrap_replicates"
] = 10_000


neural_common[
    "bootstrap_base_seed"
] = 42


neural_common[
    "bootstrap_draw_policy"
] = (
    "single canonical 92-day draw matrix"
)


# ============================================================
# 6. Unified four-model bootstrap table
# ============================================================

phase2_core_bootstrap_all_models = (
    pd.concat(
        [
            classical_common,
            neural_common,
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
            "comparator_type",
            "comparator",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    phase2_core_bootstrap_all_models
) == 192


assert (
    phase2_core_bootstrap_all_models[
        [
            "model",
            "horizon_minutes",
            "scope",
            "comparator",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
)


assert set(
    phase2_core_bootstrap_all_models[
        "model"
    ]
) == {
    "Ridge",
    "LightGBM",
    "LSTM",
    "TCN",
}


assert set(
    phase2_core_bootstrap_all_models[
        "comparator"
    ]
) == {
    "F3",
    "F4_reference",
    "F4_shift_17d",
    "F4_shift_37d",
    "F4_shift_61d",
    "F4_perm_1301",
    "F4_perm_1302",
    "F4_perm_1303",
}


assert (
    phase2_core_bootstrap_all_models[
        "n_utc_days"
    ]
    .eq(
        92
    )
    .all()
)


model_comparison_counts = (
    phase2_core_bootstrap_all_models
    .groupby(
        "model"
    )
    .size()
)


assert (
    model_comparison_counts
    .eq(
        48
    )
    .all()
)


print(
    "FOUR-MODEL BOOTSTRAP MERGE: PASSED"
)

print(
    "Total comparisons:",
    len(
        phase2_core_bootstrap_all_models
    )
)


# ============================================================
# 7. Save full 192-row table
# ============================================================

combined_bootstrap_path = (
    PHASE2_RESULTS_DIR
    /
    "phase2_core_four_model_bootstrap_192.csv"
)


phase2_core_bootstrap_all_models.to_csv(
    combined_bootstrap_path,
    index=False,
)


# ============================================================
# 8. PRIMARY contrast:
#    F3 vs correctly aligned F4_real
#
# 4 models × 3 horizons × 2 scopes = 24 rows
# ============================================================

phase2_primary_f3_vs_f4 = (
    phase2_core_bootstrap_all_models
    .loc[
        phase2_core_bootstrap_all_models[
            "comparator"
        ].eq(
            "F3"
        )
    ]
    .copy()
    .sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    phase2_primary_f3_vs_f4
) == 24


print(
    "\n"
    +
    "=" * 110
)

print(
    "FOUR-ARCHITECTURE PRIMARY CONTRAST:"
)

print(
    "F3 vs correctly aligned F4_real"
)

print(
    "Positive delta = F4_real lower MAE"
)

print(
    "=" * 110
)


display(
    phase2_primary_f3_vs_f4[
        [
            "model",
            "horizon_minutes",
            "scope",
            "F4_real_MAE_kW",
            "comparator_MAE_kW",
            "delta_MAE_comparator_minus_F4_real_kW",
            "ci95_low_kW",
            "ci95_high_kW",
            "evidence_class",
        ]
    ]
)


# ============================================================
# 9. Primary evidence counts by architecture
# ============================================================

phase2_primary_evidence_counts = (
    phase2_primary_f3_vs_f4
    .groupby(
        [
            "model",
            "evidence_class",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=[
            "F4_real_lower_MAE",
            "comparator_lower_MAE",
            "CI_crosses_zero",
        ],
        fill_value=0,
    )
    .reset_index()
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "PRIMARY F3 ↔ F4 EVIDENCE COUNTS BY ARCHITECTURE"
)

print(
    "=" * 110
)


display(
    phase2_primary_evidence_counts
)


# ============================================================
# 10. CODING-SENSITIVITY contrast:
#     F4_reference vs F4_real
#
# 24 rows
# ============================================================

phase2_coding_sensitivity = (
    phase2_core_bootstrap_all_models
    .loc[
        phase2_core_bootstrap_all_models[
            "comparator"
        ].eq(
            "F4_reference"
        )
    ]
    .copy()
    .sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    phase2_coding_sensitivity
) == 24


print(
    "\n"
    +
    "=" * 110
)

print(
    "FOUR-ARCHITECTURE CODING-SENSITIVITY CONTRAST:"
)

print(
    "F4_reference vs F4_real"
)

print(
    "Positive delta = full F4_real coding lower MAE"
)

print(
    "=" * 110
)


display(
    phase2_coding_sensitivity[
        [
            "model",
            "horizon_minutes",
            "scope",
            "F4_real_MAE_kW",
            "comparator_MAE_kW",
            "delta_MAE_comparator_minus_F4_real_kW",
            "ci95_low_kW",
            "ci95_high_kW",
            "evidence_class",
        ]
    ]
)


# ============================================================
# 11. Coding evidence counts by architecture
# ============================================================

phase2_coding_evidence_counts = (
    phase2_coding_sensitivity
    .groupby(
        [
            "model",
            "evidence_class",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=[
            "F4_real_lower_MAE",
            "comparator_lower_MAE",
            "CI_crosses_zero",
        ],
        fill_value=0,
    )
    .reset_index()
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "CODING-SENSITIVITY EVIDENCE COUNTS BY ARCHITECTURE"
)

print(
    "=" * 110
)


display(
    phase2_coding_evidence_counts
)


# ============================================================
# 12. Null-control table
#
# 6 controls:
#   3 shifts
#   3 permutations
#
# 4 models × 3 horizons × 2 scopes × 6
#     = 144 comparisons
# ============================================================

phase2_null_controls = (
    phase2_core_bootstrap_all_models
    .loc[
        phase2_core_bootstrap_all_models[
            "comparator_type"
        ].isin(
            [
                "circular_shift_null",
                "joint_row_permutation_null",
            ]
        )
    ]
    .copy()
)


assert len(
    phase2_null_controls
) == 144


# ============================================================
# 13. Null evidence counts by architecture + null type
# ============================================================

phase2_null_evidence_counts = (
    phase2_null_controls
    .groupby(
        [
            "model",
            "comparator_type",
            "evidence_class",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=[
            "F4_real_lower_MAE",
            "comparator_lower_MAE",
            "CI_crosses_zero",
        ],
        fill_value=0,
    )
    .reset_index()
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "NULL-CONTROL EVIDENCE COUNTS BY ARCHITECTURE"
)

print(
    "=" * 110
)


display(
    phase2_null_evidence_counts
)


# ============================================================
# 14. Per-condition mechanism summary
#
# One row per:
#       model × horizon × scope
#
# 4 × 3 × 2 = 24 rows
#
# Contains:
#   F3 contrast
#   reference-coding contrast
#   3 shift outcomes
#   3 permutation outcomes
# ============================================================

condition_rows = []


for model_name in [
    "Ridge",
    "LightGBM",
    "LSTM",
    "TCN",
]:

    for horizon_minutes in [
        15,
        30,
        60,
    ]:

        for scope in [
            "all",
            "daylight",
        ]:

            condition = (
                phase2_core_bootstrap_all_models
                .loc[
                    (
                        phase2_core_bootstrap_all_models[
                            "model"
                        ].eq(
                            model_name
                        )
                    )
                    &
                    (
                        phase2_core_bootstrap_all_models[
                            "horizon_minutes"
                        ].eq(
                            horizon_minutes
                        )
                    )
                    &
                    (
                        phase2_core_bootstrap_all_models[
                            "scope"
                        ].eq(
                            scope
                        )
                    )
                ]
                .copy()
            )


            assert len(
                condition
            ) == 8


            f3_row = (
                condition
                .loc[
                    condition[
                        "comparator"
                    ].eq(
                        "F3"
                    )
                ]
                .iloc[
                    0
                ]
            )


            reference_row = (
                condition
                .loc[
                    condition[
                        "comparator"
                    ].eq(
                        "F4_reference"
                    )
                ]
                .iloc[
                    0
                ]
            )


            shifts = (
                condition
                .loc[
                    condition[
                        "comparator_type"
                    ].eq(
                        "circular_shift_null"
                    )
                ]
            )


            permutations = (
                condition
                .loc[
                    condition[
                        "comparator_type"
                    ].eq(
                        "joint_row_permutation_null"
                    )
                ]
            )


            assert len(
                shifts
            ) == 3


            assert len(
                permutations
            ) == 3


            def evidence_count(
                frame,
                label,
            ):

                return int(
                    frame[
                        "evidence_class"
                    ]
                    .eq(
                        label
                    )
                    .sum()
                )


            condition_rows.append(
                {
                    "model":
                        model_name,

                    "horizon_minutes":
                        horizon_minutes,

                    "scope":
                        scope,

                    # ----------------------------------------
                    # Primary F3 contrast
                    # ----------------------------------------

                    "F3_delta_kW":
                        float(
                            f3_row[
                                "delta_MAE_comparator_minus_F4_real_kW"
                            ]
                        ),

                    "F3_ci_low_kW":
                        float(
                            f3_row[
                                "ci95_low_kW"
                            ]
                        ),

                    "F3_ci_high_kW":
                        float(
                            f3_row[
                                "ci95_high_kW"
                            ]
                        ),

                    "F3_evidence":
                        str(
                            f3_row[
                                "evidence_class"
                            ]
                        ),

                    # ----------------------------------------
                    # Coding contrast
                    # ----------------------------------------

                    "reference_delta_kW":
                        float(
                            reference_row[
                                "delta_MAE_comparator_minus_F4_real_kW"
                            ]
                        ),

                    "reference_ci_low_kW":
                        float(
                            reference_row[
                                "ci95_low_kW"
                            ]
                        ),

                    "reference_ci_high_kW":
                        float(
                            reference_row[
                                "ci95_high_kW"
                            ]
                        ),

                    "reference_evidence":
                        str(
                            reference_row[
                                "evidence_class"
                            ]
                        ),

                    # ----------------------------------------
                    # Shift controls
                    # ----------------------------------------

                    "shift_mean_delta_kW":
                        float(
                            shifts[
                                "delta_MAE_comparator_minus_F4_real_kW"
                            ]
                            .mean()
                        ),

                    "shift_F4_real_lower_count":
                        evidence_count(
                            shifts,
                            "F4_real_lower_MAE",
                        ),

                    "shift_comparator_lower_count":
                        evidence_count(
                            shifts,
                            "comparator_lower_MAE",
                        ),

                    "shift_unresolved_count":
                        evidence_count(
                            shifts,
                            "CI_crosses_zero",
                        ),

                    # ----------------------------------------
                    # Permutation controls
                    # ----------------------------------------

                    "perm_mean_delta_kW":
                        float(
                            permutations[
                                "delta_MAE_comparator_minus_F4_real_kW"
                            ]
                            .mean()
                        ),

                    "perm_F4_real_lower_count":
                        evidence_count(
                            permutations,
                            "F4_real_lower_MAE",
                        ),

                    "perm_comparator_lower_count":
                        evidence_count(
                            permutations,
                            "comparator_lower_MAE",
                        ),

                    "perm_unresolved_count":
                        evidence_count(
                            permutations,
                            "CI_crosses_zero",
                        ),
                }
            )


phase2_condition_mechanism_summary = (
    pd.DataFrame(
        condition_rows
    )
)


assert len(
    phase2_condition_mechanism_summary
) == 24


assert (
    (
        phase2_condition_mechanism_summary[
            "shift_F4_real_lower_count"
        ]
        +
        phase2_condition_mechanism_summary[
            "shift_comparator_lower_count"
        ]
        +
        phase2_condition_mechanism_summary[
            "shift_unresolved_count"
        ]
    )
    .eq(
        3
    )
    .all()
)


assert (
    (
        phase2_condition_mechanism_summary[
            "perm_F4_real_lower_count"
        ]
        +
        phase2_condition_mechanism_summary[
            "perm_comparator_lower_count"
        ]
        +
        phase2_condition_mechanism_summary[
            "perm_unresolved_count"
        ]
    )
    .eq(
        3
    )
    .all()
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "PER-CONDITION PHASE-2A MECHANISM SUMMARY"
)

print(
    "=" * 110
)


display(
    phase2_condition_mechanism_summary
)


# ============================================================
# 15. Save synthesis tables
# ============================================================

primary_path = (
    PHASE2_RESULTS_DIR
    /
    "phase2_four_model_F3_vs_F4_real.csv"
)


coding_path = (
    PHASE2_RESULTS_DIR
    /
    "phase2_four_model_coding_sensitivity.csv"
)


null_path = (
    PHASE2_RESULTS_DIR
    /
    "phase2_four_model_null_controls.csv"
)


condition_path = (
    PHASE2_RESULTS_DIR
    /
    "phase2_four_model_mechanism_summary.csv"
)


primary_counts_path = (
    PHASE2_RESULTS_DIR
    /
    "phase2_four_model_primary_evidence_counts.csv"
)


coding_counts_path = (
    PHASE2_RESULTS_DIR
    /
    "phase2_four_model_coding_evidence_counts.csv"
)


null_counts_path = (
    PHASE2_RESULTS_DIR
    /
    "phase2_four_model_null_evidence_counts.csv"
)


phase2_primary_f3_vs_f4.to_csv(
    primary_path,
    index=False,
)


phase2_coding_sensitivity.to_csv(
    coding_path,
    index=False,
)


phase2_null_controls.to_csv(
    null_path,
    index=False,
)


phase2_condition_mechanism_summary.to_csv(
    condition_path,
    index=False,
)


phase2_primary_evidence_counts.to_csv(
    primary_counts_path,
    index=False,
)


phase2_coding_evidence_counts.to_csv(
    coding_counts_path,
    index=False,
)


phase2_null_evidence_counts.to_csv(
    null_counts_path,
    index=False,
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "PHASE 2A FOUR-ARCHITECTURE CORE SYNTHESIS: PASSED"
)

print(
    "=" * 110
)


print(
    "Combined bootstrap comparisons:",
    len(
        phase2_core_bootstrap_all_models
    )
)


print(
    "Primary F3/F4 contrasts:",
    len(
        phase2_primary_f3_vs_f4
    )
)


print(
    "Coding contrasts:",
    len(
        phase2_coding_sensitivity
    )
)


print(
    "Null-control contrasts:",
    len(
        phase2_null_controls
    )
)


print(
    "Model × horizon × scope conditions:",
    len(
        phase2_condition_mechanism_summary
    )
)


print(
    "\nSaved:"
)

print(
    " ",
    combined_bootstrap_path
)

print(
    " ",
    condition_path
)


print(
    "\nSTOP HERE."
)

print(
    "No model was trained."
)

FOUR-MODEL BOOTSTRAP MERGE: PASSED
Total comparisons: 192

FOUR-ARCHITECTURE PRIMARY CONTRAST:
F3 vs correctly aligned F4_real
Positive delta = F4_real lower MAE


,model,horizon_minutes,scope,F4_real_MAE_kW,comparator_MAE_kW,delta_MAE_comparator_minus_F4_real_kW,ci95_low_kW,ci95_high_kW,evidence_class
0,LSTM,15,all,0.327740,0.325792,-0.001948,-0.009222,0.005539,CI_crosses_zero
1,LSTM,15,daylight,0.439672,0.451053,0.011381,0.001520,0.021544,F4_real_lower_MAE
2,LSTM,30,all,0.370956,0.385862,0.014906,0.009770,0.019980,F4_real_lower_MAE
3,LSTM,30,daylight,0.532072,0.525864,-0.006208,-0.011577,-0.001022,comparator_lower_MAE
4,LSTM,60,all,0.469986,0.468807,-0.001180,-0.009381,0.007295,CI_crosses_zero
5,LSTM,60,daylight,0.649063,0.651638,0.002576,-0.008825,0.014131,CI_crosses_zero
6,LightGBM,15,all,0.262057,0.260922,-0.001135,-0.002061,-0.000278,comparator_lower_MAE
7,LightGBM,15,daylight,0.386933,0.385134,-0.001799,-0.003181,-0.000522,comparator_lower_MAE
8,LightGBM,30,all,0.300646,0.302415,0.001769,0.000529,0.003158,F4_real_lower_MAE
9,LightGBM,30,daylight,0.443675,0.446240,0.002565,0.000722,0.004650,F4_real_lower_MAE



PRIMARY F3 ↔ F4 EVIDENCE COUNTS BY ARCHITECTURE


evidence_class,model,F4_real_lower_MAE,comparator_lower_MAE,CI_crosses_zero
0,LSTM,2,1,3
1,LightGBM,2,2,2
2,Ridge,0,6,0
3,TCN,3,1,2



FOUR-ARCHITECTURE CODING-SENSITIVITY CONTRAST:
F4_reference vs F4_real
Positive delta = full F4_real coding lower MAE


...


NULL-CONTROL EVIDENCE COUNTS BY ARCHITECTURE


evidence_class,model,comparator_type,F4_real_lower_MAE,comparator_lower_MAE,CI_crosses_zero
0,LSTM,circular_shift_null,6,7,5
1,LSTM,joint_row_permutation_null,5,9,4
2,LightGBM,circular_shift_null,6,0,12
3,LightGBM,joint_row_permutation_null,6,0,12
4,Ridge,circular_shift_null,8,0,10
5,Ridge,joint_row_permutation_null,0,18,0
6,TCN,circular_shift_null,2,11,5
7,TCN,joint_row_permutation_null,2,13,3



PER-CONDITION PHASE-2A MECHANISM SUMMARY


,model,horizon_minutes,scope,F3_delta_kW,F3_ci_low_kW,F3_ci_high_kW,F3_evidence,reference_delta_kW,reference_ci_low_kW,reference_ci_high_kW,reference_evidence,shift_mean_delta_kW,shift_F4_real_lower_count,shift_comparator_lower_count,shift_unresolved_count,perm_mean_delta_kW,perm_F4_real_lower_count,perm_comparator_lower_count,perm_unresolved_count
0,Ridge,15,all,-0.010945,-0.014839,-0.007035,comparator_lower_MAE,3.428228e-09,-6.514217e-09,1.361829e-08,CI_crosses_zero,0.004783,2,0,1,-0.010593,0,3,0
1,Ridge,15,daylight,-0.007050,-0.009805,-0.004321,comparator_lower_MAE,-4.919057e-09,-1.518519e-08,4.949272e-09,CI_crosses_zero,0.003259,2,0,1,-0.006723,0,3,0
2,Ridge,30,all,-0.019218,-0.026424,-0.012280,comparator_lower_MAE,2.964584e-09,-1.176503e-08,1.738539e-08,CI_crosses_zero,0.007953,1,0,2,-0.018796,0,3,0
3,Ridge,30,daylight,-0.012941,-0.018007,-0.008008,comparator_lower_MAE,-1.027437e-08,-2.537148e-08,4.180457e-09,CI_crosses_zero,0.005558,1,0,2,-0.012452,0,3,0
4,Ridge,60,all,-0.035300,-0.048168,-0.022421,comparator_lower_MAE,2.491854e-09,-1.646807e-08,2.143538e-08,CI_crosses_zero,0.013824,1,0,2,-0.035079,0,3,0
5,Ridge,60,daylight,-0.025369,-0.035259,-0.015573,comparator_lower_MAE,-1.597469e-08,-3.641257e-08,3.834391e-09,CI_crosses_zero,0.009494,1,0,2,-0.025040,0,3,0
6,LightGBM,15,all,-0.001135,-0.002061,-0.000278,comparator_lower_MAE,6.011631e-04,-2.041450e-04,1.433976e-03,CI_crosses_zero,0.000419,1,0,2,-0.000284,0,0,3
7,LightGBM,15,daylight,-0.001799,-0.003181,-0.000522,comparator_lower_MAE,8.527254e-04,-3.453151e-04,2.054061e-03,CI_crosses_zero,0.000432,1,0,2,-0.000465,0,0,3
8,LightGBM,30,all,0.001769,0.000529,0.003158,F4_real_lower_MAE,1.420501e-03,3.850983e-04,2.493285e-03,F4_real_lower_MAE,0.001301,2,0,1,0.000694,0,0,3
9,LightGBM,30,daylight,0.002565,0.000722,0.004650,F4_real_lower_MAE,1.748467e-03,2.047872e-04,3.366158e-03,F4_real_lower_MAE,0.001798,2,0,1,0.001021,0,0,3



PHASE 2A FOUR-ARCHITECTURE CORE SYNTHESIS: PASSED
Combined bootstrap comparisons: 192
Primary F3/F4 contrasts: 24
Coding contrasts: 24
Null-control contrasts: 144
Model × horizon × scope conditions: 24

Saved:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_core_four_model_bootstrap_192.csv
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/phase2_four_model_mechanism_summary.csv

STOP HERE.
No model was trained.


In [134]:
# ============================================================
# Step P2-26A:
# Prepare the five frozen add-one Panchang component variants
# for the neural pipeline.
#
# NO MODEL TRAINING occurs here.
#
# Starting representation:
#       F3 auxiliary = 30 columns
#
# Add-one components:
#
#   Tithi      +30 -> auxiliary 60
#   Paksha      +2 -> auxiliary 32
#   Karana     +11 -> auxiliary 41
#   Nakshatra  +27 -> auxiliary 57
#   Yoga       +27 -> auxiliary 57
#
# The added component columns are one-hot binary indicators,
# so they remain unscaled exactly as in full F4.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. Frozen component specification
# ============================================================

PHASE2_COMPONENT_VARIANTS = [
    "F3_plus_tithi",
    "F3_plus_paksha",
    "F3_plus_karana",
    "F3_plus_nakshatra",
    "F3_plus_yoga",
]


PHASE2_COMPONENT_FAMILY = {
    "F3_plus_tithi":
        "tithi",

    "F3_plus_paksha":
        "paksha",

    "F3_plus_karana":
        "karana",

    "F3_plus_nakshatra":
        "nakshatra",

    "F3_plus_yoga":
        "yoga",
}


PHASE2_COMPONENT_ADDON_COUNTS = {
    "F3_plus_tithi":
        30,

    "F3_plus_paksha":
        2,

    "F3_plus_karana":
        11,

    "F3_plus_nakshatra":
        27,

    "F3_plus_yoga":
        27,
}


PHASE2_COMPONENT_EXPECTED_TOTAL_FEATURES = {
    "F3_plus_tithi":
        177,

    "F3_plus_paksha":
        149,

    "F3_plus_karana":
        158,

    "F3_plus_nakshatra":
        174,

    "F3_plus_yoga":
        174,
}


PHASE2_COMPONENT_EXPECTED_AUX_DIMS = {
    variant:
        30
        +
        addon_count

    for variant, addon_count
    in PHASE2_COMPONENT_ADDON_COUNTS.items()
}


assert PHASE2_COMPONENT_EXPECTED_AUX_DIMS == {
    "F3_plus_tithi": 60,
    "F3_plus_paksha": 32,
    "F3_plus_karana": 41,
    "F3_plus_nakshatra": 57,
    "F3_plus_yoga": 57,
}


# ============================================================
# 2. Expected row counts
# ============================================================

EXPECTED_COMPONENT_ROWS = {
    15: 131_325,
    30: 131_322,
    60: 131_316,
}


# ============================================================
# 3. Extend existing neural data with component variants
#
# Key idea:
#
# The standardized F3 auxiliary block is ALREADY correct:
#
#   28 continuous standardized features
#   + weekend binary
#   + daylight binary
#   = 30
#
# Therefore:
#
#   component auxiliary
#       =
#   standardized F3 auxiliary
#       +
#   raw binary one-hot family block
#
# This avoids refitting or reconstructing any scaler.
# ============================================================

component_audit_rows = []


for horizon_minutes in [
    15,
    30,
    60,
]:

    # --------------------------------------------------------
    # Existing F3 modelling table
    # --------------------------------------------------------

    f3_table = (
        phase2_variant_tables[
            horizon_minutes
        ][
            "F3"
        ]
    )


    assert len(
        f3_table
    ) == EXPECTED_COMPONENT_ROWS[
        horizon_minutes
    ]


    assert f3_table.shape[
        1
    ] == 147


    # --------------------------------------------------------
    # Existing correctly standardized neural F3 auxiliary
    # --------------------------------------------------------

    f3_neural_variant = (
        phase2_neural_data[
            horizon_minutes
        ][
            "variants"
        ][
            "F3"
        ]
    )


    f3_auxiliary = np.asarray(
        f3_neural_variant[
            "auxiliary"
        ],
        dtype=np.float32,
    )


    assert f3_auxiliary.shape == (
        EXPECTED_COMPONENT_ROWS[
            horizon_minutes
        ],
        30,
    )


    f3_auxiliary_columns = list(
        f3_neural_variant[
            "auxiliary_columns"
        ]
    )


    f3_continuous_columns = list(
        f3_neural_variant[
            "continuous_columns"
        ]
    )


    f3_binary_columns = list(
        f3_neural_variant[
            "binary_columns"
        ]
    )


    assert len(
        f3_auxiliary_columns
    ) == 30


    assert len(
        f3_continuous_columns
    ) == 28


    assert len(
        f3_binary_columns
    ) == 2


    f3_full_column_set = set(
        f3_table.columns
    )


    # ========================================================
    # Each add-one component
    # ========================================================

    for variant_name in (
        PHASE2_COMPONENT_VARIANTS
    ):

        family_name = (
            PHASE2_COMPONENT_FAMILY[
                variant_name
            ]
        )


        component_table = (
            phase2_variant_tables[
                horizon_minutes
            ][
                variant_name
            ]
        )


        expected_rows = (
            EXPECTED_COMPONENT_ROWS[
                horizon_minutes
            ]
        )


        expected_total_features = (
            PHASE2_COMPONENT_EXPECTED_TOTAL_FEATURES[
                variant_name
            ]
        )


        expected_addon_count = (
            PHASE2_COMPONENT_ADDON_COUNTS[
                variant_name
            ]
        )


        expected_aux_dim = (
            PHASE2_COMPONENT_EXPECTED_AUX_DIMS[
                variant_name
            ]
        )


        # ----------------------------------------------------
        # Structural checks
        # ----------------------------------------------------

        assert len(
            component_table
        ) == expected_rows


        assert component_table.shape[
            1
        ] == expected_total_features


        assert component_table.index.equals(
            f3_table.index
        )


        # ----------------------------------------------------
        # Identify ONLY columns added on top of F3.
        #
        # Preserve component-table column order.
        # ----------------------------------------------------

        component_columns = [
            column
            for column
            in component_table.columns
            if column
            not in
            f3_full_column_set
        ]


        assert len(
            component_columns
        ) == expected_addon_count


        # ----------------------------------------------------
        # Strong family-prefix check
        # ----------------------------------------------------

        expected_prefix = (
            f"valid_{family_name}_"
        )


        assert all(
            str(
                column
            ).startswith(
                expected_prefix
            )
            for column
            in component_columns
        ), (
            variant_name,
            component_columns[
                :5
            ],
        )


        # ----------------------------------------------------
        # Added indicators must be strictly binary.
        # ----------------------------------------------------

        component_binary_matrix = (
            component_table[
                component_columns
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        assert component_binary_matrix.shape == (
            expected_rows,
            expected_addon_count,
        )


        assert np.isfinite(
            component_binary_matrix
        ).all()


        unique_values = np.unique(
            component_binary_matrix
        )


        assert set(
            unique_values.tolist()
        ).issubset(
            {
                0.0,
                1.0,
            }
        )


        # ----------------------------------------------------
        # One-hot activation audit:
        #
        # Exactly ONE indicator from the selected family must
        # be active for every row.
        # ----------------------------------------------------

        row_activation_count = (
            component_binary_matrix
            .sum(
                axis=1
            )
        )


        assert np.allclose(
            row_activation_count,
            1.0,
        )


        # ----------------------------------------------------
        # Build component neural auxiliary matrix
        #
        # Existing standardized F3
        # +
        # raw binary family one-hot
        # ----------------------------------------------------

        component_auxiliary = np.concatenate(
            [
                f3_auxiliary,
                component_binary_matrix,
            ],
            axis=1,
        ).astype(
            np.float32,
            copy=False,
        )


        assert component_auxiliary.shape == (
            expected_rows,
            expected_aux_dim,
        )


        assert np.isfinite(
            component_auxiliary
        ).all()


        # ----------------------------------------------------
        # Register inside existing neural-data structure
        # ----------------------------------------------------

        phase2_neural_data[
            horizon_minutes
        ][
            "variants"
        ][
            variant_name
        ] = {

            "auxiliary":
                component_auxiliary,

            "auxiliary_columns":
                (
                    f3_auxiliary_columns
                    +
                    component_columns
                ),

            "continuous_columns":
                f3_continuous_columns.copy(),

            "binary_columns":
                (
                    f3_binary_columns
                    +
                    component_columns
                ),

            "component_family":
                family_name,

            "component_columns":
                component_columns.copy(),

            "component_addon_count":
                expected_addon_count,
        }


        # ----------------------------------------------------
        # Audit row
        # ----------------------------------------------------

        component_audit_rows.append(
            {
                "horizon_minutes":
                    horizon_minutes,

                "variant":
                    variant_name,

                "family":
                    family_name,

                "full_feature_dim":
                    component_table.shape[
                        1
                    ],

                "base_F3_aux_dim":
                    30,

                "added_binary_columns":
                    expected_addon_count,

                "component_aux_dim":
                    component_auxiliary.shape[
                        1
                    ],

                "rows":
                    expected_rows,

                "one_active_per_row":
                    bool(
                        np.allclose(
                            row_activation_count,
                            1.0,
                        )
                    ),
            }
        )


# ============================================================
# 4. Final audit table
# ============================================================

phase2_component_neural_audit = (
    pd.DataFrame(
        component_audit_rows
    )
)


assert len(
    phase2_component_neural_audit
) == (
    3
    *
    5
)


assert (
    phase2_component_neural_audit[
        "one_active_per_row"
    ]
    .all()
)


print(
    "=" * 90
)

print(
    "PHASE 2A ADD-ONE COMPONENT NEURAL PREPARATION: PASSED"
)

print(
    "=" * 90
)


display(
    phase2_component_neural_audit
)


print(
    "\nExpected auxiliary dimensions:"
)

print(
    PHASE2_COMPONENT_EXPECTED_AUX_DIMS
)


print(
    "\nNo model was trained."
)

PHASE 2A ADD-ONE COMPONENT NEURAL PREPARATION: PASSED


,horizon_minutes,variant,family,full_feature_dim,base_F3_aux_dim,added_binary_columns,component_aux_dim,rows,one_active_per_row
0,15,F3_plus_tithi,tithi,177,30,30,60,131325,True
1,15,F3_plus_paksha,paksha,149,30,2,32,131325,True
2,15,F3_plus_karana,karana,158,30,11,41,131325,True
3,15,F3_plus_nakshatra,nakshatra,174,30,27,57,131325,True
4,15,F3_plus_yoga,yoga,174,30,27,57,131325,True
5,30,F3_plus_tithi,tithi,177,30,30,60,131322,True
6,30,F3_plus_paksha,paksha,149,30,2,32,131322,True
7,30,F3_plus_karana,karana,158,30,11,41,131322,True
8,30,F3_plus_nakshatra,nakshatra,174,30,27,57,131322,True
9,30,F3_plus_yoga,yoga,174,30,27,57,131322,True



Expected auxiliary dimensions:
{'F3_plus_tithi': 60, 'F3_plus_paksha': 32, 'F3_plus_karana': 41, 'F3_plus_nakshatra': 57, 'F3_plus_yoga': 57}

No model was trained.


In [135]:
# ============================================================
# Step P2-26B:
# Classical add-one Panchang component screen
#
# Models:
#   Ridge
#   LightGBM
#
# Horizons:
#   15, 30, 60 minutes
#
# Component variants:
#   F3_plus_tithi
#   F3_plus_paksha
#   F3_plus_karana
#   F3_plus_nakshatra
#   F3_plus_yoga
#
# Fits:
#   2 models × 3 horizons × 5 variants
#   = 30 model fits
#
# Metrics:
#   all
#   daylight
#
# Therefore:
#   30 fits
#   60 metric rows
#
# IMPORTANT
# ---------
# Uses the SAME frozen F4-selected model configuration as
# Phase-2 core analysis.
#
# No representation-specific tuning occurs.
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

from lightgbm import LGBMRegressor


# ============================================================
# 1. Frozen experiment grid
# ============================================================

PHASE2_COMPONENT_CLASSICAL_MODELS = [
    "Ridge",
    "LightGBM",
]


PHASE2_COMPONENT_HORIZONS = [
    15,
    30,
    60,
]


assert PHASE2_COMPONENT_VARIANTS == [
    "F3_plus_tithi",
    "F3_plus_paksha",
    "F3_plus_karana",
    "F3_plus_nakshatra",
    "F3_plus_yoga",
]


EXPECTED_COMPONENT_CLASSICAL_FITS = (
    len(
        PHASE2_COMPONENT_CLASSICAL_MODELS
    )
    *
    len(
        PHASE2_COMPONENT_HORIZONS
    )
    *
    len(
        PHASE2_COMPONENT_VARIANTS
    )
)


assert (
    EXPECTED_COMPONENT_CLASSICAL_FITS
    ==
    30
)


# ============================================================
# 2. Output directory
# ============================================================

PHASE2_COMPONENT_DIR = (
    PHASE2_RESULTS_DIR
    /
    "component_screen"
)


PHASE2_COMPONENT_CLASSICAL_DIR = (
    PHASE2_COMPONENT_DIR
    /
    "classical"
)


PHASE2_COMPONENT_CLASSICAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Classical component output directory:"
)

print(
    " ",
    PHASE2_COMPONENT_CLASSICAL_DIR
)


# ============================================================
# 3. Containers
# ============================================================

component_classical_result_rows = []


component_classical_prediction_tables = {}


component_fit_counter = 0


component_start_time = (
    time.perf_counter()
)


# ============================================================
# 4. Execute horizon-by-horizon
# ============================================================

for horizon_minutes in (
    PHASE2_COMPONENT_HORIZONS
):

    print(
        "\n"
        +
        "=" * 94
    )

    print(
        f"CLASSICAL COMPONENT SCREEN — "
        f"{horizon_minutes}-MINUTE HORIZON"
    )

    print(
        "=" * 94
    )


    # --------------------------------------------------------
    # Frozen modelling bundle
    # --------------------------------------------------------

    bundle = (
        modelling_bundles[
            horizon_minutes
        ]
    )


    variants = (
        phase2_variant_tables[
            horizon_minutes
        ]
    )


    y = (
        bundle[
            "y"
        ]
    )


    # --------------------------------------------------------
    # Same final-refit population as Stage 1
    # --------------------------------------------------------

    fit_mask = (
        bundle[
            "train_mask"
        ]
        |
        bundle[
            "validation_mask"
        ]
    )


    test_mask = (
        bundle[
            "test_mask"
        ]
    )


    daylight_test_mask = (
        test_mask
        &
        bundle[
            "daylight_mask"
        ]
    )


    y_fit = (
        y.loc[
            fit_mask
        ]
    )


    y_test = (
        y.loc[
            test_mask
        ]
    )


    test_index = (
        y_test.index
    )


    # --------------------------------------------------------
    # Population audits
    # --------------------------------------------------------

    expected_train_counts = {
        15: 87_261,
        30: 87_258,
        60: 87_252,
    }


    assert int(
        bundle[
            "train_mask"
        ].sum()
    ) == expected_train_counts[
        horizon_minutes
    ]


    assert int(
        bundle[
            "validation_mask"
        ].sum()
    ) == 17_568


    assert int(
        test_mask.sum()
    ) == 26_496


    assert len(
        y_test
    ) == 26_496


    # --------------------------------------------------------
    # TEST daylight vector
    # --------------------------------------------------------

    daylight_on_test = (
        daylight_test_mask
        .loc[
            test_index
        ]
        .to_numpy(
            dtype=bool
        )
    )


    assert daylight_on_test.shape == (
        26_496,
    )


    # --------------------------------------------------------
    # Prediction table
    # --------------------------------------------------------

    metadata_test = (
        bundle[
            "metadata"
        ]
        .loc[
            test_mask
        ]
        .copy()
    )


    prediction_table = pd.DataFrame(
        index=test_index
    )


    prediction_table[
        "target_P_Solar_kW"
    ] = (
        y_test.to_numpy(
            dtype=float
        )
    )


    prediction_table[
        "daylight"
    ] = (
        daylight_on_test
    )


    # ========================================================
    # 5. Model loop
    # ========================================================

    for model_name in (
        PHASE2_COMPONENT_CLASSICAL_MODELS
    ):

        print(
            f"\n--- {model_name} ---"
        )


        frozen_config = (
            PHASE2_FROZEN_MODEL_CONFIGS[
                model_name
            ][
                horizon_minutes
            ]
        )


        # ====================================================
        # 6. Component loop
        # ====================================================

        for variant_name in (
            PHASE2_COMPONENT_VARIANTS
        ):

            component_fit_counter += 1


            X = (
                variants[
                    variant_name
                ]
            )


            expected_feature_dim = (
                PHASE2_COMPONENT_EXPECTED_TOTAL_FEATURES[
                    variant_name
                ]
            )


            assert (
                X.shape[
                    1
                ]
                ==
                expected_feature_dim
            )


            assert X.index.equals(
                y.index
            )


            X_fit = (
                X.loc[
                    fit_mask
                ]
            )


            X_test = (
                X.loc[
                    test_mask
                ]
            )


            assert len(
                X_fit
            ) == (
                expected_train_counts[
                    horizon_minutes
                ]
                +
                17_568
            )


            assert len(
                X_test
            ) == 26_496


            fit_predict_start = (
                time.perf_counter()
            )


            # =================================================
            # RIDGE
            #
            # Exact Stage-1 procedure:
            #
            # StandardScaler()
            # Ridge(alpha=frozen F4-selected alpha)
            # =================================================

            if model_name == "Ridge":

                alpha = float(
                    frozen_config[
                        "alpha"
                    ]
                )


                model = Pipeline(
                    steps=[
                        (
                            "scaler",
                            StandardScaler(),
                        ),
                        (
                            "ridge",
                            Ridge(
                                alpha=alpha
                            ),
                        ),
                    ]
                )


                model.fit(
                    X_fit,
                    y_fit,
                )


                test_predictions = (
                    model.predict(
                        X_test
                    )
                )


                config_string = (
                    f"alpha={alpha:g}"
                )


            # =================================================
            # LIGHTGBM
            #
            # Exact Stage-1 procedure:
            #
            # frozen F4-selected parameters
            # float32 TRAIN+VALIDATION / TEST arrays
            # =================================================

            elif model_name == "LightGBM":

                lgbm_config = (
                    frozen_config.copy()
                )


                model = LGBMRegressor(
                    **lgbm_config
                )


                model.fit(

                    X_fit.to_numpy(
                        dtype=np.float32
                    ),

                    y_fit.to_numpy(
                        dtype=np.float32
                    ),
                )


                test_predictions = (
                    model.predict(

                        X_test.to_numpy(
                            dtype=np.float32
                        )
                    )
                )


                config_string = (
                    json.dumps(
                        lgbm_config,
                        sort_keys=True,
                    )
                )


            else:

                raise ValueError(
                    model_name
                )


            fit_predict_seconds = float(
                time.perf_counter()
                -
                fit_predict_start
            )


            test_predictions = np.asarray(
                test_predictions,
                dtype=np.float64,
            ).reshape(
                -1
            )


            assert test_predictions.shape == (
                26_496,
            )


            assert np.isfinite(
                test_predictions
            ).all()


            # =================================================
            # 7. Store TEST predictions
            # =================================================

            prediction_column = (
                f"{model_name}"
                f"__{variant_name}"
            )


            assert (
                prediction_column
                not in
                prediction_table.columns
            )


            prediction_table[
                prediction_column
            ] = (
                test_predictions
            )


            # =================================================
            # 8. All-row + daylight metrics
            # =================================================

            scope_masks = {

                "all":
                    np.ones(
                        len(
                            y_test
                        ),
                        dtype=bool,
                    ),

                "daylight":
                    daylight_on_test,
            }


            for (
                scope_name,
                scope_mask,
            ) in (
                scope_masks.items()
            ):

                y_scope = (
                    y_test
                    .to_numpy(
                        dtype=np.float64
                    )[
                        scope_mask
                    ]
                )


                prediction_scope = (
                    test_predictions[
                        scope_mask
                    ]
                )


                metrics = (
                    phase2_regression_metrics(
                        y_scope,
                        prediction_scope,
                    )
                )


                component_classical_result_rows.append(
                    {
                        "model":
                            model_name,

                        "horizon_minutes":
                            horizon_minutes,

                        "variant":
                            variant_name,

                        "family":
                            PHASE2_COMPONENT_FAMILY[
                                variant_name
                            ],

                        "scope":
                            scope_name,

                        "n_rows":
                            int(
                                scope_mask.sum()
                            ),

                        "n_features":
                            int(
                                X.shape[
                                    1
                                ]
                            ),

                        "frozen_config":
                            config_string,

                        "fit_predict_seconds":
                            fit_predict_seconds,

                        **metrics,
                    }
                )


            # =================================================
            # 9. Immediate run checkpoint
            # =================================================

            single_run_result = pd.DataFrame(
                [
                    row
                    for row
                    in component_classical_result_rows
                    if (
                        row[
                            "model"
                        ]
                        ==
                        model_name
                    )
                    and (
                        row[
                            "horizon_minutes"
                        ]
                        ==
                        horizon_minutes
                    )
                    and (
                        row[
                            "variant"
                        ]
                        ==
                        variant_name
                    )
                ]
            )


            assert len(
                single_run_result
            ) == 2


            run_stem = (
                f"{model_name.lower()}"
                f"__{horizon_minutes}min"
                f"__{variant_name}"
            )


            single_run_result.to_csv(

                PHASE2_COMPONENT_CLASSICAL_DIR
                /
                (
                    run_stem
                    +
                    "__metrics.csv"
                ),

                index=False,
            )


            pd.DataFrame(
                {
                    "Timestamp":
                        test_index,

                    "target_P_Solar_kW":
                        y_test.to_numpy(
                            dtype=float
                        ),

                    "daylight":
                        daylight_on_test,

                    "prediction_kW":
                        test_predictions,
                }
            ).to_csv(

                PHASE2_COMPONENT_CLASSICAL_DIR
                /
                (
                    run_stem
                    +
                    "__predictions.csv"
                ),

                index=False,
            )


            print(
                f"[{component_fit_counter:02d}/30] "
                f"{variant_name:<19} "
                f"features={X.shape[1]:>3} "
                f"MAE="
                f"{single_run_result.loc[
                    single_run_result['scope'].eq('all'),
                    'MAE_kW'
                ].iloc[0]:.6f} "
                f"time={fit_predict_seconds:.2f}s"
            )


    # ========================================================
    # 10. Add metadata and persist horizon prediction table
    # ========================================================

    prediction_table.insert(
        0,
        "valid_time",
        metadata_test[
            "valid_time"
        ],
    )


    prediction_table.insert(
        0,
        "issue_time",
        metadata_test[
            "issue_time"
        ],
    )


    prediction_table.index.name = (
        "Timestamp"
    )


    component_classical_prediction_tables[
        horizon_minutes
    ] = (
        prediction_table
    )


    prediction_table.to_csv(

        PHASE2_COMPONENT_CLASSICAL_DIR
        /
        (
            "phase2_component_classical_"
            f"predictions_{horizon_minutes}min.csv"
        ),

        index=True,
    )


# ============================================================
# 11. Aggregate results
# ============================================================

phase2_component_classical_results = (
    pd.DataFrame(
        component_classical_result_rows
    )
)


assert (
    component_fit_counter
    ==
    30
)


# 30 fits × 2 scopes = 60 result rows

assert len(
    phase2_component_classical_results
) == 60


assert (
    phase2_component_classical_results[
        [
            "model",
            "horizon_minutes",
            "variant",
            "scope",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
)


assert np.isfinite(

    phase2_component_classical_results[
        [
            "MAE_kW",
            "RMSE_kW",
            "nMAE_percent",
            "nRMSE_percent",
            "R2",
        ]
    ]
    .to_numpy(
        dtype=float
    )

).all()


# ============================================================
# 12. Persist aggregate
# ============================================================

component_classical_results_path = (
    PHASE2_COMPONENT_CLASSICAL_DIR
    /
    "phase2_component_classical_results.csv"
)


phase2_component_classical_results.to_csv(
    component_classical_results_path,
    index=False,
)


# ============================================================
# 13. All-row MAE pivot
# ============================================================

component_classical_all_mae = (
    phase2_component_classical_results
    .loc[
        phase2_component_classical_results[
            "scope"
        ].eq(
            "all"
        )
    ]
    .pivot_table(
        index=[
            "model",
            "horizon_minutes",
        ],
        columns="variant",
        values="MAE_kW",
        aggfunc="first",
    )
    .reset_index()
)


# ============================================================
# 14. Daylight MAE pivot
# ============================================================

component_classical_daylight_mae = (
    phase2_component_classical_results
    .loc[
        phase2_component_classical_results[
            "scope"
        ].eq(
            "daylight"
        )
    ]
    .pivot_table(
        index=[
            "model",
            "horizon_minutes",
        ],
        columns="variant",
        values="MAE_kW",
        aggfunc="first",
    )
    .reset_index()
)


component_classical_all_mae.to_csv(

    PHASE2_COMPONENT_CLASSICAL_DIR
    /
    "phase2_component_classical_all_mae.csv",

    index=False,
)


component_classical_daylight_mae.to_csv(

    PHASE2_COMPONENT_CLASSICAL_DIR
    /
    "phase2_component_classical_daylight_mae.csv",

    index=False,
)


# ============================================================
# 15. Completion report
# ============================================================

component_total_seconds = float(
    time.perf_counter()
    -
    component_start_time
)


print(
    "\n"
    +
    "=" * 94
)

print(
    "PHASE 2A CLASSICAL ADD-ONE COMPONENT SCREEN: PASSED"
)

print(
    "=" * 94
)


print(
    "Completed model fits:",
    component_fit_counter,
)


print(
    "Metric rows:",
    len(
        phase2_component_classical_results
    ),
)


print(
    "Runtime:",
    f"{component_total_seconds:.2f} seconds",
)


print(
    "\nAll-row MAE:"
)

display(
    component_classical_all_mae
)


print(
    "\nDaylight MAE:"
)

display(
    component_classical_daylight_mae
)


print(
    "\nSaved:"
)

print(
    " ",
    component_classical_results_path
)

Classical component output directory:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/component_screen/classical

CLASSICAL COMPONENT SCREEN — 15-MINUTE HORIZON

--- Ridge ---
[01/30] F3_plus_tithi       features=177 MAE=0.321373 time=0.35s
[02/30] F3_plus_paksha      features=149 MAE=0.318608 time=0.10s
[03/30] F3_plus_karana      features=158 MAE=0.318701 time=0.10s
[04/30] F3_plus_nakshatra   features=174 MAE=0.319186 time=0.11s
[05/30] F3_plus_yoga        features=174 MAE=0.326440 time=0.11s

--- LightGBM ---
[06/30] F3_plus_tithi       features=177 MAE=0.262153 time=3.95s
[07/30] F3_plus_paksha      features=149 MAE=0.262614 time=3.30s
[08/30] F3_plus_karana      features=158 MAE=0.261469 time=3.14s
[09/30] F3_plus_nakshatra   features=174 MAE=0.261341 time=3.46s
[10/30] F3_plus_yoga        features=174 MAE=0.260982 time=3.44s

CLASSICAL COMPONENT SCREEN — 30-MINUTE HORIZON

--- Ridge -

variant,model,horizon_minutes,F3_plus_karana,F3_plus_nakshatra,F3_plus_paksha,F3_plus_tithi,F3_plus_yoga
0,LightGBM,15,0.261469,0.261341,0.262614,0.262153,0.260982
1,LightGBM,30,0.301141,0.302187,0.301634,0.301460,0.301705
2,LightGBM,60,0.346966,0.343358,0.343733,0.343358,0.344285
3,Ridge,15,0.318701,0.319186,0.318608,0.321373,0.326440
4,Ridge,30,0.401873,0.402970,0.401799,0.406627,0.415739
5,Ridge,60,0.527761,0.530023,0.527744,0.536629,0.553418



Daylight MAE:


variant,model,horizon_minutes,F3_plus_karana,F3_plus_nakshatra,F3_plus_paksha,F3_plus_tithi,F3_plus_yoga
0,LightGBM,15,0.386048,0.385519,0.387483,0.386966,0.385036
1,LightGBM,30,0.444524,0.446079,0.444941,0.444707,0.445222
2,LightGBM,60,0.511266,0.506770,0.507112,0.506762,0.507914
3,Ridge,15,0.414188,0.414271,0.414023,0.415714,0.418876
4,Ridge,30,0.496635,0.497075,0.496471,0.499490,0.505618
5,Ridge,60,0.609452,0.610731,0.609421,0.615162,0.627653



Saved:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/component_screen/classical/phase2_component_classical_results.csv


In [136]:
# ============================================================
# Step P2-26C:
# RESUMABLE neural add-one Panchang component screen
#
# Models:
#   LSTM
#   TCN
#
# Horizons:
#   15, 30, 60 minutes
#
# Component variants:
#   F3_plus_tithi
#   F3_plus_paksha
#   F3_plus_karana
#   F3_plus_nakshatra
#   F3_plus_yoga
#
# Total:
#   2 × 3 × 5 = 30 neural fits
#
# Uses:
#   - frozen F4-selected architecture
#   - frozen F4-selected epoch count
#   - seed 42
#   - TRAIN + VALIDATION final refit
#   - batch size 256
#   - shuffle=True
#   - no early stopping
#   - no validation feedback
#   - no prediction clipping
#
# Every completed run is persisted immediately.
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. Frozen component neural grid
# ============================================================

PHASE2_COMPONENT_NEURAL_MODELS = [
    "LSTM",
    "TCN",
]


PHASE2_COMPONENT_NEURAL_HORIZONS = [
    15,
    30,
    60,
]


assert PHASE2_COMPONENT_VARIANTS == [
    "F3_plus_tithi",
    "F3_plus_paksha",
    "F3_plus_karana",
    "F3_plus_nakshatra",
    "F3_plus_yoga",
]


EXPECTED_COMPONENT_NEURAL_FITS = (
    len(
        PHASE2_COMPONENT_NEURAL_MODELS
    )
    *
    len(
        PHASE2_COMPONENT_NEURAL_HORIZONS
    )
    *
    len(
        PHASE2_COMPONENT_VARIANTS
    )
)


assert EXPECTED_COMPONENT_NEURAL_FITS == 30


# ============================================================
# 2. Confirm P2-26A component preparation still exists
# ============================================================

for horizon_minutes in (
    PHASE2_COMPONENT_NEURAL_HORIZONS
):

    for variant_name in (
        PHASE2_COMPONENT_VARIANTS
    ):

        assert (
            variant_name
            in
            phase2_neural_data[
                horizon_minutes
            ][
                "variants"
            ]
        ), (
            "Component neural representation missing. "
            "Re-run P2-26A first: "
            f"{horizon_minutes}, {variant_name}"
        )


        auxiliary = (
            phase2_neural_data[
                horizon_minutes
            ][
                "variants"
            ][
                variant_name
            ][
                "auxiliary"
            ]
        )


        assert (
            auxiliary.shape[
                1
            ]
            ==
            PHASE2_COMPONENT_EXPECTED_AUX_DIMS[
                variant_name
            ]
        )


print(
    "Component neural representations: PASSED"
)


# ============================================================
# 3. Output directory
# ============================================================

PHASE2_COMPONENT_NEURAL_DIR = (
    PHASE2_COMPONENT_DIR
    /
    "neural"
)


PHASE2_COMPONENT_NEURAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Neural component output directory:"
)

print(
    " ",
    PHASE2_COMPONENT_NEURAL_DIR
)


# ============================================================
# 4. Run-path helper
# ============================================================

def phase2_component_neural_run_paths(
    model_name,
    horizon_minutes,
    variant_name,
):

    run_id = (
        f"{model_name.lower()}"
        f"__{int(horizon_minutes)}min"
        f"__{variant_name}"
    )


    run_dir = (
        PHASE2_COMPONENT_NEURAL_DIR
        /
        run_id
    )


    return {

        "run_id":
            run_id,

        "run_dir":
            run_dir,

        "metrics":
            run_dir
            /
            "metrics.csv",

        "history":
            run_dir
            /
            "history.csv",

        "predictions":
            run_dir
            /
            "predictions.csv",

        "metadata":
            run_dir
            /
            "metadata.json",

        "failure":
            run_dir
            /
            "failure.json",
    }


# ============================================================
# 5. Atomic persistence
# ============================================================

def component_atomic_csv(
    dataframe,
    path,
    index=False,
):

    path = Path(
        path
    )


    temporary_path = (
        path.parent
        /
        (
            path.name
            +
            ".tmp"
        )
    )


    dataframe.to_csv(
        temporary_path,
        index=index,
    )


    temporary_path.replace(
        path
    )


def component_atomic_json(
    payload,
    path,
):

    path = Path(
        path
    )


    temporary_path = (
        path.parent
        /
        (
            path.name
            +
            ".tmp"
        )
    )


    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            default=str,
        )


    temporary_path.replace(
        path
    )


# ============================================================
# 6. Saved-run completion validator
# ============================================================

def phase2_component_neural_run_complete(
    model_name,
    horizon_minutes,
    variant_name,
):

    paths = (
        phase2_component_neural_run_paths(

            model_name,
            horizon_minutes,
            variant_name,
        )
    )


    required = [
        paths[
            "metrics"
        ],
        paths[
            "history"
        ],
        paths[
            "predictions"
        ],
        paths[
            "metadata"
        ],
    ]


    if not all(
        path.is_file()
        for path
        in required
    ):
        return False


    try:

        metrics = pd.read_csv(
            paths[
                "metrics"
            ]
        )


        predictions = pd.read_csv(
            paths[
                "predictions"
            ]
        )


        history = pd.read_csv(
            paths[
                "history"
            ]
        )


        if len(
            metrics
        ) != 2:

            return False


        if set(
            metrics[
                "scope"
            ]
        ) != {
            "all",
            "daylight",
        }:

            return False


        if len(
            predictions
        ) != 26_496:

            return False


        if not {
            "issue_time",
            "valid_time",
            "daylight",
            "target_P_Solar_kW",
            "prediction_kW",
            "prediction_scaled",
        }.issubset(
            predictions.columns
        ):

            return False


        if not np.isfinite(
            predictions[
                [
                    "target_P_Solar_kW",
                    "prediction_kW",
                    "prediction_scaled",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        ).all():

            return False


        expected_epochs = int(
            PHASE2_FROZEN_MODEL_CONFIGS[
                model_name
            ][
                horizon_minutes
            ][
                "epochs"
            ]
        )


        if len(
            history
        ) != expected_epochs:

            return False


        with open(
            paths[
                "metadata"
            ],
            "r",
            encoding="utf-8",
        ) as handle:

            metadata = json.load(
                handle
            )


        if not metadata.get(
            "completed_successfully",
            False,
        ):

            return False


        if (
            metadata[
                "model"
            ]
            !=
            model_name
        ):

            return False


        if (
            int(
                metadata[
                    "horizon_minutes"
                ]
            )
            !=
            int(
                horizon_minutes
            )
        ):

            return False


        if (
            metadata[
                "variant"
            ]
            !=
            variant_name
        ):

            return False


        return True


    except Exception:

        return False


# ============================================================
# 7. One frozen component neural refit
# ============================================================

def run_phase2_neural_component_refit(
    model_name,
    horizon_minutes,
    variant_name,
    verbose=0,
):

    model_name = str(
        model_name
    )

    horizon_minutes = int(
        horizon_minutes
    )

    variant_name = str(
        variant_name
    )


    assert model_name in {
        "LSTM",
        "TCN",
    }


    assert horizon_minutes in {
        15,
        30,
        60,
    }


    assert (
        variant_name
        in
        PHASE2_COMPONENT_VARIANTS
    )


    # ========================================================
    # A. Frozen neural data
    # ========================================================

    data = (
        phase2_neural_data[
            horizon_minutes
        ]
    )


    variant_data = (
        data[
            "variants"
        ][
            variant_name
        ]
    )


    sequence_all = np.asarray(
        data[
            "sequence"
        ],
        dtype=np.float32,
    )


    auxiliary_all = np.asarray(
        variant_data[
            "auxiliary"
        ],
        dtype=np.float32,
    )


    target_scaled_all = np.asarray(
        data[
            "target_scaled"
        ],
        dtype=np.float32,
    )


    target_kw_all = np.asarray(
        data[
            "target_kw"
        ],
        dtype=np.float32,
    )


    train_mask = np.asarray(
        data[
            "train_mask"
        ],
        dtype=bool,
    )


    validation_mask = np.asarray(
        data[
            "validation_mask"
        ],
        dtype=bool,
    )


    test_mask = np.asarray(
        data[
            "test_mask"
        ],
        dtype=bool,
    )


    auxiliary_dim = int(
        auxiliary_all.shape[
            1
        ]
    )


    assert (
        auxiliary_dim
        ==
        PHASE2_COMPONENT_EXPECTED_AUX_DIMS[
            variant_name
        ]
    )


    # ========================================================
    # B. Frozen population audit
    # ========================================================

    expected_train_counts = {
        15: 87_261,
        30: 87_258,
        60: 87_252,
    }


    n_train = int(
        train_mask.sum()
    )


    n_validation = int(
        validation_mask.sum()
    )


    n_test = int(
        test_mask.sum()
    )


    assert (
        n_train
        ==
        expected_train_counts[
            horizon_minutes
        ]
    )


    assert n_validation == 17_568

    assert n_test == 26_496


    n_trainval = (
        n_train
        +
        n_validation
    )


    # ========================================================
    # C. Explicit TRAIN then VALIDATION concatenation
    # ========================================================

    train_sequence = (
        sequence_all[
            train_mask
        ]
    )


    validation_sequence = (
        sequence_all[
            validation_mask
        ]
    )


    test_sequence = (
        sequence_all[
            test_mask
        ]
    )


    train_auxiliary = (
        auxiliary_all[
            train_mask
        ]
    )


    validation_auxiliary = (
        auxiliary_all[
            validation_mask
        ]
    )


    test_auxiliary = (
        auxiliary_all[
            test_mask
        ]
    )


    train_target_scaled = (
        target_scaled_all[
            train_mask
        ]
    )


    validation_target_scaled = (
        target_scaled_all[
            validation_mask
        ]
    )


    test_target_kw = np.asarray(
        target_kw_all[
            test_mask
        ],
        dtype=np.float64,
    ).reshape(
        -1
    )


    trainval_sequence = np.concatenate(
        [
            train_sequence,
            validation_sequence,
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    trainval_auxiliary = np.concatenate(
        [
            train_auxiliary,
            validation_auxiliary,
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    trainval_target_scaled = np.concatenate(
        [
            train_target_scaled,
            validation_target_scaled,
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    assert trainval_sequence.shape == (
        n_trainval,
        13,
        9,
    )


    assert trainval_auxiliary.shape == (
        n_trainval,
        auxiliary_dim,
    )


    assert test_sequence.shape == (
        n_test,
        13,
        9,
    )


    assert test_auxiliary.shape == (
        n_test,
        auxiliary_dim,
    )


    # ========================================================
    # D. Preserve Notebook-11 model-specific target shape
    # ========================================================

    if model_name == "LSTM":

        final_fit_target = (
            trainval_target_scaled
            .reshape(
                -1
            )
        )


        assert final_fit_target.shape == (
            n_trainval,
        )


    else:

        final_fit_target = (
            trainval_target_scaled
            .reshape(
                -1,
                1,
            )
        )


        assert final_fit_target.shape == (
            n_trainval,
            1,
        )


    # ========================================================
    # E. Frozen architecture / epoch count
    # ========================================================

    frozen_config = (
        PHASE2_FROZEN_MODEL_CONFIGS[
            model_name
        ][
            horizon_minutes
        ]
    )


    architecture = (
        frozen_config[
            "architecture"
        ]
        .copy()
    )


    frozen_epochs = int(
        frozen_config[
            "epochs"
        ]
    )


    expected_epochs = {

        "LSTM": {
            15: 7,
            30: 1,
            60: 2,
        },

        "TCN": {
            15: 4,
            30: 1,
            60: 1,
        },
    }


    assert (
        frozen_epochs
        ==
        expected_epochs[
            model_name
        ][
            horizon_minutes
        ]
    )


    # ========================================================
    # F. Fresh seeded model
    # ========================================================

    model = build_phase2_neural_model(

        model_name=
            model_name,

        auxiliary_dim=
            auxiliary_dim,

        architecture=
            architecture,

        seed=
            PHASE2_NEURAL_SEED,
    )


    expected_parameters = (
        expected_phase2_neural_parameter_count(

            model_name=
                model_name,

            auxiliary_dim=
                auxiliary_dim,
        )
    )


    actual_parameters = int(
        model.count_params()
    )


    assert (
        actual_parameters
        ==
        expected_parameters
    )


    # --------------------------------------------------------
    # Structural audit
    # --------------------------------------------------------

    if model_name == "LSTM":

        recurrent_layers = [
            layer
            for layer
            in model.layers
            if isinstance(
                layer,
                tf.keras.layers.LSTM,
            )
        ]


        assert len(
            recurrent_layers
        ) == 1


        assert int(
            recurrent_layers[
                0
            ].units
        ) == 32


    else:

        conv_layers = [
            layer
            for layer
            in model.layers
            if isinstance(
                layer,
                tf.keras.layers.Conv1D,
            )
        ]


        assert len(
            conv_layers
        ) == 7


        assert all(
            layer.padding
            ==
            "causal"
            for layer
            in conv_layers
        )


        assert any(
            layer.name
            ==
            "tcn_last_position"
            for layer
            in model.layers
        )


        assert not any(
            isinstance(
                layer,
                tf.keras.layers.GlobalAveragePooling1D,
            )
            for layer
            in model.layers
        )


    # ========================================================
    # G. Final frozen refit
    # ========================================================

    fit_start = (
        time.perf_counter()
    )


    history_object = model.fit(

        x=[
            trainval_sequence,
            trainval_auxiliary,
        ],

        y=
            final_fit_target,

        epochs=
            frozen_epochs,

        batch_size=
            PHASE2_NEURAL_BATCH_SIZE,

        shuffle=True,

        callbacks=None,

        validation_data=None,

        verbose=
            verbose,
    )


    fit_seconds = float(
        time.perf_counter()
        -
        fit_start
    )


    history_df = pd.DataFrame(
        history_object.history
    )


    history_df.insert(
        0,
        "epoch",
        np.arange(
            1,
            len(
                history_df
            )
            +
            1,
            dtype=int,
        ),
    )


    assert len(
        history_df
    ) == frozen_epochs


    assert (
        "val_loss"
        not in
        history_df.columns
    )


    # ========================================================
    # H. Held-out TEST prediction
    # ========================================================

    predict_start = (
        time.perf_counter()
    )


    prediction_scaled = np.asarray(

        model.predict(
            [
                test_sequence,
                test_auxiliary,
            ],
            batch_size=
                PHASE2_NEURAL_BATCH_SIZE,
            verbose=0,
        ),

        dtype=np.float64,

    ).reshape(
        -1
    )


    predict_seconds = float(
        time.perf_counter()
        -
        predict_start
    )


    assert prediction_scaled.shape == (
        n_test,
    )


    assert np.isfinite(
        prediction_scaled
    ).all()


    # ========================================================
    # I. Inverse target scaling
    # ========================================================

    target_mean = float(
        phase2_neural_scaling_statistics[
            horizon_minutes
        ][
            "target_mean"
        ]
    )


    target_std = float(
        phase2_neural_scaling_statistics[
            horizon_minutes
        ][
            "target_std"
        ]
    )


    prediction_kw = (
        target_mean
        +
        target_std
        *
        prediction_scaled
    )


    assert np.isfinite(
        prediction_kw
    ).all()


    # ========================================================
    # J. TEST metadata / daylight
    # ========================================================

    bundle = (
        modelling_bundles[
            horizon_minutes
        ]
    )


    metadata_test = (
        bundle[
            "metadata"
        ]
        .loc[
            bundle[
                "test_mask"
            ]
        ]
        .copy()
    )


    daylight_test = (
        bundle[
            "daylight_mask"
        ]
        .loc[
            bundle[
                "test_mask"
            ]
        ]
        .to_numpy(
            dtype=bool
        )
    )


    assert len(
        metadata_test
    ) == n_test


    assert daylight_test.shape == (
        n_test,
    )


    # ========================================================
    # K. Scope metrics
    # ========================================================

    metric_rows = []


    scope_masks = {

        "all":
            np.ones(
                n_test,
                dtype=bool,
            ),

        "daylight":
            daylight_test,
    }


    for scope_name, scope_mask in (
        scope_masks.items()
    ):

        metrics = phase2_regression_metrics(

            test_target_kw[
                scope_mask
            ],

            prediction_kw[
                scope_mask
            ],
        )


        metric_rows.append(
            {
                "model":
                    model_name,

                "horizon_minutes":
                    horizon_minutes,

                "variant":
                    variant_name,

                "family":
                    PHASE2_COMPONENT_FAMILY[
                        variant_name
                    ],

                "scope":
                    scope_name,

                "n_rows":
                    int(
                        scope_mask.sum()
                    ),

                "auxiliary_dim":
                    auxiliary_dim,

                "trainable_parameter_count":
                    actual_parameters,

                "frozen_epochs":
                    frozen_epochs,

                "seed":
                    PHASE2_NEURAL_SEED,

                "batch_size":
                    PHASE2_NEURAL_BATCH_SIZE,

                "fit_seconds":
                    fit_seconds,

                "predict_seconds":
                    predict_seconds,

                **metrics,
            }
        )


    metrics_df = pd.DataFrame(
        metric_rows
    )


    assert len(
        metrics_df
    ) == 2


    # ========================================================
    # L. Prediction table
    # ========================================================

    prediction_df = pd.DataFrame(
        {
            "issue_time":
                metadata_test[
                    "issue_time"
                ]
                .to_numpy(),

            "valid_time":
                metadata_test[
                    "valid_time"
                ]
                .to_numpy(),

            "daylight":
                daylight_test,

            "target_P_Solar_kW":
                test_target_kw,

            "prediction_kW":
                prediction_kw,

            "prediction_scaled":
                prediction_scaled,
        },
        index=metadata_test.index,
    )


    prediction_df.index.name = (
        "Timestamp"
    )


    return (
        metrics_df,
        history_df,
        prediction_df,
        {
            "model":
                model_name,

            "horizon_minutes":
                horizon_minutes,

            "variant":
                variant_name,

            "family":
                PHASE2_COMPONENT_FAMILY[
                    variant_name
                ],

            "auxiliary_dim":
                auxiliary_dim,

            "parameter_count":
                actual_parameters,

            "frozen_epochs":
                frozen_epochs,

            "seed":
                PHASE2_NEURAL_SEED,

            "batch_size":
                PHASE2_NEURAL_BATCH_SIZE,

            "shuffle":
                True,

            "validation_data_used":
                False,

            "callbacks_used":
                False,

            "prediction_clipping":
                False,

            "completed_successfully":
                True,
        },
    )


# ============================================================
# 8. Build frozen 30-run manifest
# ============================================================

component_neural_manifest_rows = []


for model_name in (
    PHASE2_COMPONENT_NEURAL_MODELS
):

    for horizon_minutes in (
        PHASE2_COMPONENT_NEURAL_HORIZONS
    ):

        for variant_name in (
            PHASE2_COMPONENT_VARIANTS
        ):

            auxiliary_dim = (
                PHASE2_COMPONENT_EXPECTED_AUX_DIMS[
                    variant_name
                ]
            )


            component_neural_manifest_rows.append(
                {
                    "model":
                        model_name,

                    "horizon_minutes":
                        horizon_minutes,

                    "variant":
                        variant_name,

                    "family":
                        PHASE2_COMPONENT_FAMILY[
                            variant_name
                        ],

                    "auxiliary_dim":
                        auxiliary_dim,

                    "expected_parameters":
                        expected_phase2_neural_parameter_count(

                            model_name=
                                model_name,

                            auxiliary_dim=
                                auxiliary_dim,
                        ),

                    "frozen_epochs":
                        int(
                            PHASE2_FROZEN_MODEL_CONFIGS[
                                model_name
                            ][
                                horizon_minutes
                            ][
                                "epochs"
                            ]
                        ),
                }
            )


phase2_component_neural_manifest = (
    pd.DataFrame(
        component_neural_manifest_rows
    )
)


assert len(
    phase2_component_neural_manifest
) == 30


manifest_path = (
    PHASE2_COMPONENT_NEURAL_DIR
    /
    "phase2_component_neural_manifest.csv"
)


component_atomic_csv(
    phase2_component_neural_manifest,
    manifest_path,
    index=False,
)


# ============================================================
# 9. Resume audit
# ============================================================

completed_before = []


pending_before = []


for row in (
    phase2_component_neural_manifest
    .itertuples(
        index=False
    )
):

    key = (
        row.model,
        int(
            row.horizon_minutes
        ),
        row.variant,
    )


    if phase2_component_neural_run_complete(
        *key
    ):

        completed_before.append(
            key
        )

    else:

        pending_before.append(
            key
        )


print(
    "\nResume audit"
)

print(
    "------------"
)

print(
    "Already complete:",
    len(
        completed_before
    )
)

print(
    "Pending:",
    len(
        pending_before
    )
)

print(
    "Total:",
    len(
        completed_before
    )
    +
    len(
        pending_before
    )
)


assert (
    len(
        completed_before
    )
    +
    len(
        pending_before
    )
    ==
    30
)


# ============================================================
# 10. Execute pending fits
# ============================================================

grid_start = (
    time.perf_counter()
)


newly_completed = []


for position, row in enumerate(

    phase2_component_neural_manifest
    .itertuples(
        index=False
    ),

    start=1,
):

    model_name = row.model

    horizon_minutes = int(
        row.horizon_minutes
    )

    variant_name = row.variant


    paths = (
        phase2_component_neural_run_paths(

            model_name,
            horizon_minutes,
            variant_name,
        )
    )


    if phase2_component_neural_run_complete(
        model_name,
        horizon_minutes,
        variant_name,
    ):

        print(
            f"[{position:02d}/30] "
            f"SKIP  "
            f"{model_name:<4} | "
            f"{horizon_minutes:>2} min | "
            f"{variant_name}"
        )

        continue


    paths[
        "run_dir"
    ].mkdir(
        parents=True,
        exist_ok=True,
    )


    if paths[
        "failure"
    ].is_file():

        paths[
            "failure"
        ].unlink()


    print(
        "\n"
        +
        "=" * 94
    )

    print(
        f"[{position:02d}/30] "
        f"RUN   "
        f"{model_name} | "
        f"{horizon_minutes} min | "
        f"{variant_name}"
    )

    print(
        "=" * 94
    )


    run_start = (
        time.perf_counter()
    )


    try:

        (
            metrics_df,
            history_df,
            prediction_df,
            run_metadata,
        ) = (
            run_phase2_neural_component_refit(

                model_name=
                    model_name,

                horizon_minutes=
                    horizon_minutes,

                variant_name=
                    variant_name,

                verbose=0,
            )
        )


        # ----------------------------------------------------
        # Validation before persistence
        # ----------------------------------------------------

        assert len(
            metrics_df
        ) == 2


        assert set(
            metrics_df[
                "scope"
            ]
        ) == {
            "all",
            "daylight",
        }


        assert (
            int(
                metrics_df[
                    "auxiliary_dim"
                ]
                .iloc[
                    0
                ]
            )
            ==
            int(
                row.auxiliary_dim
            )
        )


        assert (
            int(
                metrics_df[
                    "trainable_parameter_count"
                ]
                .iloc[
                    0
                ]
            )
            ==
            int(
                row.expected_parameters
            )
        )


        assert len(
            prediction_df
        ) == 26_496


        assert np.isfinite(
            metrics_df[
                [
                    "MAE_kW",
                    "RMSE_kW",
                    "nMAE_percent",
                    "nRMSE_percent",
                    "R2",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        ).all()


        # ----------------------------------------------------
        # Persist atomically
        # ----------------------------------------------------

        component_atomic_csv(
            metrics_df,
            paths[
                "metrics"
            ],
            index=False,
        )


        component_atomic_csv(
            history_df,
            paths[
                "history"
            ],
            index=False,
        )


        component_atomic_csv(
            prediction_df,
            paths[
                "predictions"
            ],
            index=True,
        )


        run_metadata[
            "tensorflow_version"
        ] = str(
            tf.__version__
        )


        component_atomic_json(
            run_metadata,
            paths[
                "metadata"
            ],
        )


        assert phase2_component_neural_run_complete(

            model_name,
            horizon_minutes,
            variant_name,
        )


        newly_completed.append(
            (
                model_name,
                horizon_minutes,
                variant_name,
            )
        )


        all_mae = float(
            metrics_df.loc[
                metrics_df[
                    "scope"
                ].eq(
                    "all"
                ),
                "MAE_kW",
            ]
            .iloc[
                0
            ]
        )


        daylight_mae = float(
            metrics_df.loc[
                metrics_df[
                    "scope"
                ].eq(
                    "daylight"
                ),
                "MAE_kW",
            ]
            .iloc[
                0
            ]
        )


        run_seconds = float(
            time.perf_counter()
            -
            run_start
        )


        print(
            f"PASS | "
            f"MAE={all_mae:.6f} kW | "
            f"Daylight MAE={daylight_mae:.6f} kW | "
            f"epochs={int(row.frozen_epochs)} | "
            f"time={run_seconds:.1f}s"
        )


    except Exception as exc:

        failure_record = {

            "model":
                model_name,

            "horizon_minutes":
                horizon_minutes,

            "variant":
                variant_name,

            "exception_type":
                type(
                    exc
                ).__name__,

            "exception_message":
                str(
                    exc
                ),

            "completed_successfully":
                False,
        }


        component_atomic_json(
            failure_record,
            paths[
                "failure"
            ],
        )


        print(
            "\nFAILED RUN:"
        )

        print(
            failure_record
        )


        raise


# ============================================================
# 11. Recover all persisted results
# ============================================================

metric_frames = []


completed_keys = []

missing_keys = []


for row in (
    phase2_component_neural_manifest
    .itertuples(
        index=False
    )
):

    key = (
        row.model,
        int(
            row.horizon_minutes
        ),
        row.variant,
    )


    paths = (
        phase2_component_neural_run_paths(
            *key
        )
    )


    if phase2_component_neural_run_complete(
        *key
    ):

        metric_frames.append(
            pd.read_csv(
                paths[
                    "metrics"
                ]
            )
        )


        completed_keys.append(
            key
        )


    else:

        missing_keys.append(
            key
        )


if metric_frames:

    phase2_component_neural_results = (
        pd.concat(
            metric_frames,
            ignore_index=True,
        )
    )

else:

    phase2_component_neural_results = (
        pd.DataFrame()
    )


# ============================================================
# 12. Save partial aggregate
# ============================================================

partial_results_path = (
    PHASE2_COMPONENT_NEURAL_DIR
    /
    "phase2_component_neural_results_partial.csv"
)


component_atomic_csv(
    phase2_component_neural_results,
    partial_results_path,
    index=False,
)


# ============================================================
# 13. Status
# ============================================================

grid_seconds = float(
    time.perf_counter()
    -
    grid_start
)


print(
    "\n"
    +
    "=" * 94
)

print(
    "PHASE 2A NEURAL COMPONENT GRID STATUS"
)

print(
    "=" * 94
)


print(
    "Completed:",
    len(
        completed_keys
    ),
    "/ 30",
)


print(
    "Remaining:",
    len(
        missing_keys
    ),
)


print(
    "Newly completed this execution:",
    len(
        newly_completed
    ),
)


print(
    "Runtime:",
    f"{grid_seconds:.1f} seconds",
)


# ============================================================
# 14. Final aggregate if complete
# ============================================================

if len(
    missing_keys
) == 0:

    assert len(
        phase2_component_neural_results
    ) == 60


    assert (
        phase2_component_neural_results[
            [
                "model",
                "horizon_minutes",
                "variant",
                "scope",
            ]
        ]
        .duplicated()
        .sum()
        ==
        0
    )


    final_results_path = (
        PHASE2_COMPONENT_NEURAL_DIR
        /
        "phase2_component_neural_results.csv"
    )


    component_atomic_csv(
        phase2_component_neural_results,
        final_results_path,
        index=False,
    )


    # --------------------------------------------------------
    # All-row MAE pivot
    # --------------------------------------------------------

    component_neural_all_mae = (
        phase2_component_neural_results
        .loc[
            phase2_component_neural_results[
                "scope"
            ].eq(
                "all"
            )
        ]
        .pivot_table(
            index=[
                "model",
                "horizon_minutes",
            ],
            columns="variant",
            values="MAE_kW",
            aggfunc="first",
        )
        .reset_index()
    )


    # --------------------------------------------------------
    # Daylight MAE pivot
    # --------------------------------------------------------

    component_neural_daylight_mae = (
        phase2_component_neural_results
        .loc[
            phase2_component_neural_results[
                "scope"
            ].eq(
                "daylight"
            )
        ]
        .pivot_table(
            index=[
                "model",
                "horizon_minutes",
            ],
            columns="variant",
            values="MAE_kW",
            aggfunc="first",
        )
        .reset_index()
    )


    component_atomic_csv(

        component_neural_all_mae,

        PHASE2_COMPONENT_NEURAL_DIR
        /
        "phase2_component_neural_all_mae.csv",

        index=False,
    )


    component_atomic_csv(

        component_neural_daylight_mae,

        PHASE2_COMPONENT_NEURAL_DIR
        /
        "phase2_component_neural_daylight_mae.csv",

        index=False,
    )


    print(
        "\nPHASE 2A NEURAL ADD-ONE COMPONENT SCREEN: PASSED"
    )


    print(
        "\nAll-row MAE:"
    )

    display(
        component_neural_all_mae
    )


    print(
        "\nDaylight MAE:"
    )

    display(
        component_neural_daylight_mae
    )


    print(
        "\nSaved:"
    )

    print(
        " ",
        final_results_path
    )


else:

    print(
        "\nGRID NOT YET COMPLETE."
    )

    print(
        "Re-run this same P2-26C cell to resume."
    )


    print(
        "\nRemaining:"
    )


    for key in missing_keys:
        print(
            " ",
            key
        )

Component neural representations: PASSED
Neural component output directory:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/component_screen/neural

Resume audit
------------
Already complete: 0
Pending: 30
Total: 30

[01/30] RUN   LSTM | 15 min | F3_plus_tithi


2026-09-29 23:14:38.937694: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-09-29 23:14:38.938875: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

PASS | MAE=0.298790 kW | Daylight MAE=0.425635 kW | epochs=7 | time=38.4s

[02/30] RUN   LSTM | 15 min | F3_plus_paksha


2026-09-29 23:15:50.257743: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-09-29 23:15:50.258005: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

PASS | MAE=0.348072 kW | Daylight MAE=0.454488 kW | epochs=7 | time=35.1s

[03/30] RUN   LSTM | 15 min | F3_plus_karana


...

PASS | MAE=0.785343 kW | Daylight MAE=0.968875 kW | epochs=1 | time=19.8s

PHASE 2A NEURAL COMPONENT GRID STATUS
Completed: 30 / 30
Remaining: 0
Newly completed this execution: 30
Runtime: 778.8 seconds

PHASE 2A NEURAL ADD-ONE COMPONENT SCREEN: PASSED

All-row MAE:


variant,model,horizon_minutes,F3_plus_karana,F3_plus_nakshatra,F3_plus_paksha,F3_plus_tithi,F3_plus_yoga
0,LSTM,15,0.302603,0.367337,0.348072,0.298790,0.315341
1,LSTM,30,0.384198,0.381044,0.376148,0.399899,0.402761
2,LSTM,60,0.474653,0.471044,0.465534,0.482613,0.479021
3,TCN,15,0.455357,0.419485,0.403566,0.307228,0.444609
4,TCN,30,0.498494,0.442978,0.488057,0.647101,0.937696
5,TCN,60,0.534752,0.566586,0.621392,0.746923,0.785343



Daylight MAE:


variant,model,horizon_minutes,F3_plus_karana,F3_plus_nakshatra,F3_plus_paksha,F3_plus_tithi,F3_plus_yoga
0,LSTM,15,0.427115,0.453515,0.454488,0.425635,0.432505
1,LSTM,30,0.519457,0.533489,0.532342,0.538735,0.533623
2,LSTM,60,0.660600,0.649792,0.653766,0.675075,0.654048
3,TCN,15,0.563342,0.569089,0.548156,0.422150,0.609941
4,TCN,30,0.624214,0.609508,0.618160,0.805213,0.983705
5,TCN,60,0.698570,0.747248,0.775991,0.907252,0.968875



Saved:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/component_screen/neural/phase2_component_neural_results.csv


In [137]:
# ============================================================
# Step P2-27:
# FOUR-MODEL add-one component paired UTC-day bootstrap
#
# Models:
#   Ridge
#   LightGBM
#   LSTM
#   TCN
#
# Horizons:
#   15, 30, 60 minutes
#
# Scopes:
#   all
#   daylight
#
# Components:
#   Tithi
#   Paksha
#   Karana
#   Nakshatra
#   Yoga
#
# Total comparisons:
#
#   4 models
#   × 3 horizons
#   × 2 scopes
#   × 5 components
#   = 120
#
# Contrast:
#
#   delta_MAE =
#       MAE_F3
#       -
#       MAE_F3_plus_component
#
# Positive:
#   add-one component has LOWER MAE than F3.
#
# Negative:
#   F3 has LOWER MAE.
#
# NO MODEL TRAINING occurs here.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Frozen protocol
# ============================================================

PHASE2_COMPONENT_BOOTSTRAP_REPLICATES = 10_000
PHASE2_COMPONENT_BOOTSTRAP_SEED = 42


PHASE2_COMPONENT_BOOTSTRAP_MODELS = [
    "Ridge",
    "LightGBM",
    "LSTM",
    "TCN",
]


PHASE2_COMPONENT_BOOTSTRAP_HORIZONS = [
    15,
    30,
    60,
]


PHASE2_COMPONENT_BOOTSTRAP_SCOPES = [
    "all",
    "daylight",
]


if "PHASE2_COMPONENT_VARIANTS" not in globals():

    PHASE2_COMPONENT_VARIANTS = [
        "F3_plus_tithi",
        "F3_plus_paksha",
        "F3_plus_karana",
        "F3_plus_nakshatra",
        "F3_plus_yoga",
    ]


if "PHASE2_COMPONENT_FAMILY" not in globals():

    PHASE2_COMPONENT_FAMILY = {
        "F3_plus_tithi":
            "tithi",

        "F3_plus_paksha":
            "paksha",

        "F3_plus_karana":
            "karana",

        "F3_plus_nakshatra":
            "nakshatra",

        "F3_plus_yoga":
            "yoga",
    }


assert (
    4
    * 3
    * 2
    * 5
) == 120


# ============================================================
# 2. Restart-safe directory recovery
# ============================================================

if "PHASE2_COMPONENT_DIR" not in globals():

    PHASE2_COMPONENT_DIR = (
        PHASE2_RESULTS_DIR
        /
        "component_screen"
    )


if "PHASE2_COMPONENT_CLASSICAL_DIR" not in globals():

    PHASE2_COMPONENT_CLASSICAL_DIR = (
        PHASE2_COMPONENT_DIR
        /
        "classical"
    )


if "PHASE2_COMPONENT_NEURAL_DIR" not in globals():

    PHASE2_COMPONENT_NEURAL_DIR = (
        PHASE2_COMPONENT_DIR
        /
        "neural"
    )


if "PHASE2_NEURAL_GRID_DIR" not in globals():

    PHASE2_NEURAL_GRID_DIR = (
        PHASE2_RESULTS_DIR
        /
        "neural_core_grid_v1"
    )


# ============================================================
# 3. Robust prediction-table parser
# ============================================================

def prepare_component_prediction_table(
    table,
):

    table = (
        table
        .copy()
        .reset_index(
            drop=True
        )
    )


    table[
        "issue_time"
    ] = pd.to_datetime(
        table[
            "issue_time"
        ],
        utc=True,
    )


    table[
        "valid_time"
    ] = pd.to_datetime(
        table[
            "valid_time"
        ],
        utc=True,
    )


    if (
        table[
            "daylight"
        ].dtype
        != bool
    ):

        table[
            "daylight"
        ] = (
            table[
                "daylight"
            ]
            .astype(
                str
            )
            .str
            .lower()
            .map(
                {
                    "true": True,
                    "false": False,
                    "1": True,
                    "0": False,
                }
            )
        )


    assert (
        table[
            "daylight"
        ]
        .notna()
        .all()
    )


    table[
        "daylight"
    ] = (
        table[
            "daylight"
        ]
        .astype(
            bool
        )
    )


    assert len(
        table
    ) == 26_496


    assert np.isfinite(
        table[
            "target_P_Solar_kW"
        ]
        .to_numpy(
            dtype=float
        )
    ).all()


    return table


# ============================================================
# 4. Recover classical F3 core predictions
# ============================================================

classical_core_predictions = {}


for horizon_minutes in (
    PHASE2_COMPONENT_BOOTSTRAP_HORIZONS
):

    path = (
        PHASE2_RESULTS_DIR
        /
        (
            "phase2_stage1_classical_core_"
            f"predictions_{horizon_minutes}min.csv"
        )
    )


    assert path.is_file(), (
        f"Missing classical core prediction file: "
        f"{path}"
    )


    classical_core_predictions[
        horizon_minutes
    ] = (
        prepare_component_prediction_table(
            pd.read_csv(
                path,
                index_col=0,
            )
        )
    )


print(
    "Classical F3 prediction recovery: PASSED"
)


# ============================================================
# 5. Recover classical component predictions
# ============================================================

classical_component_predictions = {}


for horizon_minutes in (
    PHASE2_COMPONENT_BOOTSTRAP_HORIZONS
):

    path = (
        PHASE2_COMPONENT_CLASSICAL_DIR
        /
        (
            "phase2_component_classical_"
            f"predictions_{horizon_minutes}min.csv"
        )
    )


    assert path.is_file(), (
        f"Missing classical component prediction file: "
        f"{path}"
    )


    classical_component_predictions[
        horizon_minutes
    ] = (
        prepare_component_prediction_table(
            pd.read_csv(
                path,
                index_col=0,
            )
        )
    )


print(
    "Classical component prediction recovery: PASSED"
)


# ============================================================
# 6. Neural prediction loader
# ============================================================

def load_neural_prediction_file(
    path,
):

    assert path.is_file(), (
        f"Missing neural prediction file: {path}"
    )


    return prepare_component_prediction_table(
        pd.read_csv(
            path,
            index_col=0,
        )
    )


def neural_core_f3_path(
    model_name,
    horizon_minutes,
):

    run_id = (
        f"{model_name.lower()}"
        f"__{int(horizon_minutes)}min"
        "__F3"
    )


    return (
        PHASE2_NEURAL_GRID_DIR
        /
        run_id
        /
        "predictions.csv"
    )


def neural_component_path(
    model_name,
    horizon_minutes,
    variant_name,
):

    run_id = (
        f"{model_name.lower()}"
        f"__{int(horizon_minutes)}min"
        f"__{variant_name}"
    )


    return (
        PHASE2_COMPONENT_NEURAL_DIR
        /
        run_id
        /
        "predictions.csv"
    )


# ============================================================
# 7. Canonical TEST-day population
# ============================================================

reference_table = (
    classical_core_predictions[
        15
    ]
)


canonical_test_days = pd.DatetimeIndex(
    reference_table[
        "valid_time"
    ]
    .dt
    .floor(
        "D"
    )
    .unique()
).sort_values()


assert len(
    canonical_test_days
) == 92


assert (
    canonical_test_days[
        0
    ]
    ==
    pd.Timestamp(
        "2019-06-01 00:00:00+00:00"
    )
)


assert (
    canonical_test_days[
        -1
    ]
    ==
    pd.Timestamp(
        "2019-08-31 00:00:00+00:00"
    )
)


print(
    "Canonical TEST-day population: PASSED"
)


# ============================================================
# 8. Bootstrap draws
#
# Preserve the already-used model-family policies:
#
# Classical:
#   SeedSequence([42, horizon, scope_index])
#
# Neural:
#   one canonical RNG(seed=42) 92-day matrix
#
# This makes component CIs directly comparable with the
# earlier core bootstrap for the corresponding model family.
# ============================================================

classical_component_draws = {}


for horizon_minutes in (
    PHASE2_COMPONENT_BOOTSTRAP_HORIZONS
):

    for scope_index, scope in enumerate(
        PHASE2_COMPONENT_BOOTSTRAP_SCOPES
    ):

        seed_sequence = (
            np.random.SeedSequence(
                [
                    PHASE2_COMPONENT_BOOTSTRAP_SEED,
                    int(
                        horizon_minutes
                    ),
                    int(
                        scope_index
                    ),
                ]
            )
        )


        rng = np.random.default_rng(
            seed_sequence
        )


        classical_component_draws[
            (
                horizon_minutes,
                scope,
            )
        ] = rng.integers(
            low=0,
            high=92,
            size=(
                PHASE2_COMPONENT_BOOTSTRAP_REPLICATES,
                92,
            ),
            endpoint=False,
        )


neural_component_rng = (
    np.random.default_rng(
        PHASE2_COMPONENT_BOOTSTRAP_SEED
    )
)


neural_component_draws = (
    neural_component_rng.integers(
        low=0,
        high=92,
        size=(
            PHASE2_COMPONENT_BOOTSTRAP_REPLICATES,
            92,
        ),
        dtype=np.int16,
    )
)


assert neural_component_draws.shape == (
    10_000,
    92,
)


print(
    "Bootstrap draw construction: PASSED"
)


# ============================================================
# 9. Pair-alignment audit
# ============================================================

def assert_component_pair_alignment(
    baseline_table,
    component_table,
):

    assert len(
        baseline_table
    ) == len(
        component_table
    ) == 26_496


    assert np.array_equal(
        baseline_table[
            "issue_time"
        ]
        .to_numpy(),

        component_table[
            "issue_time"
        ]
        .to_numpy(),
    )


    assert np.array_equal(
        baseline_table[
            "valid_time"
        ]
        .to_numpy(),

        component_table[
            "valid_time"
        ]
        .to_numpy(),
    )


    assert np.array_equal(
        baseline_table[
            "daylight"
        ]
        .to_numpy(
            dtype=bool
        ),

        component_table[
            "daylight"
        ]
        .to_numpy(
            dtype=bool
        ),
    )


    assert np.allclose(
        baseline_table[
            "target_P_Solar_kW"
        ]
        .to_numpy(
            dtype=float
        ),

        component_table[
            "target_P_Solar_kW"
        ]
        .to_numpy(
            dtype=float
        ),

        atol=1e-12,
        rtol=0.0,
    )


# ============================================================
# 10. Paired component-vs-F3 bootstrap helper
# ============================================================

def paired_component_vs_f3_bootstrap(
    valid_time,
    daylight,
    y_true,
    f3_prediction,
    component_prediction,
    scope,
    bootstrap_draws,
):

    assert scope in {
        "all",
        "daylight",
    }


    valid_time = pd.Series(
        pd.to_datetime(
            valid_time,
            utc=True,
        )
    )


    daylight = np.asarray(
        daylight,
        dtype=bool,
    )


    y_true = np.asarray(
        y_true,
        dtype=np.float64,
    )


    f3_prediction = np.asarray(
        f3_prediction,
        dtype=np.float64,
    )


    component_prediction = np.asarray(
        component_prediction,
        dtype=np.float64,
    )


    if scope == "all":

        scope_mask = np.ones(
            len(
                y_true
            ),
            dtype=bool,
        )

    else:

        scope_mask = daylight


    scoped_time = (
        valid_time[
            scope_mask
        ]
        .reset_index(
            drop=True
        )
    )


    scoped_true = (
        y_true[
            scope_mask
        ]
    )


    scoped_f3 = (
        f3_prediction[
            scope_mask
        ]
    )


    scoped_component = (
        component_prediction[
            scope_mask
        ]
    )


    f3_absolute_error = np.abs(
        scoped_true
        -
        scoped_f3
    )


    component_absolute_error = np.abs(
        scoped_true
        -
        scoped_component
    )


    working = pd.DataFrame(
        {
            "utc_day":
                scoped_time
                .dt
                .floor(
                    "D"
                ),

            "F3_absolute_error":
                f3_absolute_error,

            "component_absolute_error":
                component_absolute_error,
        }
    )


    daily = (
        working
        .groupby(
            "utc_day",
            sort=True,
        )
        .agg(
            F3_error_sum=(
                "F3_absolute_error",
                "sum",
            ),

            component_error_sum=(
                "component_absolute_error",
                "sum",
            ),

            row_count=(
                "F3_absolute_error",
                "size",
            ),
        )
        .reindex(
            canonical_test_days
        )
    )


    assert len(
        daily
    ) == 92


    assert (
        daily[
            "row_count"
        ]
        .notna()
        .all()
    )


    assert (
        daily[
            "row_count"
        ]
        >
        0
    ).all()


    counts = (
        daily[
            "row_count"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    f3_daily_error = (
        daily[
            "F3_error_sum"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    component_daily_error = (
        daily[
            "component_error_sum"
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    # --------------------------------------------------------
    # Point estimates
    # --------------------------------------------------------

    F3_mae = float(
        f3_absolute_error.mean()
    )


    component_mae = float(
        component_absolute_error.mean()
    )


    # Positive = component improves over F3.
    point_delta = float(
        F3_mae
        -
        component_mae
    )


    # --------------------------------------------------------
    # Paired UTC-day bootstrap
    # --------------------------------------------------------

    sampled_counts = (
        counts[
            bootstrap_draws
        ]
        .sum(
            axis=1
        )
    )


    sampled_f3_mae = (
        f3_daily_error[
            bootstrap_draws
        ]
        .sum(
            axis=1
        )
        /
        sampled_counts
    )


    sampled_component_mae = (
        component_daily_error[
            bootstrap_draws
        ]
        .sum(
            axis=1
        )
        /
        sampled_counts
    )


    bootstrap_delta = (
        sampled_f3_mae
        -
        sampled_component_mae
    )


    assert bootstrap_delta.shape == (
        10_000,
    )


    assert np.isfinite(
        bootstrap_delta
    ).all()


    ci_low = float(
        np.percentile(
            bootstrap_delta,
            2.5,
        )
    )


    ci_high = float(
        np.percentile(
            bootstrap_delta,
            97.5,
        )
    )


    if ci_low > 0.0:

        evidence_class = (
            "component_lower_MAE"
        )

    elif ci_high < 0.0:

        evidence_class = (
            "F3_lower_MAE"
        )

    else:

        evidence_class = (
            "CI_crosses_zero"
        )


    relative_reduction_percent = float(
        100.0
        *
        point_delta
        /
        F3_mae
    )


    return {

        "n_rows":
            int(
                scope_mask.sum()
            ),

        "n_utc_days":
            92,

        "F3_MAE_kW":
            F3_mae,

        "component_MAE_kW":
            component_mae,

        "delta_MAE_F3_minus_component_kW":
            point_delta,

        "component_MAE_reduction_vs_F3_percent":
            relative_reduction_percent,

        "ci95_low_kW":
            ci_low,

        "ci95_high_kW":
            ci_high,

        "evidence_class":
            evidence_class,

        "bootstrap_replicates":
            PHASE2_COMPONENT_BOOTSTRAP_REPLICATES,

        "bootstrap_seed":
            PHASE2_COMPONENT_BOOTSTRAP_SEED,
    }


# ============================================================
# 11. Execute classical comparisons
#
# 2 models × 3 horizons × 2 scopes × 5 families
# = 60 comparisons
# ============================================================

component_bootstrap_rows = []


for model_name in [
    "Ridge",
    "LightGBM",
]:

    for horizon_minutes in (
        PHASE2_COMPONENT_BOOTSTRAP_HORIZONS
    ):

        core_table = (
            classical_core_predictions[
                horizon_minutes
            ]
        )


        component_table = (
            classical_component_predictions[
                horizon_minutes
            ]
        )


        assert_component_pair_alignment(
            core_table,
            component_table,
        )


        f3_prediction_column = (
            f"{model_name}__F3"
        )


        assert (
            f3_prediction_column
            in
            core_table.columns
        )


        for variant_name in (
            PHASE2_COMPONENT_VARIANTS
        ):

            component_prediction_column = (
                f"{model_name}"
                f"__{variant_name}"
            )


            assert (
                component_prediction_column
                in
                component_table.columns
            )


            for scope in (
                PHASE2_COMPONENT_BOOTSTRAP_SCOPES
            ):

                result = (
                    paired_component_vs_f3_bootstrap(

                        valid_time=
                            core_table[
                                "valid_time"
                            ],

                        daylight=
                            core_table[
                                "daylight"
                            ],

                        y_true=
                            core_table[
                                "target_P_Solar_kW"
                            ],

                        f3_prediction=
                            core_table[
                                f3_prediction_column
                            ],

                        component_prediction=
                            component_table[
                                component_prediction_column
                            ],

                        scope=
                            scope,

                        bootstrap_draws=
                            classical_component_draws[
                                (
                                    horizon_minutes,
                                    scope,
                                )
                            ],
                    )
                )


                component_bootstrap_rows.append(
                    {
                        "model":
                            model_name,

                        "model_family":
                            "classical",

                        "horizon_minutes":
                            horizon_minutes,

                        "scope":
                            scope,

                        "variant":
                            variant_name,

                        "component_family":
                            PHASE2_COMPONENT_FAMILY[
                                variant_name
                            ],

                        "bootstrap_draw_policy":
                            (
                                "SeedSequence("
                                "[42,horizon,scope_index])"
                            ),

                        **result,
                    }
                )


print(
    "Classical component bootstrap: PASSED"
)


# ============================================================
# 12. Execute neural comparisons
#
# 2 models × 3 horizons × 2 scopes × 5 families
# = 60 comparisons
# ============================================================

for model_name in [
    "LSTM",
    "TCN",
]:

    for horizon_minutes in (
        PHASE2_COMPONENT_BOOTSTRAP_HORIZONS
    ):

        f3_table = (
            load_neural_prediction_file(
                neural_core_f3_path(
                    model_name,
                    horizon_minutes,
                )
            )
        )


        for variant_name in (
            PHASE2_COMPONENT_VARIANTS
        ):

            component_table = (
                load_neural_prediction_file(
                    neural_component_path(
                        model_name,
                        horizon_minutes,
                        variant_name,
                    )
                )
            )


            assert_component_pair_alignment(
                f3_table,
                component_table,
            )


            for scope in (
                PHASE2_COMPONENT_BOOTSTRAP_SCOPES
            ):

                result = (
                    paired_component_vs_f3_bootstrap(

                        valid_time=
                            f3_table[
                                "valid_time"
                            ],

                        daylight=
                            f3_table[
                                "daylight"
                            ],

                        y_true=
                            f3_table[
                                "target_P_Solar_kW"
                            ],

                        f3_prediction=
                            f3_table[
                                "prediction_kW"
                            ],

                        component_prediction=
                            component_table[
                                "prediction_kW"
                            ],

                        scope=
                            scope,

                        bootstrap_draws=
                            neural_component_draws,
                    )
                )


                component_bootstrap_rows.append(
                    {
                        "model":
                            model_name,

                        "model_family":
                            "neural",

                        "horizon_minutes":
                            horizon_minutes,

                        "scope":
                            scope,

                        "variant":
                            variant_name,

                        "component_family":
                            PHASE2_COMPONENT_FAMILY[
                                variant_name
                            ],

                        "bootstrap_draw_policy":
                            (
                                "single canonical "
                                "92-day draw matrix"
                            ),

                        **result,
                    }
                )


print(
    "Neural component bootstrap: PASSED"
)


# ============================================================
# 13. Unified 120-row table
# ============================================================

phase2_component_bootstrap_summary = (
    pd.DataFrame(
        component_bootstrap_rows
    )
    .sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
            "component_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    phase2_component_bootstrap_summary
) == 120


assert (
    phase2_component_bootstrap_summary[
        [
            "model",
            "horizon_minutes",
            "scope",
            "variant",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
)


assert set(
    phase2_component_bootstrap_summary[
        "model"
    ]
) == {
    "Ridge",
    "LightGBM",
    "LSTM",
    "TCN",
}


assert set(
    phase2_component_bootstrap_summary[
        "component_family"
    ]
) == {
    "tithi",
    "paksha",
    "karana",
    "nakshatra",
    "yoga",
}


assert (
    phase2_component_bootstrap_summary[
        "n_utc_days"
    ]
    .eq(
        92
    )
    .all()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "PHASE 2A FOUR-MODEL COMPONENT BOOTSTRAP: PASSED"
)

print(
    "=" * 100
)


print(
    "Comparisons:",
    len(
        phase2_component_bootstrap_summary
    )
)

print(
    "Bootstrap replicates:",
    PHASE2_COMPONENT_BOOTSTRAP_REPLICATES
)

print(
    "UTC day blocks:",
    92
)


# ============================================================
# 14. Evidence counts by MODEL × COMPONENT FAMILY
#
# Each row represents:
#
#   3 horizons × 2 scopes = 6 comparisons
# ============================================================

component_evidence_counts = (
    phase2_component_bootstrap_summary
    .groupby(
        [
            "model",
            "component_family",
            "evidence_class",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=[
            "component_lower_MAE",
            "F3_lower_MAE",
            "CI_crosses_zero",
        ],
        fill_value=0,
    )
    .reset_index()
)


assert (
    (
        component_evidence_counts[
            "component_lower_MAE"
        ]
        +
        component_evidence_counts[
            "F3_lower_MAE"
        ]
        +
        component_evidence_counts[
            "CI_crosses_zero"
        ]
    )
    .eq(
        6
    )
    .all()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "COMPONENT EVIDENCE COUNTS BY MODEL × FAMILY"
)

print(
    "Each row contains 6 horizon/scope comparisons."
)

print(
    "=" * 100
)


display(
    component_evidence_counts
)


# ============================================================
# 15. All-row point-delta pivot
#
# Positive:
#   component better than F3
# ============================================================

component_all_delta_pivot = (
    phase2_component_bootstrap_summary
    .loc[
        phase2_component_bootstrap_summary[
            "scope"
        ].eq(
            "all"
        )
    ]
    .pivot_table(
        index=[
            "model",
            "horizon_minutes",
        ],
        columns="component_family",
        values="delta_MAE_F3_minus_component_kW",
        aggfunc="first",
    )
    .reset_index()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "ALL-ROW COMPONENT ΔMAE"
)

print(
    "Positive = add-one component lower MAE than F3"
)

print(
    "=" * 100
)


display(
    component_all_delta_pivot
)


# ============================================================
# 16. Daylight point-delta pivot
# ============================================================

component_daylight_delta_pivot = (
    phase2_component_bootstrap_summary
    .loc[
        phase2_component_bootstrap_summary[
            "scope"
        ].eq(
            "daylight"
        )
    ]
    .pivot_table(
        index=[
            "model",
            "horizon_minutes",
        ],
        columns="component_family",
        values="delta_MAE_F3_minus_component_kW",
        aggfunc="first",
    )
    .reset_index()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "DAYLIGHT COMPONENT ΔMAE"
)

print(
    "Positive = add-one component lower MAE than F3"
)

print(
    "=" * 100
)


display(
    component_daylight_delta_pivot
)


# ============================================================
# 17. Resolved comparisons only
# ============================================================

resolved_component_results = (
    phase2_component_bootstrap_summary
    .loc[
        ~phase2_component_bootstrap_summary[
            "evidence_class"
        ].eq(
            "CI_crosses_zero"
        )
    ]
    .copy()
    .sort_values(
        [
            "model",
            "horizon_minutes",
            "scope",
            "component_family",
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nResolved comparisons:",
    len(
        resolved_component_results
    ),
    "/ 120",
)


# ============================================================
# 18. Save all outputs
# ============================================================

component_bootstrap_path = (
    PHASE2_COMPONENT_DIR
    /
    "phase2_component_four_model_bootstrap.csv"
)


component_counts_path = (
    PHASE2_COMPONENT_DIR
    /
    "phase2_component_evidence_counts.csv"
)


component_all_delta_path = (
    PHASE2_COMPONENT_DIR
    /
    "phase2_component_allrow_delta_mae.csv"
)


component_daylight_delta_path = (
    PHASE2_COMPONENT_DIR
    /
    "phase2_component_daylight_delta_mae.csv"
)


component_resolved_path = (
    PHASE2_COMPONENT_DIR
    /
    "phase2_component_resolved_comparisons.csv"
)


phase2_component_bootstrap_summary.to_csv(
    component_bootstrap_path,
    index=False,
)


component_evidence_counts.to_csv(
    component_counts_path,
    index=False,
)


component_all_delta_pivot.to_csv(
    component_all_delta_path,
    index=False,
)


component_daylight_delta_pivot.to_csv(
    component_daylight_delta_path,
    index=False,
)


resolved_component_results.to_csv(
    component_resolved_path,
    index=False,
)


print(
    "\nSaved:"
)

print(
    " ",
    component_bootstrap_path
)


print(
    "\nSTOP HERE."
)

print(
    "No model was trained."
)

Classical F3 prediction recovery: PASSED
Classical component prediction recovery: PASSED
Canonical TEST-day population: PASSED
Bootstrap draw construction: PASSED
Classical component bootstrap: PASSED
Neural component bootstrap: PASSED

PHASE 2A FOUR-MODEL COMPONENT BOOTSTRAP: PASSED
Comparisons: 120
Bootstrap replicates: 10000
UTC day blocks: 92

COMPONENT EVIDENCE COUNTS BY MODEL × FAMILY
Each row contains 6 horizon/scope comparisons.


evidence_class,model,component_family,component_lower_MAE,F3_lower_MAE,CI_crosses_zero
0,LSTM,karana,3,0,3
1,LSTM,nakshatra,1,2,3
2,LSTM,paksha,1,2,3
3,LSTM,tithi,2,4,0
4,LSTM,yoga,2,3,1
5,LightGBM,karana,1,2,3
6,LightGBM,nakshatra,0,0,6
7,LightGBM,paksha,0,2,4
8,LightGBM,tithi,0,2,4
9,LightGBM,yoga,0,0,6



ALL-ROW COMPONENT ΔMAE
Positive = add-one component lower MAE than F3


component_family,model,horizon_minutes,karana,nakshatra,paksha,tithi,yoga
0,LSTM,15,0.023189,-0.041545,-0.022279,0.027002,0.010451
1,LSTM,30,0.001664,0.004818,0.009714,-0.014038,-0.016899
2,LSTM,60,-0.005846,-0.002237,0.003273,-0.013806,-0.010214
3,LightGBM,15,-0.000547,-0.000419,-0.001692,-0.001231,-0.000061
4,LightGBM,30,0.001274,0.000228,0.000781,0.000955,0.000710
5,LightGBM,60,-0.003617,-0.000010,-0.000384,-0.000009,-0.000936
6,Ridge,15,-0.000180,-0.000665,-0.000087,-0.002852,-0.007919
7,Ridge,30,-0.000210,-0.001307,-0.000136,-0.004964,-0.014076
8,Ridge,60,-0.000284,-0.002547,-0.000268,-0.009152,-0.025941
9,TCN,15,-0.016835,0.019037,0.034956,0.131294,-0.006087



DAYLIGHT COMPONENT ΔMAE
Positive = add-one component lower MAE than F3


component_family,model,horizon_minutes,karana,nakshatra,paksha,tithi,yoga
0,LSTM,15,0.023938,-0.002462,-0.003435,0.025418,0.018548
1,LSTM,30,0.006407,-0.007625,-0.006478,-0.012872,-0.007759
2,LSTM,60,-0.008961,0.001847,-0.002127,-0.023436,-0.002410
3,LightGBM,15,-0.000914,-0.000386,-0.002350,-0.001833,0.000097
4,LightGBM,30,0.001716,0.000161,0.001299,0.001533,0.001018
5,LightGBM,60,-0.004286,0.000209,-0.000132,0.000217,-0.000935
6,Ridge,15,-0.000172,-0.000255,-0.000007,-0.001698,-0.004860
7,Ridge,30,-0.000201,-0.000641,-0.000036,-0.003056,-0.009183
8,Ridge,60,-0.000212,-0.001492,-0.000182,-0.005922,-0.018414
9,TCN,15,-0.023878,-0.029625,-0.008693,0.117314,-0.070477



Resolved comparisons: 61 / 120

Saved:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/component_screen/phase2_component_four_model_bootstrap.csv

STOP HERE.
No model was trained.


In [138]:
# ============================================================
# Step P2-28:
# Full F4 vs add-one component mechanism synthesis
#
# NO MODEL TRAINING.
#
# For every:
#   model × horizon × scope
#
# compare:
#
#   Full F4 effect:
#       MAE_F3 - MAE_F4_real
#
# with individual add-one effects:
#       MAE_F3 - MAE_F3_plus_component
#
# IMPORTANT:
# The sum of independently refitted component effects is only
# a DESCRIPTIVE diagnostic. It is NOT a formal decomposition
# or Shapley attribution.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. Restart-safe recovery of full-F4 primary contrasts
# ============================================================

if (
    "phase2_primary_f3_vs_f4"
    not in globals()
):

    primary_path = (
        PHASE2_RESULTS_DIR
        /
        "phase2_four_model_F3_vs_F4_real.csv"
    )


    assert primary_path.is_file()


    phase2_primary_f3_vs_f4 = (
        pd.read_csv(
            primary_path
        )
    )


assert len(
    phase2_primary_f3_vs_f4
) == 24


# ============================================================
# 2. Restart-safe component bootstrap recovery
# ============================================================

if (
    "phase2_component_bootstrap_summary"
    not in globals()
):

    component_path = (
        PHASE2_RESULTS_DIR
        /
        "component_screen"
        /
        "phase2_component_four_model_bootstrap.csv"
    )


    assert component_path.is_file()


    phase2_component_bootstrap_summary = (
        pd.read_csv(
            component_path
        )
    )


assert len(
    phase2_component_bootstrap_summary
) == 120


# ============================================================
# 3. Normalize full-F4 effect
#
# Existing primary statistic:
#
#   comparator(F3) - F4_real
#
# therefore already equals:
#
#   MAE_F3 - MAE_F4_real
#
# Positive = complete F4 improves over F3.
# ============================================================

full_effect = (
    phase2_primary_f3_vs_f4[
        [
            "model",
            "horizon_minutes",
            "scope",
            "delta_MAE_comparator_minus_F4_real_kW",
            "ci95_low_kW",
            "ci95_high_kW",
            "evidence_class",
        ]
    ]
    .rename(
        columns={
            "delta_MAE_comparator_minus_F4_real_kW":
                "full_F4_delta_kW",

            "ci95_low_kW":
                "full_F4_ci_low_kW",

            "ci95_high_kW":
                "full_F4_ci_high_kW",

            "evidence_class":
                "full_F4_evidence",
        }
    )
)


# ============================================================
# 4. Pivot component point effects
# ============================================================

component_delta_wide = (
    phase2_component_bootstrap_summary
    .pivot_table(
        index=[
            "model",
            "horizon_minutes",
            "scope",
        ],
        columns="component_family",
        values="delta_MAE_F3_minus_component_kW",
        aggfunc="first",
    )
    .reset_index()
)


component_delta_wide.columns.name = None


component_delta_wide = (
    component_delta_wide.rename(
        columns={
            "tithi":
                "tithi_delta_kW",

            "paksha":
                "paksha_delta_kW",

            "karana":
                "karana_delta_kW",

            "nakshatra":
                "nakshatra_delta_kW",

            "yoga":
                "yoga_delta_kW",
        }
    )
)


assert len(
    component_delta_wide
) == 24


# ============================================================
# 5. Merge full F4 and five individual families
# ============================================================

phase2_full_component_synthesis = (
    full_effect
    .merge(
        component_delta_wide,
        on=[
            "model",
            "horizon_minutes",
            "scope",
        ],
        how="inner",
        validate="one_to_one",
    )
)


assert len(
    phase2_full_component_synthesis
) == 24


# ============================================================
# 6. Descriptive sum of independently fitted components
# ============================================================

component_delta_columns = [
    "tithi_delta_kW",
    "paksha_delta_kW",
    "karana_delta_kW",
    "nakshatra_delta_kW",
    "yoga_delta_kW",
]


phase2_full_component_synthesis[
    "sum_add_one_delta_kW"
] = (
    phase2_full_component_synthesis[
        component_delta_columns
    ]
    .sum(
        axis=1
    )
)


# Positive residual:
# complete F4 performs better than suggested by simple
# summation of independent add-one point effects.
#
# Negative residual:
# complete F4 performs worse than that descriptive sum.

phase2_full_component_synthesis[
    "full_minus_sum_delta_kW"
] = (
    phase2_full_component_synthesis[
        "full_F4_delta_kW"
    ]
    -
    phase2_full_component_synthesis[
        "sum_add_one_delta_kW"
    ]
)


phase2_full_component_synthesis[
    "absolute_full_minus_sum_kW"
] = np.abs(
    phase2_full_component_synthesis[
        "full_minus_sum_delta_kW"
    ]
)


# ============================================================
# 7. Strongest favorable/harmful single component
# ============================================================

family_names = [
    "tithi",
    "paksha",
    "karana",
    "nakshatra",
    "yoga",
]


component_matrix = (
    phase2_full_component_synthesis[
        component_delta_columns
    ]
    .to_numpy(
        dtype=float
    )
)


best_positions = np.argmax(
    component_matrix,
    axis=1,
)


worst_positions = np.argmin(
    component_matrix,
    axis=1,
)


phase2_full_component_synthesis[
    "largest_positive_component"
] = [
    family_names[
        position
    ]
    for position
    in best_positions
]


phase2_full_component_synthesis[
    "largest_positive_component_delta_kW"
] = (
    component_matrix[
        np.arange(
            len(
                component_matrix
            )
        ),
        best_positions,
    ]
)


phase2_full_component_synthesis[
    "most_negative_component"
] = [
    family_names[
        position
    ]
    for position
    in worst_positions
]


phase2_full_component_synthesis[
    "most_negative_component_delta_kW"
] = (
    component_matrix[
        np.arange(
            len(
                component_matrix
            )
        ),
        worst_positions,
    ]
)


# ============================================================
# 8. Number of resolved component effects per condition
# ============================================================

resolved_lookup = (
    phase2_component_bootstrap_summary
    .assign(
        resolved=lambda frame:
            ~frame[
                "evidence_class"
            ].eq(
                "CI_crosses_zero"
            ),

        resolved_benefit=lambda frame:
            frame[
                "evidence_class"
            ].eq(
                "component_lower_MAE"
            ),

        resolved_harm=lambda frame:
            frame[
                "evidence_class"
            ].eq(
                "F3_lower_MAE"
            ),
    )
    .groupby(
        [
            "model",
            "horizon_minutes",
            "scope",
        ]
    )
    .agg(
        resolved_component_count=(
            "resolved",
            "sum",
        ),

        resolved_beneficial_component_count=(
            "resolved_benefit",
            "sum",
        ),

        resolved_harmful_component_count=(
            "resolved_harm",
            "sum",
        ),
    )
    .reset_index()
)


phase2_full_component_synthesis = (
    phase2_full_component_synthesis
    .merge(
        resolved_lookup,
        on=[
            "model",
            "horizon_minutes",
            "scope",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# 9. Ridge-specific Tithi + Yoga diagnostic
#
# Motivated by the empirical component result.
#
# This is descriptive only.
# ============================================================

phase2_full_component_synthesis[
    "tithi_plus_yoga_delta_kW"
] = (
    phase2_full_component_synthesis[
        "tithi_delta_kW"
    ]
    +
    phase2_full_component_synthesis[
        "yoga_delta_kW"
    ]
)


phase2_full_component_synthesis[
    "full_minus_tithi_yoga_delta_kW"
] = (
    phase2_full_component_synthesis[
        "full_F4_delta_kW"
    ]
    -
    phase2_full_component_synthesis[
        "tithi_plus_yoga_delta_kW"
    ]
)


ridge_diagnostic = (
    phase2_full_component_synthesis
    .loc[
        phase2_full_component_synthesis[
            "model"
        ].eq(
            "Ridge"
        ),
        [
            "model",
            "horizon_minutes",
            "scope",
            "full_F4_delta_kW",
            "tithi_delta_kW",
            "yoga_delta_kW",
            "tithi_plus_yoga_delta_kW",
            "full_minus_tithi_yoga_delta_kW",
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# 10. Display complete mechanism table
# ============================================================

display_columns = [
    "model",
    "horizon_minutes",
    "scope",
    "full_F4_delta_kW",
    "tithi_delta_kW",
    "paksha_delta_kW",
    "karana_delta_kW",
    "nakshatra_delta_kW",
    "yoga_delta_kW",
    "sum_add_one_delta_kW",
    "full_minus_sum_delta_kW",
    "largest_positive_component",
    "most_negative_component",
    "resolved_beneficial_component_count",
    "resolved_harmful_component_count",
]


print(
    "=" * 110
)

print(
    "FULL F4 vs ADD-ONE COMPONENT SYNTHESIS"
)

print(
    "Positive delta = lower MAE than F3"
)

print(
    "=" * 110
)


display(
    phase2_full_component_synthesis[
        display_columns
    ]
)


# ============================================================
# 11. Ridge mechanism diagnostic
# ============================================================

print(
    "\n"
    +
    "=" * 110
)

print(
    "RIDGE: TITHI + YOGA DESCRIPTIVE DIAGNOSTIC"
)

print(
    "=" * 110
)


display(
    ridge_diagnostic
)


# ============================================================
# 12. Architecture-level non-additivity summary
# ============================================================

architecture_nonadditivity = (
    phase2_full_component_synthesis
    .groupby(
        "model"
    )
    .agg(
        mean_absolute_full_minus_sum_kW=(
            "absolute_full_minus_sum_kW",
            "mean",
        ),

        max_absolute_full_minus_sum_kW=(
            "absolute_full_minus_sum_kW",
            "max",
        ),

        mean_full_F4_delta_kW=(
            "full_F4_delta_kW",
            "mean",
        ),
    )
    .reset_index()
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "DESCRIPTIVE FULL-vs-SUM DISCREPANCY BY ARCHITECTURE"
)

print(
    "Smaller values indicate closer agreement with the"
    " sum of independently fitted add-one point effects."
)

print(
    "=" * 110
)


display(
    architecture_nonadditivity
)


# ============================================================
# 13. Save outputs
# ============================================================

synthesis_path = (
    PHASE2_COMPONENT_DIR
    /
    "phase2_full_F4_vs_component_synthesis.csv"
)


ridge_path = (
    PHASE2_COMPONENT_DIR
    /
    "phase2_ridge_tithi_yoga_diagnostic.csv"
)


nonadditivity_path = (
    PHASE2_COMPONENT_DIR
    /
    "phase2_architecture_component_nonadditivity.csv"
)


phase2_full_component_synthesis.to_csv(
    synthesis_path,
    index=False,
)


ridge_diagnostic.to_csv(
    ridge_path,
    index=False,
)


architecture_nonadditivity.to_csv(
    nonadditivity_path,
    index=False,
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "PHASE 2A COMPONENT-MECHANISM SYNTHESIS: PASSED"
)

print(
    "=" * 110
)


print(
    "Conditions:",
    len(
        phase2_full_component_synthesis
    )
)


print(
    "\nSaved:"
)

print(
    " ",
    synthesis_path
)


print(
    "\nSTOP HERE."
)

print(
    "No model was trained."
)

FULL F4 vs ADD-ONE COMPONENT SYNTHESIS
Positive delta = lower MAE than F3


,model,horizon_minutes,scope,full_F4_delta_kW,tithi_delta_kW,paksha_delta_kW,karana_delta_kW,nakshatra_delta_kW,yoga_delta_kW,sum_add_one_delta_kW,full_minus_sum_delta_kW,largest_positive_component,most_negative_component,resolved_beneficial_component_count,resolved_harmful_component_count
0,LSTM,15,all,-0.001948,0.027002,-0.022279,0.023189,-0.041545,0.010451,-0.003182,0.001234,tithi,nakshatra,3,2
1,LSTM,15,daylight,0.011381,0.025418,-0.003435,0.023938,-0.002462,0.018548,0.062007,-0.050626,tithi,paksha,3,0
2,LSTM,30,all,0.014906,-0.014038,0.009714,0.001664,0.004818,-0.016899,-0.014741,0.029647,paksha,yoga,2,2
3,LSTM,30,daylight,-0.006208,-0.012872,-0.006478,0.006407,-0.007625,-0.007759,-0.028327,0.022119,karana,tithi,1,4
4,LSTM,60,all,-0.001180,-0.013806,0.003273,-0.005846,-0.002237,-0.010214,-0.028830,0.027651,paksha,tithi,0,2
5,LSTM,60,daylight,0.002576,-0.023436,-0.002127,-0.008961,0.001847,-0.002410,-0.035089,0.037664,nakshatra,tithi,0,1
6,LightGBM,15,all,-0.001135,-0.001231,-0.001692,-0.000547,-0.000419,-0.000061,-0.003950,0.002815,yoga,paksha,0,2
7,LightGBM,15,daylight,-0.001799,-0.001833,-0.002350,-0.000914,-0.000386,0.000097,-0.005385,0.003586,yoga,paksha,0,2
8,LightGBM,30,all,0.001769,0.000955,0.000781,0.001274,0.000228,0.000710,0.003947,-0.002178,karana,nakshatra,1,0
9,LightGBM,30,daylight,0.002565,0.001533,0.001299,0.001716,0.000161,0.001018,0.005728,-0.003163,karana,nakshatra,0,0



RIDGE: TITHI + YOGA DESCRIPTIVE DIAGNOSTIC


,model,horizon_minutes,scope,full_F4_delta_kW,tithi_delta_kW,yoga_delta_kW,tithi_plus_yoga_delta_kW,full_minus_tithi_yoga_delta_kW
0,Ridge,15,all,-0.010945,-0.002852,-0.007919,-0.010770,-0.000174
1,Ridge,15,daylight,-0.007050,-0.001698,-0.004860,-0.006558,-0.000492
2,Ridge,30,all,-0.019218,-0.004964,-0.014076,-0.019040,-0.000178
3,Ridge,30,daylight,-0.012941,-0.003056,-0.009183,-0.012239,-0.000702
4,Ridge,60,all,-0.035300,-0.009152,-0.025941,-0.035094,-0.000206
5,Ridge,60,daylight,-0.025369,-0.005922,-0.018414,-0.024336,-0.001033



DESCRIPTIVE FULL-vs-SUM DISCREPANCY BY ARCHITECTURE
Smaller values indicate closer agreement with the sum of independently fitted add-one point effects.


,model,mean_absolute_full_minus_sum_kW,max_absolute_full_minus_sum_kW,mean_full_F4_delta_kW
0,LSTM,0.028157,0.050626,0.003255
1,LightGBM,0.003848,0.006166,0.000478
2,Ridge,0.001035,0.002893,-0.018470
3,TCN,0.174889,0.321626,0.013621



PHASE 2A COMPONENT-MECHANISM SYNTHESIS: PASSED
Conditions: 24

Saved:
  /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/results/notebook13_phase2_f4_representation_analysis/component_screen/phase2_full_F4_vs_component_synthesis.csv

STOP HERE.
No model was trained.
